# Haru AIO Downloader

Satu notebook untuk semua downloader & tools:

- **Mirror (GDrive / GoFile / URL → GDrive / HF)** → `haru-mirror` (mirror file/folder ke Google Drive atau Hugging Face Dataset dengan subfolder)
- **YouTube / Playlist** → `haru-ytdl` \(auto-upload **Gofile** + kirim link ke bot, opsional cookies.txt\)
- **LRC Lirik** → `haru-lrc` \(cari & download LRC, konversi ke SRT, kirim file ke bot\)
- **Manga 4 Sumber** → `haru-manga` (MangaDex, HentaiRead.com, Kanzenin.info, CrotPedia.net; download chapter, preview 10 halaman + ZIP otomatis dikirim ke bot)
- **Netflix Checker** → `haru-check` \(cek akun valid/hold/invalid, hasil + `Hits.zip` + ringkasan Telegra.ph ke bot\)
- **Transfer.it Renew** → `haru-transferit` \(perpanjang & hidupkan kembali semua transfer ke 90 hari\)
- **Subtitle SubSource** → `haru-sub` \(cari subtitle, lihat uploader/deskripsi/preview seperti di web, download & kirim ke bot\)
- **Cookie Cloudflare** → `haru-cookie` — buka Chrome otomatis, lolos challenge HentaiRead, tulis cookies.txt Netscape + MANGACOOKIE/UA siap-tempel


**Cara pakai:** jalankan cell **Install**, buka **Web Terminal**, ketik salah satu command di atas. Hasil masuk `/content/downloads/` per jenis.

> Companion dari `mkvtoolnix.ipynb` (muxing & extract MKV). Secrets dipakai bersama (`/content/.haru_secrets.json`).

---

### Persiapan

Buka menu **Rahasia** (ikon kunci), tambah secret berikut (aktifkan toggle-nya):

| Key | Value | Keterangan |
|-----|-------|------------|
| `HARU_BOT_TOKEN` | *(token BotFather)* | Notif & kirim file Telegram |
| `OWNER_ID` | *(chat ID)* | Tujuan notif Telegram |
| `GOFILE_API_TOKEN` | *(dari akun gofile.io)* | Upload file ke Gofile |
| `SUBSOURCE_API_KEY` | *(subsource.net → Profile → API Key)* | Downloader subtitle `haru-sub` |
| `GDRIVE_CLIENT_ID` | *(Google Cloud)* | Upload GDrive via API |
| `GDRIVE_CLIENT_SECRET` | *(Google Cloud)* | Upload GDrive via API |
| `GDRIVE_REFRESH_TOKEN` | *(OAuth flow)* | Upload GDrive via API |
| `GDRIVE_FOLDER_ID` | *(opsional)* | Folder GDrive tujuan |
| `MANGACOOKIE` | *(opsional)* | Cookie utk situs kena Cloudflare (HentaiRead) — boleh hasil ekspor cookies.txt (Netscape) atau raw Cookie header; otomatis ambil domain hentairead saja |
| `MANGACOOKIE_UA` | *(opsional)* | User-Agent browser yg membuat cookie tsb (wajib sama, default: Chrome Windows) |
| `HF_TOKEN` | *(huggingface.co/settings/tokens, role Write)* | Token HuggingFace untuk upload dataset |
| `HF_REPO_ID` | *(username/nama-dataset)* | Dataset repo Hugging Face tujuan (mis. username/my-dataset) |
| `MANGACOOKIE` | *(opsional)* | Cookie `cf_clearance` (dll) utk situs kena Cloudflare spt HentaiRead — diambil dari DevTools browser |
| `MANGACOOKIE_UA` | *(opsional)* | User-Agent browser yg membuat cookie tsb (wajib sama, default: Chrome Windows) |


## 1 — Install AIO Downloader


In [ ]:
#@title Install AIO { display-mode: "form" }
install_aio = True #@param {type:"boolean"}

if install_aio:
    import subprocess, os, base64, json
    print('Install system (ffmpeg, yt-dlp, requests, colorama, gdown)...')
    subprocess.run(['apt-get', 'update', '-qq'], capture_output=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg', 'tmux', 'curl', 'tree', 'wget', 'mkvtoolnix', 'mediainfo', 'mc', 'jq', 'unzip'], capture_output=True)
    subprocess.run(['pip', 'install', '-q', 'yt-dlp', 'requests', 'colorama', 'gdown', 'cloudscraper', 'huggingface_hub'], capture_output=True)
    print('Pasang commands...')
    # Install Yazi (modern TUI file manager)
    if not os.path.exists('/usr/local/bin/yazi'):
        try:
            from pathlib import Path
            yazi_url = 'https://github.com/sxyazi/yazi/releases/latest/download/yazi-x86_64-unknown-linux-musl.zip'
            subprocess.run(['curl', '-s', '-L', yazi_url, '-o', '/tmp/yazi.zip'], check=True)
            subprocess.run(['unzip', '-q', '-o', '/tmp/yazi.zip', '-d', '/tmp/yazi_extracted'], check=True)
            import shutil
            for p in Path('/tmp/yazi_extracted').rglob('yazi'):
                if p.is_file() and os.access(p, os.X_OK):
                    shutil.copy2(p, '/usr/local/bin/yazi')
                    break
            subprocess.run(['chmod', '+x', '/usr/local/bin/yazi'])
        except Exception: pass
    try:
        if os.path.exists('/usr/local/bin/yazi') and not os.path.exists('/usr/bin/yazi'):
            os.symlink('/usr/local/bin/yazi', '/usr/bin/yazi')
    except Exception: pass
    TOOLS = {
        'haru-mirror': """...""",
        'haru-lrc': """...""",
        'haru-manga': """...""",
        'haru-ytdl': """IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJoYXJ1LXl0ZGwgLSBZb3VUdWJlIGRvd25sb2FkZXIgKHBvcnQgb2YgeW91dHViZV9kb3dubG9hZGVyLmJhdCBtZW51KS4iIiIKaW1wb3J0IHN1YnByb2Nlc3MsIHN5cywgb3MsIHJlLCBqc29uCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKQkFTRSA9IFBhdGgoJy9jb250ZW50L2Rvd25sb2FkcycpClZJRCA9IEJBU0UgLyAnVmlkZW8nCkFVRCA9IEJBU0UgLyAnQXVkaW8nClBMID0gQkFTRSAvICdQbGF5bGlzdCcKZm9yIGQgaW4gW1ZJRCwgQVVELCBQTF06CiAgICBkLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKQ09PS0lFUyA9IFBhdGgoJy9jb250ZW50L2Nvb2tpZXMudHh0JykKClRHQk9UID0gJycKT1dORVIgPSAnJwoKZGVmIGxvYWRfc2VjcmV0cygpOgogICAgZ2xvYmFsIFRHQk9ULCBPV05FUgogICAgdHJ5OgogICAgICAgIGlmIG9zLnBhdGguZXhpc3RzKCcvY29udGVudC8uaGFydV9zZWNyZXRzLmpzb24nKToKICAgICAgICAgICAgZCA9IGpzb24ubG9hZChvcGVuKCcvY29udGVudC8uaGFydV9zZWNyZXRzLmpzb24nKSkKICAgICAgICAgICAgZm9yIGssIHYgaW4gZC5pdGVtcygpOgogICAgICAgICAgICAgICAgaWYgdiBhbmQgbm90IG9zLmVudmlyb24uZ2V0KGspOgogICAgICAgICAgICAgICAgICAgIG9zLmVudmlyb25ba10gPSBzdHIodikKICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgcGFzcwogICAgVEdCT1QgPSBvcy5lbnZpcm9uLmdldCgnSEFSVV9CT1RfVE9LRU4nLCAnJykKICAgIE9XTkVSID0gb3MuZW52aXJvbi5nZXQoJ09XTkVSX0lEJywgJycpCiAgICBpZiBub3QgVEdCT1Qgb3Igbm90IE9XTkVSOgogICAgICAgIHRyeToKICAgICAgICAgICAgZnJvbSBnb29nbGUuY29sYWIgaW1wb3J0IHVzZXJkYXRhCiAgICAgICAgICAgIGlmIG5vdCBUR0JPVDoKICAgICAgICAgICAgICAgIFRHQk9UID0gc3RyKHVzZXJkYXRhLmdldCgnSEFSVV9CT1RfVE9LRU4nKSBvciAnJykKICAgICAgICAgICAgaWYgbm90IE9XTkVSOgogICAgICAgICAgICAgICAgT1dORVIgPSBzdHIodXNlcmRhdGEuZ2V0KCdPV05FUl9JRCcpIG9yICcnKQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgIHBhc3MKCmRlZiB0Z19zZW5kKG1zZyk6CiAgICBpZiBub3QgVEdCT1Qgb3Igbm90IE9XTkVSOgogICAgICAgIHJldHVybgogICAgdHJ5OgogICAgICAgIGltcG9ydCByZXF1ZXN0cwogICAgICAgIHJlcXVlc3RzLnBvc3QoJ2h0dHBzOi8vYXBpLnRlbGVncmFtLm9yZy9ib3QnICsgVEdCT1QgKyAnL3NlbmRNZXNzYWdlJywKICAgICAgICAgICAgICAgICAgICAgIGpzb249eydjaGF0X2lkJzogT1dORVIsICd0ZXh0JzogbXNnLCAncGFyc2VfbW9kZSc6ICdIVE1MJywKICAgICAgICAgICAgICAgICAgICAgICAgICAgICdkaXNhYmxlX3dlYl9wYWdlX3ByZXZpZXcnOiBUcnVlfSwgdGltZW91dD0xMCkKICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgcGFzcwoKZGVmIGNpKCk6CiAgICBvcy5zeXN0ZW0oJ2NscycgaWYgb3MubmFtZSA9PSAnbnQnIGVsc2UgJ2NsZWFyJykKCmRlZiBoZHIodCk6CiAgICBwcmludCgnXG4nICsgJz0nICogNjIpCiAgICBwcmludCgnICAnICsgdCkKICAgIHByaW50KCc9JyAqIDYyKQoKZGVmIGNvb2tpZV9hcmdzKCk6CiAgICBpZiBDT09LSUVTLmV4aXN0cygpOgogICAgICAgIHVzZSA9IGlucHV0KCcgIGNvb2tpZXMudHh0IGRpdGVtdWthbiwgcGFrYWk/IFtZL25dOiAnKS5zdHJpcCgpLmxvd2VyKCkKICAgICAgICBpZiB1c2UgaW4gKCcnLCAneScpOgogICAgICAgICAgICByZXR1cm4gWyctLWNvb2tpZXMnLCBzdHIoQ09PS0lFUyldCiAgICByZXR1cm4gW10KCmRlZiBwcmV2aWV3KHVybCk6CiAgICBwcmludCgnICBDZWsgaW5mby4uLicpCiAgICByID0gc3VicHJvY2Vzcy5ydW4oWyd5dC1kbHAnLCAnLS1uby1wbGF5bGlzdCcsICctLXByaW50JywKICAgICAgICAgICAgICAgICAgICAgICAgJ0p1ZHVsOiAlKHRpdGxlKXMgfCBEdXJhc2k6ICUoZHVyYXRpb25fc3RyaW5nKXMgfCBDaGFubmVsOiAlKHVwbG9hZGVyKXMnLAogICAgICAgICAgICAgICAgICAgICAgICAnLS1za2lwLWRvd25sb2FkJywgdXJsXSwgY2FwdHVyZV9vdXRwdXQ9VHJ1ZSwgdGV4dD1UcnVlLCB0aW1lb3V0PTYwKQogICAgcHJpbnQoJyAgJyArIChyLnN0ZG91dC5zdHJpcCgpIG9yICd0aWRhayBiaXNhIHByZXZpZXcnKSArICdcbicpCgpkZWYgYXNrX3Jlc29sdXRpb24oKToKICAgIHByaW50KCcgIFJlc29sdXNpOiBbMV0gQmVzdCAgWzJdIDEwODBwICBbM10gNzIwcCAgWzRdIDQ4MHAnKQogICAgYyA9IChpbnB1dCgnICBQaWxpaCBbMS00XSAoRW50ZXI9MSk6ICcpLnN0cmlwKCkgb3IgJzEnKQogICAgaWYgYyA9PSAnMic6CiAgICAgICAgcmV0dXJuICdidipbaGVpZ2h0PD0xMDgwXStiYS9iW2hlaWdodDw9MTA4MF0vYnYqK2JhL2InCiAgICBpZiBjID09ICczJzoKICAgICAgICByZXR1cm4gJ2J2KltoZWlnaHQ8PTcyMF0rYmEvYltoZWlnaHQ8PTcyMF0vYnYqK2JhL2InCiAgICBpZiBjID09ICc0JzoKICAgICAgICByZXR1cm4gJ2J2KltoZWlnaHQ8PTQ4MF0rYmEvYltoZWlnaHQ8PTQ4MF0vYnYqK2JhL2InCiAgICByZXR1cm4gJ2J2KitiYS9iJwoKZGVmIGFza19tZXJnZSgpOgogICAgcHJpbnQoJyAgRm9ybWF0OiBbMV0gbXA0ICBbMl0gbWt2JykKICAgIGMgPSAoaW5wdXQoJyAgUGlsaWggWzEtMl0gKEVudGVyPTEpOiAnKS5zdHJpcCgpIG9yICcxJykKICAgIHJldHVybiAnbWt2JyBpZiBjID09ICcyJyBlbHNlICdtcDQnCgpkZWYgYXNrX3N1YnRpdGxlKCk6CiAgICBwcmludCgnICBTdWJ0aXRsZTogWzFdIFRhbnBhICBbMl0gRW1iZWQgIFszXSBGaWxlIHBpc2FoICBbNF0gS2VkdWFueWEnKQogICAgYyA9IChpbnB1dCgnICBQaWxpaCBbMS00XSAoRW50ZXI9MSk6ICcpLnN0cmlwKCkgb3IgJzEnKQogICAgaWYgYyA9PSAnMSc6CiAgICAgICAgcmV0dXJuIFtdCiAgICBsYW5ncyA9IGlucHV0KCcgIEJhaGFzYSAoaWQsZW4samEgLyBhbGwpIFthbGxdOiAnKS5zdHJpcCgpIG9yICdhbGwnCiAgICBhcmdzID0gWyctLXN1Yi1sYW5ncycsIGxhbmdzXQogICAgaWYgYyBpbiAoJzInLCAnNCcpOgogICAgICAgIGFyZ3MuYXBwZW5kKCctLWVtYmVkLXN1YnMnKQogICAgaWYgYyBpbiAoJzMnLCAnNCcpOgogICAgICAgIGFyZ3MuYXBwZW5kKCctLXdyaXRlLXN1YnMnKQogICAgcmV0dXJuIGFyZ3MKCmRlZiBydW5fZGwoYXJncywgZGVzYyk6CiAgICBwcmludCgnXG4gID09PSBET1dOTE9BRDogJyArIGRlc2MgKyAnID09PVxuJykKICAgIHIgPSBzdWJwcm9jZXNzLnJ1bihbJ3l0LWRscCddICsgYXJncykKICAgIGlmIHIucmV0dXJuY29kZSA9PSAwOgogICAgICAgIHByaW50KCdcbiAgU2VsZXNhaS4nKQogICAgICAgIHRnX3NlbmQoJzxiPnl0ZGwgc2VsZXNhaTwvYj5cbicgKyBkZXNjKQogICAgZWxzZToKICAgICAgICBwcmludCgnXG4gIEdhZ2FsIC8gZGliYXRhbGthbi4nKQogICAgaW5wdXQoJ1xuICBFbnRlci4uLicpCgpkZWYgc2luZ2xlX3ZpZGVvKCk6CiAgICBjaSgpCiAgICBoZHIoJ1ZJREVPIFNBVFVBTicpCiAgICB1cmwgPSBpbnB1dCgnXG4gIFVSTDogJykuc3RyaXAoKQogICAgaWYgbm90IHVybDoKICAgICAgICByZXR1cm4KICAgIHByZXZpZXcodXJsKQogICAgZm10ID0gYXNrX3Jlc29sdXRpb24oKQogICAgbWVyZ2UgPSBhc2tfbWVyZ2UoKQogICAgc3VicyA9IGFza19zdWJ0aXRsZSgpCiAgICBjayA9IGNvb2tpZV9hcmdzKCkKICAgIG91dCA9IHN0cihWSUQgLyAnJSh0aXRsZSlzIFslKGlkKXNdLiUoZXh0KXMnKQogICAgcnVuX2RsKFsnLS1uby1wbGF5bGlzdCddICsgY2sgKyBbJy1mJywgZm10LCAnLS1tZXJnZS1vdXRwdXQtZm9ybWF0JywgbWVyZ2VdICsKICAgICAgICAgICBzdWJzICsgWyctbycsIG91dCwgdXJsXSwgJ3ZpZGVvOiAnICsgdXJsKQoKZGVmIGF1ZGlvX29ubHkocGxheWxpc3Q9RmFsc2UpOgogICAgY2koKQogICAgaGRyKCdBVURJTyBTQUpBJyArICgnIChQTEFZTElTVCknIGlmIHBsYXlsaXN0IGVsc2UgJycpKQogICAgdXJsID0gaW5wdXQoJ1xuICBVUkw6ICcpLnN0cmlwKCkKICAgIGlmIG5vdCB1cmw6CiAgICAgICAgcmV0dXJuCiAgICBwcmludCgnICBGb3JtYXQ6IFsxXSBtcDMgIFsyXSBtNGEgIFszXSBvcHVzICBbNF0gZmxhYyAgWzVdIHdhdicpCiAgICBjID0gKGlucHV0KCcgIFBpbGloIChFbnRlcj0xKTogJykuc3RyaXAoKSBvciAnMScpCiAgICBleHQgPSB7JzEnOiAnbXAzJywgJzInOiAnbTRhJywgJzMnOiAnb3B1cycsICc0JzogJ2ZsYWMnLCAnNSc6ICd3YXYnfS5nZXQoYywgJ21wMycpCiAgICBwcmludCgnICBLdWFsaXRhczogWzFdIEJlc3QgIFsyXSAzMjBLICBbM10gMjU2SyAgWzRdIDE5MksgIFs1XSAxMjhLJykKICAgIHEgPSAoaW5wdXQoJyAgUGlsaWggKEVudGVyPTEpOiAnKS5zdHJpcCgpIG9yICcxJykKICAgIHF1YWwgPSB7JzEnOiAnMCcsICcyJzogJzMyMEsnLCAnMyc6ICcyNTZLJywgJzQnOiAnMTkySycsICc1JzogJzEyOEsnfS5nZXQocSwgJzAnKQogICAgY2sgPSBjb29raWVfYXJncygpCiAgICBvdXRkaXIgPSBQTCBpZiBwbGF5bGlzdCBlbHNlIEFVRAogICAgb3V0ID0gc3RyKG91dGRpciAvICclKHRpdGxlKXMgWyUoaWQpc10uJShleHQpcycpCiAgICBhcmdzID0gKFtdIGlmIHBsYXlsaXN0IGVsc2UgWyctLW5vLXBsYXlsaXN0J10pICsgY2sgKyBbJy0tZXh0cmFjdC1hdWRpbycsCiAgICAgICAgICAgICctLWF1ZGlvLWZvcm1hdCcsIGV4dCwgJy0tYXVkaW8tcXVhbGl0eScsIHF1YWwsCiAgICAgICAgICAgICctLWVtYmVkLXRodW1ibmFpbCcsICctLWFkZC1tZXRhZGF0YScsICctbycsIG91dCwgdXJsXQogICAgcnVuX2RsKGFyZ3MsICdhdWRpbyAnICsgZXh0ICsgJzogJyArIHVybCkKCmRlZiBwbGF5bGlzdF92aWRlbygpOgogICAgY2koKQogICAgaGRyKCdQTEFZTElTVCBWSURFTycpCiAgICB1cmwgPSBpbnB1dCgnXG4gIFVSTCBwbGF5bGlzdDogJykuc3RyaXAoKQogICAgaWYgbm90IHVybDoKICAgICAgICByZXR1cm4KICAgIHByaW50KCcgIEFtYmlsIGRhZnRhciBpc2kuLi4nKQogICAgciA9IHN1YnByb2Nlc3MucnVuKFsneXQtZGxwJywgJy0tZmxhdC1wbGF5bGlzdCcsICctLXByaW50JywKICAgICAgICAgICAgICAgICAgICAgICAgJyUocGxheWxpc3RfaW5kZXgpMDNkIHwgJSh0aXRsZSlzIHwgJShkdXJhdGlvbl9zdHJpbmcpcycsIHVybF0sCiAgICAgICAgICAgICAgICAgICAgICAgY2FwdHVyZV9vdXRwdXQ9VHJ1ZSwgdGV4dD1UcnVlLCB0aW1lb3V0PTEyMCkKICAgIHByaW50KHIuc3Rkb3V0WzozMDAwXSkKICAgIHNlbCA9IChpbnB1dCgnICBQaWxpaCBbQSBzZW11YSAvIDEtNSAvIDEsMyw1LTddIChFbnRlcj1BKTogJykuc3RyaXAoKSBvciAnQScpCiAgICBwYXJncyA9IFtdCiAgICBpZiBzZWwudXBwZXIoKSAhPSAnQSc6CiAgICAgICAgcGFyZ3MgPSBbJy0tcGxheWxpc3QtaXRlbXMnLCBzZWxdCiAgICBmbXQgPSBhc2tfcmVzb2x1dGlvbigpCiAgICBtZXJnZSA9IGFza19tZXJnZSgpCiAgICBjayA9IGNvb2tpZV9hcmdzKCkKICAgIG91dCA9IHN0cihQTCAvICclKHBsYXlsaXN0X2luZGV4KTAzZCAtICUodGl0bGUpcyBbJShpZClzXS4lKGV4dClzJykKICAgIHJ1bl9kbChwYXJncyArIGNrICsgWyctZicsIGZtdCwgJy0tbWVyZ2Utb3V0cHV0LWZvcm1hdCcsIG1lcmdlLCAnLW8nLCBvdXQsIHVybF0sCiAgICAgICAgICAgJ3BsYXlsaXN0OiAnICsgdXJsICsgJyBpdGVtcz0nICsgc2VsKQoKZGVmIHN1YnRpdGxlX21vZGUoKToKICAgIGNpKCkKICAgIGhkcignVklERU8gKyBTVUJUSVRMRSBTUEVTSUFMJykKICAgIHVybCA9IGlucHV0KCdcbiAgVVJMOiAnKS5zdHJpcCgpCiAgICBpZiBub3QgdXJsOgogICAgICAgIHJldHVybgogICAgcHJldmlldyh1cmwpCiAgICBsYW5ncyA9IGlucHV0KCcgIEJhaGFzYSAoaWQsZW4samEgLyBhbGwpIFthbGxdOiAnKS5zdHJpcCgpIG9yICdhbGwnCiAgICBmbXQgPSBhc2tfcmVzb2x1dGlvbigpCiAgICBtZXJnZSA9IGFza19tZXJnZSgpCiAgICBjayA9IGNvb2tpZV9hcmdzKCkKICAgIG91dCA9IHN0cihWSUQgLyAnJSh0aXRsZSlzIFslKGlkKXNdLiUoZXh0KXMnKQogICAgcnVuX2RsKFsnLS1uby1wbGF5bGlzdCddICsgY2sgKyBbJy1mJywgZm10LCAnLS1tZXJnZS1vdXRwdXQtZm9ybWF0JywgbWVyZ2UsCiAgICAgICAgICAgICctLXN1Yi1sYW5ncycsIGxhbmdzLCAnLS1lbWJlZC1zdWJzJywgJy0td3JpdGUtc3VicycsICctbycsIG91dCwgdXJsXSwKICAgICAgICAgICAndmlkZW8rc3ViOiAnICsgdXJsKQoKZGVmIGFkdmFuY2VkKCk6CiAgICBjaSgpCiAgICBoZHIoJ01PREUgQURWQU5DRUQnKQogICAgcHJpbnQoJyAgQ29udG9oOiAtLW5vLXBsYXlsaXN0IC1mICJidiorYmEvYiIgVVJMJykKICAgIHJhdyA9IGlucHV0KCdcbiAgQXJndW1lbiB5dC1kbHA6ICcpLnN0cmlwKCkKICAgIGlmIG5vdCByYXc6CiAgICAgICAgcmV0dXJuCiAgICBpbXBvcnQgc2hsZXgKICAgIHJ1bl9kbChzaGxleC5zcGxpdChyYXcpLCAnY3VzdG9tOiAnICsgcmF3Wzo4MF0pCgpkZWYgdXBkYXRlX3l0ZGwoKToKICAgIHByaW50KCcgIFVwZGF0ZSB5dC1kbHAuLi4nKQogICAgc3VicHJvY2Vzcy5ydW4oW3N5cy5leGVjdXRhYmxlLCAnLW0nLCAncGlwJywgJ2luc3RhbGwnLCAnLXEnLCAnLVUnLCAneXQtZGxwJ10pCiAgICByID0gc3VicHJvY2Vzcy5ydW4oWyd5dC1kbHAnLCAnLS12ZXJzaW9uJ10sIGNhcHR1cmVfb3V0cHV0PVRydWUsIHRleHQ9VHJ1ZSkKICAgIHByaW50KCcgIFZlcnNpOiAnICsgci5zdGRvdXQuc3RyaXAoKSkKICAgIGlucHV0KCdcbiAgRW50ZXIuLi4nKQoKZGVmIG1haW4oKToKICAgIGxvYWRfc2VjcmV0cygpCiAgICB3aGlsZSBUcnVlOgogICAgICAgIGNpKCkKICAgICAgICBwcmludCgnXG4nICsgJz0nICogNjIpCiAgICAgICAgcHJpbnQoJyAgaGFydS15dGRsIC0tIFlvdVR1YmUgRG93bmxvYWRlcicpCiAgICAgICAgcHJpbnQoJz0nICogNjIpCiAgICAgICAgcHJpbnQoKQogICAgICAgIHByaW50KCcgIFsxXSBWaWRlbyBzYXR1YW4nKQogICAgICAgIHByaW50KCcgIFsyXSBBdWRpbyBzYWphIChtcDMvbTRhL29wdXMvZmxhYy93YXYpJykKICAgICAgICBwcmludCgnICBbM10gUGxheWxpc3QgdmlkZW8nKQogICAgICAgIHByaW50KCcgIFs0XSBQbGF5bGlzdCBhdWRpbycpCiAgICAgICAgcHJpbnQoJyAgWzVdIFZpZGVvICsgc3VidGl0bGUgc3Blc2lhbCcpCiAgICAgICAgcHJpbnQoJyAgWzZdIEFkdmFuY2VkIChhcmd1bWVuIHNlbmRpcmkpJykKICAgICAgICBwcmludCgnICBbN10gVXBkYXRlIHl0LWRscCcpCiAgICAgICAgcHJpbnQoKQogICAgICAgIHByaW50KCcgIFtRXSBLZWx1YXInKQogICAgICAgIHByaW50KCkKICAgICAgICBjID0gaW5wdXQoJyAgUGlsaWg6ICcpLnN0cmlwKCkudXBwZXIoKQogICAgICAgIGlmIGMgPT0gJ1EnOgogICAgICAgICAgICBwcmludCgnXG4gIEJ5ZSEnKQogICAgICAgICAgICByZXR1cm4KICAgICAgICBlbGlmIGMgPT0gJzEnOgogICAgICAgICAgICBzaW5nbGVfdmlkZW8oKQogICAgICAgIGVsaWYgYyA9PSAnMic6CiAgICAgICAgICAgIGF1ZGlvX29ubHkoKQogICAgICAgIGVsaWYgYyA9PSAnMyc6CiAgICAgICAgICAgIHBsYXlsaXN0X3ZpZGVvKCkKICAgICAgICBlbGlmIGMgPT0gJzQnOgogICAgICAgICAgICBhdWRpb19vbmx5KHBsYXlsaXN0PVRydWUpCiAgICAgICAgZWxpZiBjID09ICc1JzoKICAgICAgICAgICAgc3VidGl0bGVfbW9kZSgpCiAgICAgICAgZWxpZiBjID09ICc2JzoKICAgICAgICAgICAgYWR2YW5jZWQoKQogICAgICAgIGVsaWYgYyA9PSAnNyc6CiAgICAgICAgICAgIHVwZGF0ZV95dGRsKCkKCmlmIF9fbmFtZV9fID09ICdfX21haW5fXyc6CiAgICB0cnk6CiAgICAgICAgbWFpbigpCiAgICBleGNlcHQgS2V5Ym9hcmRJbnRlcnJ1cHQ6CiAgICAgICAgcHJpbnQoJ1xuICBEaWJhdGFsa2FuLicpCg==""",
        'haru-check': """...""",
        'haru-transferit': """IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJoYXJ1LXRyYW5zZmVyaXQgLSBUcmFuc2Zlci5pdCBBdXRvLVJlbmV3IENMSSAocG9ydCBUcmFuc2Zlcml0X1JlbmV3YWwpLgpQZXJwYW5qYW5nICYgaGlkdXBrYW4ga2VtYmFsaSBTRU1VQSB0cmFuc2ZlciBkaSB0cmFuc2Zlci5pdCAoYmFja2VuZCBNRUdBKS4KVGFucGEgZGVwZW5kZW5jeSByaWNoLCBjdWt1cCByZXF1ZXN0cy4KIiIiCmltcG9ydCBiYXNlNjQKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCByYW5kb20KaW1wb3J0IHRpbWUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgcmVxdWVzdHMKCkNPTkZJR19QQVRIID0gUGF0aCgiL2NvbnRlbnQvLmhhcnVfdHJhbnNmZXJpdC5qc29uIikKCkRFRkFVTFRTID0gewogICAgInNpZCI6ICIiLAogICAgImRheXMiOiA5MCwKICAgICJkZWxheV9taW4iOiAwLjYsCiAgICAiZGVsYXlfbWF4IjogMS40LAogICAgImFwaV91cmwiOiAiaHR0cHM6Ly9idDcuYXBpLm1lZ2EuY28ubnovY3MiLAogICAgInVzZXJfYWdlbnQiOiAoCiAgICAgICAgIk1vemlsbGEvNS4wIChXaW5kb3dzIE5UIDEwLjA7IFdpbjY0OyB4NjQpICIKICAgICAgICAiQXBwbGVXZWJLaXQvNTM3LjM2IChLSFRNTCwgbGlrZSBHZWNrbykgIgogICAgICAgICJDaHJvbWUvMTI3LjAuMC4wIFNhZmFyaS81MzcuMzYiCiAgICApLAp9CgoKZGVmIGxvYWRfY29uZmlnKCkgLT4gZGljdDoKICAgIGNvbmZpZyA9IGRpY3QoREVGQVVMVFMpCiAgICBpZiBDT05GSUdfUEFUSC5leGlzdHMoKToKICAgICAgICB0cnk6CiAgICAgICAgICAgIGNvbmZpZy51cGRhdGUoanNvbi5sb2FkcyhDT05GSUdfUEFUSC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpKQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgIHBhc3MKICAgIHJldHVybiBjb25maWcKCgpkZWYgc2F2ZV9jb25maWcoY29uZmlnOiBkaWN0KSAtPiBOb25lOgogICAgQ09ORklHX1BBVEgud3JpdGVfdGV4dChqc29uLmR1bXBzKGNvbmZpZywgaW5kZW50PTQsIGVuc3VyZV9hc2NpaT1GYWxzZSksIGVuY29kaW5nPSJ1dGYtOCIpCgoKY2xhc3MgTWVnYUFQSUVycm9yKFJ1bnRpbWVFcnJvcik6CiAgICBwYXNzCgoKZGVmIGI2NHVybF9kZWNvZGUoZGF0YTogc3RyKSAtPiBieXRlczoKICAgIHBhZGRpbmcgPSAiPSIgKiAoLWxlbihkYXRhKSAlIDQpCiAgICByZXR1cm4gYmFzZTY0LnVybHNhZmVfYjY0ZGVjb2RlKGRhdGEgKyBwYWRkaW5nKQoKCmNsYXNzIFRyYW5zZmVyaXRNYW5hZ2VyOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHNpZDogc3RyLCBjb25maWc6IGRpY3QpOgogICAgICAgIGlmIG5vdCBzaWQgb3Igbm90IHNpZC5zdHJpcCgpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJTSUQga29zb25nLiBNYXN1a2thbiBTZXNzaW9uIElEIChsb2NhbFN0b3JhZ2UgJ3NpZCcpLiIpCiAgICAgICAgc2VsZi5jb25maWcgPSBjb25maWcgb3IgbG9hZF9jb25maWcoKQogICAgICAgIHNlbGYuc2lkID0gc2lkLnN0cmlwKCkKICAgICAgICBzZWxmLnNlcW5vID0gcmFuZG9tLnJhbmRpbnQoMTAwXzAwMCwgOTk5Xzk5OSkKICAgICAgICBzZWxmLnNlc3Npb24gPSByZXF1ZXN0cy5TZXNzaW9uKCkKICAgICAgICBzZWxmLnNlc3Npb24uaGVhZGVycy51cGRhdGUoewogICAgICAgICAgICAiVXNlci1BZ2VudCI6IHNlbGYuY29uZmlnLmdldCgidXNlcl9hZ2VudCIpLAogICAgICAgICAgICAiQWNjZXB0IjogImFwcGxpY2F0aW9uL2pzb24sIHRleHQvcGxhaW4sICovKiIsCiAgICAgICAgICAgICJPcmlnaW4iOiAiaHR0cHM6Ly90cmFuc2Zlci5pdCIsCiAgICAgICAgICAgICJSZWZlcmVyIjogImh0dHBzOi8vdHJhbnNmZXIuaXQvIiwKICAgICAgICAgICAgIkNvbnRlbnQtVHlwZSI6ICJhcHBsaWNhdGlvbi9qc29uIiwKICAgICAgICB9KQoKICAgIGRlZiBfYXBpX3VybChzZWxmKToKICAgICAgICByZXR1cm4gc2VsZi5jb25maWcuZ2V0KCJhcGlfdXJsIiwgImh0dHBzOi8vYnQ3LmFwaS5tZWdhLmNvLm56L2NzIikKCiAgICBkZWYgX3JlcShzZWxmLCBwYXlsb2FkKToKICAgICAgICBzZWxmLnNlcW5vICs9IDEKICAgICAgICBwYXJhbXMgPSB7ImlkIjogc2VsZi5zZXFubywgInNpZCI6IHNlbGYuc2lkfQogICAgICAgIGJvZHkgPSBwYXlsb2FkIGlmIGlzaW5zdGFuY2UocGF5bG9hZCwgbGlzdCkgZWxzZSBbcGF5bG9hZF0KICAgICAgICBmb3IgYXR0ZW1wdCBpbiByYW5nZSg0KToKICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgcmVzcCA9IHNlbGYuc2Vzc2lvbi5wb3N0KHNlbGYuX2FwaV91cmwoKSwgcGFyYW1zPXBhcmFtcywganNvbj1ib2R5LCB0aW1lb3V0PTMwKQogICAgICAgICAgICAgICAgcmVzcC5yYWlzZV9mb3Jfc3RhdHVzKCkKICAgICAgICAgICAgICAgIGRhdGEgPSByZXNwLmpzb24oKQogICAgICAgICAgICBleGNlcHQgKHJlcXVlc3RzLlJlcXVlc3RFeGNlcHRpb24sIFZhbHVlRXJyb3IpIGFzIGV4YzoKICAgICAgICAgICAgICAgIGlmIGF0dGVtcHQgPCAzOgogICAgICAgICAgICAgICAgICAgIHRpbWUuc2xlZXAoMSArIGF0dGVtcHQpCiAgICAgICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgICAgIHJhaXNlIE1lZ2FBUElFcnJvcihmIk5ldHdvcmsvSFRUUCBlcnJvcjoge2V4Y30iKSBmcm9tIGV4YwogICAgICAgICAgICBjb2RlID0gZGF0YSBpZiBpc2luc3RhbmNlKGRhdGEsIGludCkgZWxzZSAoCiAgICAgICAgICAgICAgICBkYXRhWzBdIGlmIGlzaW5zdGFuY2UoZGF0YSwgbGlzdCkgYW5kIGxlbihkYXRhKSA9PSAxIGFuZCBpc2luc3RhbmNlKGRhdGFbMF0sIGludCkgZWxzZSBOb25lCiAgICAgICAgICAgICkKICAgICAgICAgICAgaWYgY29kZSBpcyBub3QgTm9uZSBhbmQgY29kZSA8IDA6CiAgICAgICAgICAgICAgICBpZiBjb2RlID09IC0zIGFuZCBhdHRlbXB0IDwgMzoKICAgICAgICAgICAgICAgICAgICB0aW1lLnNsZWVwKDEgKyBhdHRlbXB0KQogICAgICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgICAgICByYWlzZSBNZWdhQVBJRXJyb3IoZiJNRUdBIEFQSSBFcnJvciBDb2RlOiB7Y29kZX0iKQogICAgICAgICAgICByZXR1cm4gZGF0YVswXSBpZiBpc2luc3RhbmNlKHBheWxvYWQsIGRpY3QpIGVsc2UgZGF0YQogICAgICAgIHJhaXNlIE1lZ2FBUElFcnJvcigiUmVxdWVzdCBnYWdhbCBzZXRlbGFoIGJlYmVyYXBhIHBlcmNvYmFhbi4iKQoKICAgIGRlZiBsaXN0X3RyYW5zZmVycyhzZWxmKToKICAgICAgICBkYXRhID0gc2VsZi5fcmVxKHsiYSI6ICJ4bCJ9KQogICAgICAgIGlmIGlzaW5zdGFuY2UoZGF0YSwgbGlzdCk6CiAgICAgICAgICAgIHJldHVybiBbdCBmb3IgdCBpbiBkYXRhIGlmIGlzaW5zdGFuY2UodCwgZGljdCldCiAgICAgICAgcmV0dXJuIFtdCgogICAgZGVmIHJlbmV3X2FuZF9yZXZpdmUoc2VsZiwgeGgsIGRheXM9OTApOgogICAgICAgIHJldHVybiBzZWxmLl9yZXEoeyJhIjogInhtIiwgInhoIjogeGgsICJlIjogZGF5cyAqIDg2NDAwfSkKCiAgICBkZWYgZ2V0X21ldGFkYXRhKHNlbGYsIHhoKToKICAgICAgICBkYXRhID0gc2VsZi5fcmVxKHsiYSI6ICJ4aSIsICJ4aCI6IHhofSkKICAgICAgICByZXR1cm4gZGF0YSBpZiBpc2luc3RhbmNlKGRhdGEsIGRpY3QpIGVsc2Uge30KCiAgICBkZWYgZGVsZXRlX3RyYW5zZmVyKHNlbGYsIHhoKToKICAgICAgICByZXR1cm4gc2VsZi5fcmVxKHsiYSI6ICJ4ZCIsICJ4aCI6IHhofSkKCiAgICBkZWYgcmVuZXdfYWxsKHNlbGYsIGRheXM9OTAsIGRlbGF5X3JhbmdlPU5vbmUsIG9uX3Byb2dyZXNzPU5vbmUpOgogICAgICAgIGlmIGRlbGF5X3JhbmdlIGlzIE5vbmU6CiAgICAgICAgICAgIGRlbGF5X3JhbmdlID0gKAogICAgICAgICAgICAgICAgZmxvYXQoc2VsZi5jb25maWcuZ2V0KCJkZWxheV9taW4iLCAwLjYpKSwKICAgICAgICAgICAgICAgIGZsb2F0KHNlbGYuY29uZmlnLmdldCgiZGVsYXlfbWF4IiwgMS40KSksCiAgICAgICAgICAgICkKICAgICAgICB0cmFuc2ZlcnMgPSBzZWxmLmxpc3RfdHJhbnNmZXJzKCkKICAgICAgICByZXN1bHRzID0geyJ0b3RhbCI6IGxlbih0cmFuc2ZlcnMpLCAic3VjY2VzcyI6IDAsICJmYWlsZWQiOiAwLCAiZGV0YWlscyI6IFtdfQogICAgICAgIGZvciBpZHgsIHQgaW4gZW51bWVyYXRlKHRyYW5zZmVycywgMSk6CiAgICAgICAgICAgIHhoID0gdC5nZXQoInhoIiwgIiIpCiAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgIHNlbGYucmVuZXdfYW5kX3Jldml2ZSh4aCwgZGF5cz1kYXlzKQogICAgICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgICAgIG1ldGEgPSBzZWxmLmdldF9tZXRhZGF0YSh4aCkKICAgICAgICAgICAgICAgICAgICByYXdfdGl0bGUgPSBtZXRhLmdldCgidCIsICIiKQogICAgICAgICAgICAgICAgICAgIHRpdGxlID0gYjY0dXJsX2RlY29kZShyYXdfdGl0bGUpLmRlY29kZSgidXRmLTgiLCBlcnJvcnM9Imlnbm9yZSIpIGlmIHJhd190aXRsZSBlbHNlICJObyBUaXRsZSIKICAgICAgICAgICAgICAgICAgICB0b3RhbF9ieXRlcyA9IG1ldGEuZ2V0KCJzaXplIiwgWzBdKVswXSBpZiBpc2luc3RhbmNlKG1ldGEuZ2V0KCJzaXplIiksIGxpc3QpIGVsc2UgMAogICAgICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgICAgICAgICB0aXRsZSA9ICJBY3RpdmUgKE1ldGFkYXRhIHVuYXZhaWxhYmxlKSIKICAgICAgICAgICAgICAgICAgICB0b3RhbF9ieXRlcyA9IDAKICAgICAgICAgICAgICAgIHJlc3VsdHNbInN1Y2Nlc3MiXSArPSAxCiAgICAgICAgICAgICAgICByZXN1bHRzWyJkZXRhaWxzIl0uYXBwZW5kKCh4aCwgdGl0bGUsIHRvdGFsX2J5dGVzLCBmIkFDVElWRSAoe2RheXN9IERheXMpIikpCiAgICAgICAgICAgICAgICBpZiBvbl9wcm9ncmVzczoKICAgICAgICAgICAgICAgICAgICBvbl9wcm9ncmVzcyhpZHgsIGxlbih0cmFuc2ZlcnMpLCB4aCwgdGl0bGUsICJTVUNDRVNTIikKICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBleGM6CiAgICAgICAgICAgICAgICByZXN1bHRzWyJmYWlsZWQiXSArPSAxCiAgICAgICAgICAgICAgICByZXN1bHRzWyJkZXRhaWxzIl0uYXBwZW5kKCh4aCwgIkVycm9yIiwgMCwgc3RyKGV4YykpKQogICAgICAgICAgICAgICAgaWYgb25fcHJvZ3Jlc3M6CiAgICAgICAgICAgICAgICAgICAgb25fcHJvZ3Jlc3MoaWR4LCBsZW4odHJhbnNmZXJzKSwgeGgsICJFcnJvciIsIHN0cihleGMpKQogICAgICAgICAgICBpZiBpZHggPCBsZW4odHJhbnNmZXJzKToKICAgICAgICAgICAgICAgIHRpbWUuc2xlZXAocmFuZG9tLnVuaWZvcm0oKmRlbGF5X3JhbmdlKSkKICAgICAgICByZXR1cm4gcmVzdWx0cwoKCmRlZiBvayh0KTogcmV0dXJuICdcMDMzWzkybScgKyB0ICsgJ1wwMzNbMG0nCmRlZiBlcih0KTogcmV0dXJuICdcMDMzWzkxbScgKyB0ICsgJ1wwMzNbMG0nCmRlZiBkaW0odCk6IHJldHVybiAnXDAzM1s5MG0nICsgdCArICdcMDMzWzBtJwpkZWYgY3kodCk6IHJldHVybiAnXDAzM1s5Nm0nICsgdCArICdcMDMzWzBtJwpkZWYgeWwodCk6IHJldHVybiAnXDAzM1s5M20nICsgdCArICdcMDMzWzBtJwoKZGVmIGNpKCk6CiAgICBpbXBvcnQgc3lzCiAgICBzeXMuc3Rkb3V0LndyaXRlKCdceDFiWzJKXHgxYltIJykKICAgIHN5cy5zdGRvdXQuZmx1c2goKQoKZGVmIGhkcih0aXRsZSk6CiAgICBwcmludCgnXG4nICsgJz0nICogNjApCiAgICBwcmludCgnICAnICsgdGl0bGUpCiAgICBwcmludCgnPScgKiA2MCkKCgpkZWYgaHVtYW5fc2l6ZShudW0pOgogICAgZm9yIHVuaXQgaW4gKCJCIiwgIktCIiwgIk1CIiwgIkdCIiwgIlRCIik6CiAgICAgICAgaWYgYWJzKG51bSkgPCAxMDI0LjA6CiAgICAgICAgICAgIHJldHVybiBmIntudW0gaWYgdW5pdCA9PSAnQicgZWxzZSBmJ3tudW06LjFmfSd9IHt1bml0fSIKICAgICAgICBudW0gLz0gMTAyNC4wCiAgICByZXR1cm4gZiJ7bnVtOi4xZn0gUEIiCgoKZGVmIGVuc3VyZV9zaWQoY29uZmlnKToKICAgIHNpZCA9IGNvbmZpZy5nZXQoInNpZCIsICIiKS5zdHJpcCgpCiAgICBpZiBzaWQ6CiAgICAgICAgcmV0dXJuIHNpZAogICAgcHJpbnQoeWwoIlxuICBTZXNzaW9uIElEIChTSUQpIGJlbHVtIHRlcnNpbXBhbi4iKSkKICAgIHByaW50KGRpbSgiICBDYXJhIGRhcGF0OiBidWthIHRyYW5zZmVyLml0IGRpIGJyb3dzZXIsIERldlRvb2xzIChGMTIpIC0+IEFwcGxpY2F0aW9uXG4gIC0+IExvY2FsIFN0b3JhZ2UgLT4gc2FsaW4gbmlsYWkgJ3NpZCcuIikpCiAgICBzaWQgPSBpbnB1dChjeSgiICBNYXN1a2thbiBTZXNzaW9uIElEOiAiKSkuc3RyaXAoKQogICAgd2hpbGUgbm90IHNpZDoKICAgICAgICBwcmludChlcigiICBTSUQgdGlkYWsgYm9sZWgga29zb25nLiIpKQogICAgICAgIHNpZCA9IGlucHV0KGN5KCIgIE1hc3Vra2FuIFNlc3Npb24gSUQ6ICIpKS5zdHJpcCgpCiAgICBjb25maWdbInNpZCJdID0gc2lkCiAgICBzYXZlX2NvbmZpZyhjb25maWcpCiAgICBwcmludChvaygiICBTSUQgYmVyaGFzaWwgZGlzaW1wYW4uIikpCiAgICByZXR1cm4gc2lkCgoKZGVmIGNtZF9yZW5ld19hbGwoY29uZmlnKToKICAgIGRheXNfaW4gPSBpbnB1dChmIiAgQmVyYXBhIGhhcmkgbWFzYSBha3RpZj8gW3tjb25maWcuZ2V0KCdkYXlzJywgOTApfV06ICIpLnN0cmlwKCkKICAgIGRheXMgPSBpbnQoZGF5c19pbikgaWYgZGF5c19pbi5sc3RyaXAoJy0nKS5pc2RpZ2l0KCkgZWxzZSBpbnQoY29uZmlnLmdldCgnZGF5cycsIDkwKSkKICAgIHNpZCA9IGVuc3VyZV9zaWQoY29uZmlnKQogICAgbWFuYWdlciA9IFRyYW5zZmVyaXRNYW5hZ2VyKHNpZCwgY29uZmlnKQogICAgcHJpbnQoZGltKCIgIE1lbmdhbWJpbCBkYWZ0YXIgdHJhbnNmZXIuLi4iKSkKICAgIHRyeToKICAgICAgICB0cmFuc2ZlcnMgPSBtYW5hZ2VyLmxpc3RfdHJhbnNmZXJzKCkKICAgIGV4Y2VwdCBNZWdhQVBJRXJyb3IgYXMgZXhjOgogICAgICAgIHByaW50KGVyKGYiICBHYWdhbCBtZW5nYW1iaWwgdHJhbnNmZXI6IHtleGN9IikpCiAgICAgICAgcmV0dXJuCiAgICBpZiBub3QgdHJhbnNmZXJzOgogICAgICAgIHByaW50KHlsKCIgIFRpZGFrIGFkYSB0cmFuc2ZlciB5YW5nIGRpdGVtdWthbiBkaSBha3VuLiIpKQogICAgICAgIHJldHVybgogICAgcHJpbnQoZGltKGYiICBEaXRlbXVrYW4ge2xlbih0cmFuc2ZlcnMpfSB0cmFuc2Zlci4gTXVsYWkgcmVuZXcuLi4iKSkKICAgIGRldGFpbHMgPSBbXQogICAgdG90YWwgPSBsZW4odHJhbnNmZXJzKQogICAgZGVmIG9uX3Byb2dyZXNzKGlkeCwgdCwgeGgsIHRpdGxlLCBzdGF0dXMpOgogICAgICAgIGRldGFpbHMuYXBwZW5kKCh4aCwgdGl0bGUsIDAsIHN0YXR1cykpCiAgICAgICAgcHJpbnQoZiIgIFt7aWR4fS97dH1dIHt0aXRsZVs6NDBdfSAtPiB7c3RhdHVzfSIpCiAgICB0cnk6CiAgICAgICAgcmVzdWx0cyA9IG1hbmFnZXIucmVuZXdfYWxsKGRheXM9ZGF5cywgb25fcHJvZ3Jlc3M9b25fcHJvZ3Jlc3MpCiAgICBleGNlcHQgTWVnYUFQSUVycm9yIGFzIGV4YzoKICAgICAgICBwcmludChlcihmIiAgRXJyb3IgZmF0YWw6IHtleGN9IikpCiAgICAgICAgcmV0dXJuCiAgICBwcmludCgpCiAgICBwcmludChvayhmIiAgQmVyaGFzaWw6IHtyZXN1bHRzWydzdWNjZXNzJ119ICAgIikgKyBlcihmIkdhZ2FsOiB7cmVzdWx0c1snZmFpbGVkJ119ICAgIikgKyBmIlRvdGFsOiB7cmVzdWx0c1sndG90YWwnXX0iKQogICAgaWYgcmVzdWx0c1siZGV0YWlscyJdOgogICAgICAgIHByaW50KCdcbicgKyAnPScgKiA2MCkKICAgICAgICBwcmludCgnICBIQVNJTCBSRU5FVyAnICsgZiIoe2RheXN9IEhhcmkpIikKICAgICAgICBwcmludCgnPScgKiA2MCkKICAgICAgICBmb3IgaSwgKHhoLCB0aXRsZSwgc2l6ZSwgc3RhdHVzKSBpbiBlbnVtZXJhdGUocmVzdWx0c1siZGV0YWlscyJdLCAxKToKICAgICAgICAgICAgc2l6ZV9zdHIgPSBodW1hbl9zaXplKHNpemUpIGlmIHNpemUgZWxzZSAiLSIKICAgICAgICAgICAgYmFkZ2UgPSBvayhzdGF0dXMpIGlmIHN0YXR1cy5zdGFydHN3aXRoKCJBQ1RJVkUiKSBlbHNlIGVyKHN0YXR1cykKICAgICAgICAgICAgcHJpbnQoZiIgIHtpOj4yfS4ge3hofSAge3RpdGxlWzo0MF06PDQwfSB7c2l6ZV9zdHI6Pjh9ICB7YmFkZ2V9IikKICAgIGlucHV0KCdcbiAgRW50ZXIuLi4nKQoKCmRlZiBjbWRfbGlzdF90cmFuc2ZlcnMoY29uZmlnKToKICAgIHNpZCA9IGVuc3VyZV9zaWQoY29uZmlnKQogICAgbWFuYWdlciA9IFRyYW5zZmVyaXRNYW5hZ2VyKHNpZCwgY29uZmlnKQogICAgcHJpbnQoZGltKCIgIE1lbmdhbWJpbCBkYWZ0YXIgdHJhbnNmZXIuLi4iKSkKICAgIHRyeToKICAgICAgICB0cmFuc2ZlcnMgPSBtYW5hZ2VyLmxpc3RfdHJhbnNmZXJzKCkKICAgIGV4Y2VwdCBNZWdhQVBJRXJyb3IgYXMgZXhjOgogICAgICAgIHByaW50KGVyKGYiICBHYWdhbCBtZW5nYW1iaWwgdHJhbnNmZXI6IHtleGN9IikpCiAgICAgICAgcmV0dXJuCiAgICBpZiBub3QgdHJhbnNmZXJzOgogICAgICAgIHByaW50KHlsKCIgIFRpZGFrIGFkYSB0cmFuc2ZlciB5YW5nIGRpdGVtdWthbi4iKSkKICAgICAgICByZXR1cm4KICAgIHByaW50KCdcbicgKyAnPScgKiA2MCkKICAgIHByaW50KCcgIERBRlRBUiBUUkFOU0ZFUicpCiAgICBwcmludCgnPScgKiA2MCkKICAgIGZvciB0IGluIHRyYW5zZmVyczoKICAgICAgICB4aCA9IHQuZ2V0KCJ4aCIsICIiKQogICAgICAgIHRpdGxlLCB0b3RhbF9ieXRlcyA9IHhoLCAwCiAgICAgICAgdHJ5OgogICAgICAgICAgICBtZXRhID0gbWFuYWdlci5nZXRfbWV0YWRhdGEoeGgpCiAgICAgICAgICAgIHJhdyA9IG1ldGEuZ2V0KCJ0IiwgIiIpCiAgICAgICAgICAgIGlmIHJhdzoKICAgICAgICAgICAgICAgIHRpdGxlID0gYjY0dXJsX2RlY29kZShyYXcpLmRlY29kZSgidXRmLTgiLCBlcnJvcnM9Imlnbm9yZSIpCiAgICAgICAgICAgIHNpemUgPSBtZXRhLmdldCgic2l6ZSIpCiAgICAgICAgICAgIHRvdGFsX2J5dGVzID0gc2l6ZVswXSBpZiBpc2luc3RhbmNlKHNpemUsIGxpc3QpIGFuZCBzaXplIGVsc2UgMAogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgIHBhc3MKICAgICAgICB0cyA9IHQuZ2V0KCJ0cyIsICIiKQogICAgICAgIHByaW50KGYiICB7eGh9ICB7dGl0bGVbOjQwXTo8NDB9IHtodW1hbl9zaXplKHRvdGFsX2J5dGVzKSBpZiB0b3RhbF9ieXRlcyBlbHNlICctJzo+OH0gIHRzPXt0c30iKQogICAgaW5wdXQoJ1xuICBFbnRlci4uLicpCgoKZGVmIGNtZF9kZWxldGVfdHJhbnNmZXIoY29uZmlnKToKICAgIHNpZCA9IGVuc3VyZV9zaWQoY29uZmlnKQogICAgbWFuYWdlciA9IFRyYW5zZmVyaXRNYW5hZ2VyKHNpZCwgY29uZmlnKQogICAgdHJ5OgogICAgICAgIHRyYW5zZmVycyA9IG1hbmFnZXIubGlzdF90cmFuc2ZlcnMoKQogICAgZXhjZXB0IE1lZ2FBUElFcnJvciBhcyBleGM6CiAgICAgICAgcHJpbnQoZXIoZiIgIEdhZ2FsIG1lbmdhbWJpbCB0cmFuc2Zlcjoge2V4Y30iKSkKICAgICAgICByZXR1cm4KICAgIGlmIG5vdCB0cmFuc2ZlcnM6CiAgICAgICAgcHJpbnQoeWwoIiAgVGlkYWsgYWRhIHRyYW5zZmVyIHlhbmcgZGl0ZW11a2FuLiIpKQogICAgICAgIHJldHVybgogICAgY2hvaWNlcyA9IHtzdHIoaSk6IHQuZ2V0KCJ4aCIsICIiKSBmb3IgaSwgdCBpbiBlbnVtZXJhdGUodHJhbnNmZXJzLCAxKX0KICAgIGZvciBpLCB0IGluIGVudW1lcmF0ZSh0cmFuc2ZlcnMsIDEpOgogICAgICAgIHByaW50KGYiICBbe2l9XSAge3QuZ2V0KCd4aCcsICcnKX0iKQogICAgcGljayA9IGlucHV0KGN5KCIgIFBpbGloIG5vbW9yIHRyYW5zZmVyIHVudHVrIGRpaGFwdXM6ICIpKS5zdHJpcCgpCiAgICB4aCA9IGNob2ljZXMuZ2V0KHBpY2spCiAgICBpZiBub3QgeGg6CiAgICAgICAgcHJpbnQoZXIoIiAgUGlsaWhhbiB0aWRhayB2YWxpZC4iKSkKICAgICAgICByZXR1cm4KICAgIHkgPSBpbnB1dChlcihmIiAgWWFraW4gaGFwdXMgdHJhbnNmZXIge3hofT8gKHkvbik6ICIpKS5zdHJpcCgpLmxvd2VyKCkKICAgIGlmIHkgIT0gJ3knOgogICAgICAgIHByaW50KGRpbSgiICBEaWJhdGFsa2FuLiIpKQogICAgICAgIHJldHVybgogICAgdHJ5OgogICAgICAgIG1hbmFnZXIuZGVsZXRlX3RyYW5zZmVyKHhoKQogICAgICAgIHByaW50KG9rKGYiICBUcmFuc2ZlciB7eGh9IGJlcmhhc2lsIGRpaGFwdXMuIikpCiAgICBleGNlcHQgTWVnYUFQSUVycm9yIGFzIGV4YzoKICAgICAgICBwcmludChlcihmIiAgR2FnYWwgbWVuZ2hhcHVzOiB7ZXhjfSIpKQoKCmRlZiBjbWRfc2V0dGluZ3MoY29uZmlnKToKICAgIGhkcignUEVOR0FUVVJBTicpCiAgICBwcmludChmIiAgU0lEIHRlcnNpbXBhbiA6IHsnWWEnIGlmIGNvbmZpZy5nZXQoJ3NpZCcpIGVsc2UgJ0JlbHVtJ30iKQogICAgcHJpbnQoZiIgIEhhcmkgZGVmYXVsdCAgOiB7Y29uZmlnLmdldCgnZGF5cycpfSIpCiAgICBwcmludChmIiAgSmVkYSAgICAgICAgIDoge2NvbmZpZy5nZXQoJ2RlbGF5X21pbicpfSAtIHtjb25maWcuZ2V0KCdkZWxheV9tYXgnKX0gZGV0aWsiKQogICAgcHJpbnQoKQogICAgZGF5c19pbiA9IGlucHV0KGYiICBIYXJpIGFrdGlmIGRlZmF1bHQgW3tjb25maWcuZ2V0KCdkYXlzJywgOTApfV06ICIpLnN0cmlwKCkKICAgIGlmIGRheXNfaW4ubHN0cmlwKCctJykuaXNkaWdpdCgpOgogICAgICAgIGNvbmZpZ1siZGF5cyJdID0gaW50KGRheXNfaW4pCiAgICBkbWluID0gaW5wdXQoZiIgIEplZGEgbWluaW11bSAoZGV0aWspIFt7Y29uZmlnLmdldCgnZGVsYXlfbWluJywgMC42KX1dOiAiKS5zdHJpcCgpCiAgICBpZiBkbWluOgogICAgICAgIHRyeToKICAgICAgICAgICAgY29uZmlnWyJkZWxheV9taW4iXSA9IGZsb2F0KGRtaW4pCiAgICAgICAgZXhjZXB0IFZhbHVlRXJyb3I6CiAgICAgICAgICAgIHBhc3MKICAgIGRtYXggPSBpbnB1dChmIiAgSmVkYSBtYWtzaW11bSAoZGV0aWspIFt7Y29uZmlnLmdldCgnZGVsYXlfbWF4JywgMS40KX1dOiAiKS5zdHJpcCgpCiAgICBpZiBkbWF4OgogICAgICAgIHRyeToKICAgICAgICAgICAgY29uZmlnWyJkZWxheV9tYXgiXSA9IGZsb2F0KGRtYXgpCiAgICAgICAgZXhjZXB0IFZhbHVlRXJyb3I6CiAgICAgICAgICAgIHBhc3MKICAgIHIgPSBpbnB1dCgiICBSZXNldCBTSUQgKGhhcHVzIHlhbmcgdGVyc2ltcGFuKT8gKHkvbiwgRW50ZXI9bik6ICIpLnN0cmlwKCkubG93ZXIoKQogICAgaWYgciA9PSAneSc6CiAgICAgICAgY29uZmlnWyJzaWQiXSA9ICIiCiAgICBzYXZlX2NvbmZpZyhjb25maWcpCiAgICBwcmludChvaygiICBQZW5nYXR1cmFuIHRlcnNpbXBhbi4iKSkKCgpkZWYgbWFpbigpOgogICAgY29uZmlnID0gbG9hZF9jb25maWcoKQogICAgd2hpbGUgVHJ1ZToKICAgICAgICBjaSgpCiAgICAgICAgcHJpbnQoJ1xuJyArICc9JyAqIDYwKQogICAgICAgIHByaW50KGN5KCcgICBUUkFOU0ZFUi5JVCBBVVRPLVJFTkVXJykpCiAgICAgICAgcHJpbnQoZGltKCcgICBQZXJwYW5qYW5nICYgaGlkdXBrYW4ga2VtYmFsaSBzZW11YSB0cmFuc2ZlciBrZSA5MCBoYXJpJykpCiAgICAgICAgcHJpbnQoJz0nICogNjApCiAgICAgICAgcHJpbnQoKQogICAgICAgIHByaW50KCcgIFsxXSBSZW5ldyAmIFJldml2ZSBTZW11YSBUcmFuc2ZlcicpCiAgICAgICAgcHJpbnQoJyAgWzJdIExpaGF0IFNlbXVhIFRyYW5zZmVyJykKICAgICAgICBwcmludCgnICBbM10gSGFwdXMgVHJhbnNmZXInKQogICAgICAgIHByaW50KCcgIFs0XSBQZW5nYXR1cmFuJykKICAgICAgICBwcmludCgnICBbNV0gS2VsdWFyJykKICAgICAgICBwcmludCgpCiAgICAgICAgYyA9IGlucHV0KCcgIFBpbGloIG1lbnU6ICcpLnN0cmlwKCkKICAgICAgICBpZiBjID09ICcxJzoKICAgICAgICAgICAgY21kX3JlbmV3X2FsbChjb25maWcpCiAgICAgICAgZWxpZiBjID09ICcyJzoKICAgICAgICAgICAgY21kX2xpc3RfdHJhbnNmZXJzKGNvbmZpZykKICAgICAgICBlbGlmIGMgPT0gJzMnOgogICAgICAgICAgICBjbWRfZGVsZXRlX3RyYW5zZmVyKGNvbmZpZykKICAgICAgICBlbGlmIGMgPT0gJzQnOgogICAgICAgICAgICBjbWRfc2V0dGluZ3MoY29uZmlnKQogICAgICAgIGVsaWYgYyA9PSAnNSc6CiAgICAgICAgICAgIHByaW50KCdcbiAgU2FtcGFpIGp1bXBhIScpCiAgICAgICAgICAgIHJldHVybgoKCmlmIF9fbmFtZV9fID09ICdfX21haW5fXyc6CiAgICB0cnk6CiAgICAgICAgbWFpbigpCiAgICBleGNlcHQgS2V5Ym9hcmRJbnRlcnJ1cHQ6CiAgICAgICAgcHJpbnQoJ1xuICBEaWJhdGFsa2FuLicp""",
        'haru-sub': """IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJoYXJ1LXN1YiAtIFN1YlNvdXJjZSBzdWJ0aXRsZSBkb3dubG9hZGVyLgpBUEk6IGh0dHBzOi8vYXBpLnN1YnNvdXJjZS5uZXQgICh3YWppYiBYLUFQSS1LZXkgZGFyaSBkYXNoYm9hcmQgcHJvZmlsZSBzdWJzb3VyY2UubmV0KQpLZXkgZGliYWNhIGRhcmkgc2VjcmV0IFNVQlNPVVJDRV9BUElfS0VZIC8gZW52IC8gL2NvbnRlbnQvLnN1YnNvdXJjZV9rZXkuCk1lbnlpbXBhbiBoYXNpbCBrZSAvY29udGVudC9kb3dubG9hZHMvc3VidGl0bGVzLCBvcHNpIGtpcmltIGtlIGJvdC4KIiIiCmltcG9ydCBpbwppbXBvcnQganNvbgppbXBvcnQgb3MKaW1wb3J0IHJlCmltcG9ydCBzeXMKaW1wb3J0IHRpbWUKaW1wb3J0IHppcGZpbGUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgcmVxdWVzdHMKCkFQSV9CQVNFID0gImh0dHBzOi8vYXBpLnN1YnNvdXJjZS5uZXQiCk9VVF9ESVIgPSBQYXRoKCIvY29udGVudC9kb3dubG9hZHMvc3VidGl0bGVzIikKS0VZX0ZJTEUgPSBQYXRoKCIvY29udGVudC8uc3Vic291cmNlX2tleSIpCgpMQU5HX01BUCA9IHsKICAgICJpZCI6ICJpbmRvbmVzaWFuIiwgImVuZyI6ICJlbmdsaXNoIiwgImVuIjogImVuZ2xpc2giLCAiZXMiOiAic3BhbmlzaCIsCiAgICAiZnIiOiAiZnJlbmNoIiwgImRlIjogImdlcm1hbiIsICJwdCI6ICJwb3J0dWd1ZXNlIiwgIml0IjogIml0YWxpYW4iLAogICAgIm5sIjogImR1dGNoIiwgInRyIjogInR1cmtpc2giLCAicnUiOiAicnVzc2lhbiIsICJhciI6ICJhcmFiaWMiLAogICAgImphIjogImphcGFuZXNlIiwgImtvIjogImtvcmVhbiIsICJ6aCI6ICJjaGluZXNlIiwgImhpIjogImhpbmRpIiwKICAgICJ2aSI6ICJ2aWV0bmFtZXNlIiwgInRoIjogInRoYWkiLCAibXMiOiAibWFsYXkiLCAicGwiOiAicG9saXNoIiwKfQoKU1VCX0VYVFMgPSAoIi5zcnQiLCAiLmFzcyIsICIuc3NhIiwgIi5zdWIiLCAiLnZ0dCIsICIuaWR4IikKCmRlZiBjaSgpOgogICAgc3lzLnN0ZG91dC53cml0ZSgnXHgxYlsySlx4MWJbSCcpCiAgICBzeXMuc3Rkb3V0LmZsdXNoKCkKCmRlZiBvayh0KTogcmV0dXJuICdcMDMzWzkybScgKyB0ICsgJ1wwMzNbMG0nCmRlZiBlcih0KTogcmV0dXJuICdcMDMzWzkxbScgKyB0ICsgJ1wwMzNbMG0nCmRlZiBkaW0odCk6IHJldHVybiAnXDAzM1s5MG0nICsgdCArICdcMDMzWzBtJwpkZWYgY3kodCk6IHJldHVybiAnXDAzM1s5Nm0nICsgdCArICdcMDMzWzBtJwpkZWYgeWwodCk6IHJldHVybiAnXDAzM1s5M20nICsgdCArICdcMDMzWzBtJwoKZGVmIGhkcih0aXRsZSk6CiAgICBwcmludCgnXG4nICsgJz0nICogNjIpCiAgICBwcmludCgnICAnICsgdGl0bGUpCiAgICBwcmludCgnPScgKiA2MikKCiMgLS0tLS0tLS0tLSBzZWNyZXRzIC8gdGVsZWdyYW0gLS0tLS0tLS0tLQpkZWYgbG9hZF9zZWNyZXRzKCk6CiAgICB0cnk6CiAgICAgICAgaWYgb3MucGF0aC5leGlzdHMoJy9jb250ZW50Ly5oYXJ1X3NlY3JldHMuanNvbicpOgogICAgICAgICAgICBkID0ganNvbi5sb2FkKG9wZW4oJy9jb250ZW50Ly5oYXJ1X3NlY3JldHMuanNvbicpKQogICAgICAgICAgICBmb3IgaywgdiBpbiBkLml0ZW1zKCk6CiAgICAgICAgICAgICAgICBpZiB2IGFuZCBub3Qgb3MuZW52aXJvbi5nZXQoayk6CiAgICAgICAgICAgICAgICAgICAgb3MuZW52aXJvbltrXSA9IHN0cih2KQogICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICBwYXNzCgpkZWYgZ2V0X2FwaV9rZXkoKToKICAgIGxvYWRfc2VjcmV0cygpCiAgICB2ID0gb3MuZW52aXJvbi5nZXQoJ1NVQlNPVVJDRV9BUElfS0VZJywgJycpLnN0cmlwKCkKICAgIGlmIHY6CiAgICAgICAgcmV0dXJuIHYKICAgIGlmIEtFWV9GSUxFLmV4aXN0cygpOgogICAgICAgIGsgPSBLRVlfRklMRS5yZWFkX3RleHQoZW5jb2Rpbmc9J3V0Zi04Jykuc3RyaXAoKQogICAgICAgIGlmIGs6CiAgICAgICAgICAgIHJldHVybiBrCiAgICB0cnk6CiAgICAgICAgZnJvbSBnb29nbGUuY29sYWIgaW1wb3J0IHVzZXJkYXRhCiAgICAgICAgdiA9IHN0cih1c2VyZGF0YS5nZXQoJ1NVQlNPVVJDRV9BUElfS0VZJykgb3IgJycpLnN0cmlwKCkKICAgICAgICBpZiB2OgogICAgICAgICAgICByZXR1cm4gdgogICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICBwYXNzCiAgICByZXR1cm4gTm9uZQoKZGVmIHNhdmVfYXBpX2tleShrZXkpOgogICAgdHJ5OgogICAgICAgIEtFWV9GSUxFLndyaXRlX3RleHQoa2V5LnN0cmlwKCksIGVuY29kaW5nPSd1dGYtOCcpCiAgICAgICAgcmV0dXJuIFRydWUKICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgcmV0dXJuIEZhbHNlCgpkZWYgdGdfY3JlZGVudGlhbHMoKToKICAgIGxvYWRfc2VjcmV0cygpCiAgICB0b2sgPSBvcy5lbnZpcm9uLmdldCgnSEFSVV9CT1RfVE9LRU4nLCAnJykKICAgIG9pZCA9IG9zLmVudmlyb24uZ2V0KCdPV05FUl9JRCcsICcnKQogICAgaWYgbm90IHRvayBvciBub3Qgb2lkOgogICAgICAgIHRyeToKICAgICAgICAgICAgZnJvbSBnb29nbGUuY29sYWIgaW1wb3J0IHVzZXJkYXRhCiAgICAgICAgICAgIGlmIG5vdCB0b2s6CiAgICAgICAgICAgICAgICB0b2sgPSBzdHIodXNlcmRhdGEuZ2V0KCdIQVJVX0JPVF9UT0tFTicpIG9yICcnKQogICAgICAgICAgICBpZiBub3Qgb2lkOgogICAgICAgICAgICAgICAgb2lkID0gc3RyKHVzZXJkYXRhLmdldCgnT1dORVJfSUQnKSBvciAnJykKICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICBwYXNzCiAgICByZXR1cm4gdG9rLCBvaWQKCmRlZiB0Z19zZW5kX2RvY3VtZW50KHBhdGgsIGNhcHRpb249JycpOgogICAgdG9rLCBvaWQgPSB0Z19jcmVkZW50aWFscygpCiAgICBpZiBub3QgdG9rIG9yIG5vdCBvaWQgb3Igbm90IG9zLnBhdGguZXhpc3RzKHN0cihwYXRoKSk6CiAgICAgICAgcmV0dXJuIEZhbHNlCiAgICB0cnk6CiAgICAgICAgd2l0aCBvcGVuKHN0cihwYXRoKSwgJ3JiJykgYXMgZmg6CiAgICAgICAgICAgIHIgPSByZXF1ZXN0cy5wb3N0KCdodHRwczovL2FwaS50ZWxlZ3JhbS5vcmcvYm90JyArIHRvayArICcvc2VuZERvY3VtZW50JywKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZGF0YT17J2NoYXRfaWQnOiBvaWQsICdjYXB0aW9uJzogY2FwdGlvbn0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGZpbGVzPXsnZG9jdW1lbnQnOiAoUGF0aChwYXRoKS5uYW1lLCBmaCl9LCB0aW1lb3V0PTYwKQogICAgICAgIHJldHVybiByLnN0YXR1c19jb2RlID09IDIwMAogICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICByZXR1cm4gRmFsc2UKCiMgLS0tLS0tLS0tLSBzdWJzb3VyY2UgYXBpIC0tLS0tLS0tLS0KZGVmIGFwaV9nZXQocGF0aCwgcGFyYW1zPU5vbmUsIHJldHJpZXM9Myk6CiAgICBrZXkgPSBnZXRfYXBpX2tleSgpCiAgICBoZWFkZXJzID0geyJYLUFQSS1LZXkiOiBrZXksICJBY2NlcHQiOiAiYXBwbGljYXRpb24vanNvbiJ9CiAgICB1cmwgPSBBUElfQkFTRSArICIvYXBpL3YxIiArIHBhdGgKICAgIGZvciBhdHRlbXB0IGluIHJhbmdlKHJldHJpZXMpOgogICAgICAgIHRyeToKICAgICAgICAgICAgciA9IHJlcXVlc3RzLmdldCh1cmwsIHBhcmFtcz1wYXJhbXMgb3Ige30sIGhlYWRlcnM9aGVhZGVycywgdGltZW91dD0yMCkKICAgICAgICBleGNlcHQgcmVxdWVzdHMuZXhjZXB0aW9ucy5SZXF1ZXN0RXhjZXB0aW9uIGFzIGU6CiAgICAgICAgICAgIHByaW50KGVyKGYiICBOZXR3b3JrIGVycm9yOiB7ZX0iKSkKICAgICAgICAgICAgcmV0dXJuIE5vbmUKICAgICAgICBpZiByLnN0YXR1c19jb2RlID09IDQyOToKICAgICAgICAgICAgd2FpdCA9IDUgKiAoYXR0ZW1wdCArIDEpCiAgICAgICAgICAgIHByaW50KHlsKGYiICDij7MgUmF0ZSBsaW1pdGVkLiBUdW5nZ3Uge3dhaXR9cy4uLiIpKQogICAgICAgICAgICB0aW1lLnNsZWVwKHdhaXQpCiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgaWYgci5zdGF0dXNfY29kZSA9PSA0MDE6CiAgICAgICAgICAgIHByaW50KGVyKCIgIEFQSSBrZXkgaW52YWxpZC9leHBpcmVkLiBDZWsga2V5IGRpIHN1YnNvdXJjZS5uZXQvZGFzaGJvYXJkL3Byb2ZpbGUiKSkKICAgICAgICAgICAgcmV0dXJuIE5vbmUKICAgICAgICBpZiByLnN0YXR1c19jb2RlICE9IDIwMDoKICAgICAgICAgICAgcHJpbnQoZXIoZiIgIEhUVFAge3Iuc3RhdHVzX2NvZGV9OiB7ci50ZXh0WzoxMjBdfSIpKQogICAgICAgICAgICByZXR1cm4gTm9uZQogICAgICAgIHRyeToKICAgICAgICAgICAgcmV0dXJuIHIuanNvbigpCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgcmV0dXJuIE5vbmUKICAgIHJldHVybiBOb25lCgpkZWYgc2VhcmNoX21vdmllcyhxdWVyeSwgbXR5cGU9ImFsbCIsIHllYXI9Tm9uZSk6CiAgICBwYXJhbXMgPSB7InNlYXJjaFR5cGUiOiAidGV4dCIsICJxIjogcXVlcnl9CiAgICBpZiBtdHlwZSBhbmQgbXR5cGUgaW4gKCJtb3ZpZSIsICJzZXJpZXMiKToKICAgICAgICBwYXJhbXNbInR5cGUiXSA9IG10eXBlCiAgICBpZiB5ZWFyOgogICAgICAgIHBhcmFtc1sieWVhciJdID0geWVhcgogICAgZGF0YSA9IGFwaV9nZXQoIi9tb3ZpZXMvc2VhcmNoIiwgcGFyYW1zKQogICAgaWYgZGF0YSBhbmQgaXNpbnN0YW5jZShkYXRhLmdldCgiZGF0YSIpLCBsaXN0KToKICAgICAgICByZXR1cm4gZGF0YVsiZGF0YSJdCiAgICByZXR1cm4gW10KCmRlZiBnZXRfc3VidGl0bGVzKG1vdmllX2lkLCBsYW5ndWFnZT1Ob25lLCBsaW1pdD0zMCwgc29ydD0icG9wdWxhciIpOgogICAgcGFyYW1zID0geyJtb3ZpZUlkIjogbW92aWVfaWQsICJsaW1pdCI6IGxpbWl0LCAic29ydCI6IHNvcnR9CiAgICBpZiBsYW5ndWFnZToKICAgICAgICBwYXJhbXNbImxhbmd1YWdlIl0gPSBsYW5ndWFnZQogICAgZGF0YSA9IGFwaV9nZXQoIi9zdWJ0aXRsZXMiLCBwYXJhbXMpCiAgICBpZiBkYXRhIGFuZCBpc2luc3RhbmNlKGRhdGEuZ2V0KCJkYXRhIiksIGxpc3QpOgogICAgICAgIHJldHVybiBkYXRhWyJkYXRhIl0KICAgIHJldHVybiBbXQoKZGVmIGRvd25sb2FkX3N1YnRpdGxlKHN1YnRpdGxlX2lkKToKICAgIGtleSA9IGdldF9hcGlfa2V5KCkKICAgIGhlYWRlcnMgPSB7IlgtQVBJLUtleSI6IGtleSwgIkFjY2VwdCI6ICJhcHBsaWNhdGlvbi9vY3RldC1zdHJlYW0ifQogICAgdXJsID0gZiJ7QVBJX0JBU0V9L2FwaS92MS9zdWJ0aXRsZXMve3N1YnRpdGxlX2lkfS9kb3dubG9hZCIKICAgIGZvciBhdHRlbXB0IGluIHJhbmdlKDMpOgogICAgICAgIHRyeToKICAgICAgICAgICAgciA9IHJlcXVlc3RzLmdldCh1cmwsIGhlYWRlcnM9aGVhZGVycywgdGltZW91dD0zMCkKICAgICAgICBleGNlcHQgcmVxdWVzdHMuZXhjZXB0aW9ucy5SZXF1ZXN0RXhjZXB0aW9uIGFzIGU6CiAgICAgICAgICAgIHByaW50KGVyKGYiICBOZXR3b3JrIGVycm9yOiB7ZX0iKSkKICAgICAgICAgICAgcmV0dXJuIE5vbmUKICAgICAgICBpZiByLnN0YXR1c19jb2RlID09IDQyOToKICAgICAgICAgICAgdGltZS5zbGVlcCg1ICogKGF0dGVtcHQgKyAxKSkKICAgICAgICAgICAgY29udGludWUKICAgICAgICBpZiByLnN0YXR1c19jb2RlID09IDIwMDoKICAgICAgICAgICAgcmV0dXJuIHIuY29udGVudAogICAgICAgIGlmIHIuc3RhdHVzX2NvZGUgPT0gNDAxOgogICAgICAgICAgICBwcmludChlcigiICBBUEkga2V5IGludmFsaWQvZXhwaXJlZC4iKSkKICAgICAgICAgICAgcmV0dXJuIE5vbmUKICAgICAgICBwcmludChlcihmIiAgSFRUUCB7ci5zdGF0dXNfY29kZX06IHtyLnRleHRbOjEyMF19IikpCiAgICAgICAgcmV0dXJuIE5vbmUKICAgIHJldHVybiBOb25lCgpkZWYgZ2V0X3N1YnRpdGxlX2RldGFpbChzdWJ0aXRsZV9pZCk6CiAgICBkYXRhID0gYXBpX2dldChmIi9zdWJ0aXRsZXMve3N1YnRpdGxlX2lkfSIpCiAgICBpZiBkYXRhIGFuZCBpc2luc3RhbmNlKGRhdGEuZ2V0KCJkYXRhIiksIGRpY3QpOgogICAgICAgIHJldHVybiBkYXRhWyJkYXRhIl0KICAgIGlmIGRhdGEgYW5kIGlzaW5zdGFuY2UoZGF0YS5nZXQoImRhdGEiKSwgbGlzdCkgYW5kIGRhdGFbImRhdGEiXToKICAgICAgICByZXR1cm4gZGF0YVsiZGF0YSJdWzBdCiAgICByZXR1cm4gTm9uZQoKZGVmIHNhbml0aXplKG5hbWUpOgogICAgbmFtZSA9IHJlLnN1YihyJ1s8PjoiL1xcfD8qXHgwMC1ceDFmXScsICcnLCBuYW1lKQogICAgcmV0dXJuIHJlLnN1YihyJ1xzKycsICcgJywgbmFtZSkuc3RyaXAoKVs6MTIwXSBvciAndW50aXRsZWQnCgpkZWYgZXh0cmFjdF9zdWJ0aXRsZXMoemlwX2J5dGVzLCBvdXRfZGlyLCBiYXNlX25hbWUpOgogICAgc2F2ZWQgPSBbXQogICAgdHJ5OgogICAgICAgIHdpdGggemlwZmlsZS5aaXBGaWxlKGlvLkJ5dGVzSU8oemlwX2J5dGVzKSkgYXMgejoKICAgICAgICAgICAgbmFtZXMgPSBbbiBmb3IgbiBpbiB6Lm5hbWVsaXN0KCkKICAgICAgICAgICAgICAgICAgICAgaWYgbm90IG4uZW5kc3dpdGgoIi8iKSBhbmQgbi5sb3dlcigpLmVuZHN3aXRoKFNVQl9FWFRTKV0KICAgICAgICAgICAgIyBwcmVmZXIgc3J0IHBlcnRhbWEKICAgICAgICAgICAgbmFtZXMuc29ydChrZXk9bGFtYmRhIG46IChuLmxvd2VyKCkuZW5kc3dpdGgoIi5zcnQiKSwgbikpCiAgICAgICAgICAgIGZvciBpLCBuIGluIGVudW1lcmF0ZShuYW1lcyk6CiAgICAgICAgICAgICAgICBleHQgPSBQYXRoKG4pLnN1ZmZpeAogICAgICAgICAgICAgICAgZGF0YSA9IHoucmVhZChuKQogICAgICAgICAgICAgICAgaWYgbGVuKG5hbWVzKSA9PSAxOgogICAgICAgICAgICAgICAgICAgIGZuYW1lID0gZiJ7YmFzZV9uYW1lfXtleHR9IgogICAgICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgICAgICBmbmFtZSA9IGYie2Jhc2VfbmFtZX1fe2krMX17ZXh0fSIKICAgICAgICAgICAgICAgIG91dCA9IG91dF9kaXIgLyBmbmFtZQogICAgICAgICAgICAgICAgb3V0LndyaXRlX2J5dGVzKGRhdGEpCiAgICAgICAgICAgICAgICBzYXZlZC5hcHBlbmQob3V0KQogICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOgogICAgICAgIHByaW50KGVyKGYiICBHYWdhbCBleHRyYWN0OiB7ZX0iKSkKICAgIHJldHVybiBzYXZlZAoKZGVmIGh1bWFuX3NpemUobnVtKToKICAgIGZvciB1bml0IGluICgiQiIsICJLQiIsICJNQiIsICJHQiIpOgogICAgICAgIGlmIGFicyhudW0pIDwgMTAyNC4wOgogICAgICAgICAgICByZXR1cm4gZiJ7bnVtIGlmIHVuaXQgPT0gJ0InIGVsc2UgZid7bnVtOi4xZn0nfSB7dW5pdH0iCiAgICAgICAgbnVtIC89IDEwMjQuMAogICAgcmV0dXJuIGYie251bTouMWZ9IFRCIgoKIyAtLS0tLS0tLS0tIGludGVyYWN0aXZlIC0tLS0tLS0tLS0KZGVmIGFza19rZXkoKToKICAgIGtleSA9IGdldF9hcGlfa2V5KCkKICAgIGlmIGtleToKICAgICAgICByZXR1cm4ga2V5CiAgICBwcmludCh5bCgiXG4gIPCflJEgQVBJIEtleSBTdWJTb3VyY2UgYmVsdW0gZGlrb25maWd1cmFzaS4iKSkKICAgIHByaW50KGRpbSgiICBDYXJhIGRhcGF0OlxuICAgIDEuIEJ1a2EgaHR0cHM6Ly9zdWJzb3VyY2UubmV0XG4gICAgMi4gTG9naW4gYXRhdSBidWF0IGFrdW5cbiAgICAzLiBNZW51IFByb2ZpbGUg4oaSIEFQSSBLZXlcbiAgICA0LiBTYWxpbiBrZXktbnlhIikpCiAgICBrZXkgPSBpbnB1dChjeSgiICBQYXN0ZSBBUEkgS2V5OiAiKSkuc3RyaXAoKQogICAgd2hpbGUgbm90IGtleToKICAgICAgICBwcmludChlcigiICBBUEkgS2V5IHRpZGFrIGJvbGVoIGtvc29uZy4iKSkKICAgICAgICBrZXkgPSBpbnB1dChjeSgiICBQYXN0ZSBBUEkgS2V5OiAiKSkuc3RyaXAoKQogICAgb3MuZW52aXJvblsnU1VCU09VUkNFX0FQSV9LRVknXSA9IGtleQogICAgc2F2ZV9hcGlfa2V5KGtleSkKICAgIHByaW50KG9rKCIgIEFQSSBLZXkgdGVyc2ltcGFuIChndW5ha2FuIHNlY3JldCBTVUJTT1VSQ0VfQVBJX0tFWSBiaWFyIHBlcm1hbmVuKS4iKSkKICAgIHJldHVybiBrZXkKCmRlZiBtYWluKCk6CiAgICBpZiBub3QgZ2V0X2FwaV9rZXkoKToKICAgICAgICBhc2tfa2V5KCkKICAgIHdoaWxlIFRydWU6CiAgICAgICAgY2koKQogICAgICAgIHByaW50KCdcbicgKyAnPScgKiA2MikKICAgICAgICBwcmludChjeSgnICAgaGFydS1zdWIgLS0gU3ViU291cmNlIFN1YnRpdGxlIERvd25sb2FkZXInKSkKICAgICAgICBwcmludChkaW0oJyAgIEFQSTogc3Vic291cmNlLm5ldCB8IFgtQVBJLUtleScpKQogICAgICAgIHByaW50KCc9JyAqIDYyKQogICAgICAgIHByaW50KCkKICAgICAgICBwcmludCgnICBDYXJpIHN1YnRpdGxlIGp1ZHVsIGZpbG0vc2VyaS4gQ29udG9oIHF1ZXJ5OicpCiAgICAgICAgcHJpbnQoJyAgICAiSW5jZXB0aW9uIiAgLyAgIlRoZSBCZWFyIDIwMjIiICAvICAiTmFydXRvIDIwMDIiJykKICAgICAgICBwcmludCgpCiAgICAgICAgcSA9IGlucHV0KCcgIEp1ZHVsIChlbnRlcj1rZWx1YXIpOiAnKS5zdHJpcCgpCiAgICAgICAgaWYgbm90IHE6CiAgICAgICAgICAgIHByaW50KCdcbiAgQnllIScpCiAgICAgICAgICAgIHJldHVybgogICAgICAgIG10eXBlID0gaW5wdXQoJyAgVGlwZSBbbW92aWUvc2VyaWVzL2FsbF0gKEVudGVyPWFsbCk6ICcpLnN0cmlwKCkubG93ZXIoKSBvciAnYWxsJwogICAgICAgIHllYXIgPSBOb25lCiAgICAgICAgeW0gPSByZS5zZWFyY2gocidcYigxOXwyMClcZHsyfVxiJywgcSkKICAgICAgICBpZiB5bToKICAgICAgICAgICAgeWVhciA9IHltLmdyb3VwKDApCiAgICAgICAgICAgIHEgPSBxLnJlcGxhY2UoeWVhciwgJycpLnN0cmlwKCcgLScpCiAgICAgICAgcHJpbnQoZidcbiAgTWVuY2FyaSAie3F9IicgKyAoZicgKHt5ZWFyfSknIGlmIHllYXIgZWxzZSAnJykgKyAnLi4uJykKICAgICAgICByZXN1bHRzID0gc2VhcmNoX21vdmllcyhxLCBtdHlwZSwgeWVhcikKICAgICAgICBpZiBub3QgcmVzdWx0czoKICAgICAgICAgICAgcHJpbnQoeWwoJyAgVGlkYWsgYWRhIGhhc2lsLicpKQogICAgICAgICAgICBpbnB1dCgnXG4gIEVudGVyLi4uJykKICAgICAgICAgICAgY29udGludWUKICAgICAgICBwcmludChmJ1xuICBEaXRlbXVrYW4ge2xlbihyZXN1bHRzKX0ganVkdWw6JykKICAgICAgICBmb3IgaSwgbSBpbiBlbnVtZXJhdGUocmVzdWx0cywgMSk6CiAgICAgICAgICAgIHN1YnMgPSBtLmdldCgnc3VidGl0bGVDb3VudCcpCiAgICAgICAgICAgIHN1YnNfcyA9IGYiICh7c3Vic30gc3ViKSIgaWYgc3VicyBpcyBub3QgTm9uZSBlbHNlICIiCiAgICAgICAgICAgIHByaW50KGYiICBbe2l9XSB7bS5nZXQoJ3RpdGxlJywnPycpfSAoe20uZ2V0KCdyZWxlYXNlWWVhcicsJz8nKX0pIFt7bS5nZXQoJ3R5cGUnLCc/Jyl9XXtzdWJzX3N9IikKICAgICAgICBzZWwgPSBpbnB1dChjeShmIlxuICBQaWxpaCBub21vciBbMS17bGVuKHJlc3VsdHMpfV06ICIpKS5zdHJpcCgpCiAgICAgICAgaWYgbm90IHNlbC5sc3RyaXAoJy0nKS5pc2RpZ2l0KCkgb3Igbm90ICgxIDw9IGludChzZWwpIDw9IGxlbihyZXN1bHRzKSk6CiAgICAgICAgICAgIHByaW50KGVyKCcgIFBpbGloYW4gdGlkYWsgdmFsaWQuJykpCiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgbW92aWUgPSByZXN1bHRzW2ludChzZWwpIC0gMV0KICAgICAgICBtaWQgPSBtb3ZpZS5nZXQoJ21vdmllSWQnKQogICAgICAgIHRpdGxlID0gc2FuaXRpemUobW92aWUuZ2V0KCd0aXRsZScpIG9yICdUaXRsZScpCiAgICAgICAgcHJpbnQoZiJcbiAg8J+TiyBKdWR1bDoge21vdmllLmdldCgndGl0bGUnKX0gKHttb3ZpZS5nZXQoJ3JlbGVhc2VZZWFyJyl9KSIpCiAgICAgICAgbGFuZ19xID0gaW5wdXQoJyAgQmFoYXNhIChrb3Nvbmc9c2VtdWEsIGNvbnRvaDogaWQvZW5nbGlzaC9pbmRvbmVzaWFuKSBbaWRdOiAnKS5zdHJpcCgpLmxvd2VyKCkgb3IgJ2lkJwogICAgICAgIGxhbmdfbmFtZSA9IE5vbmUKICAgICAgICBpZiBsYW5nX3E6CiAgICAgICAgICAgIGxhbmdfbmFtZSA9IExBTkdfTUFQLmdldChsYW5nX3EsIGxhbmdfcSBpZiBsZW4obGFuZ19xKSA+IDMgZWxzZSBMQU5HX01BUC5nZXQobGFuZ19xLCBOb25lKSkKICAgICAgICBwcmludCgnICBNZW5nYW1iaWwgZGFmdGFyIHN1YnRpdGxlLi4uJykKICAgICAgICBzdWJzID0gZ2V0X3N1YnRpdGxlcyhtaWQsIGxhbmdfbmFtZSkKICAgICAgICBpZiBub3Qgc3ViczoKICAgICAgICAgICAgcHJpbnQoeWwoJyAgVGlkYWsgYWRhIHN1YnRpdGxlIHRlcnNlZGlhLicpKQogICAgICAgICAgICBpbnB1dCgnXG4gIEVudGVyLi4uJykKICAgICAgICAgICAgY29udGludWUKICAgICAgICB1cF9pZHMgPSBzb3J0ZWQoe3N0cihzLmdldCgndXBsb2FkZXJJZCcpIG9yICc/JykgZm9yIHMgaW4gc3Vic30pCiAgICAgICAgcHJpbnQoZidcbiAgU3VidGl0bGUgdGVyc2VkaWEgKHtsZW4oc3Vicyl9IHN1Yiwge2xlbih1cF9pZHMpfSB1cGxvYWRlcik6JykKICAgICAgICBmb3IgaSwgcyBpbiBlbnVtZXJhdGUoc3VicywgMSk6CiAgICAgICAgICAgIHJlbCA9ICIsICIuam9pbihzLmdldCgncmVsZWFzZUluZm8nKSBvciBbXSlbOjM4XSBvciAnLScKICAgICAgICAgICAgZGwgPSBzLmdldCgnZG93bmxvYWRzJykgb3IgMAogICAgICAgICAgICByYXRlID0gKHMuZ2V0KCdyYXRpbmcnKSBvciB7fSkuZ2V0KCdnb29kJykKICAgICAgICAgICAgcmF0ZV9zID0gZiLirZB7cmF0ZX0iIGlmIHJhdGUgZWxzZSAnJwogICAgICAgICAgICBoaSA9ICdISScgaWYgcy5nZXQoJ2hlYXJpbmdJbXBhaXJlZCcpIGVsc2UgJ05vJwogICAgICAgICAgICBzeiA9IGh1bWFuX3NpemUocy5nZXQoJ3NpemUnKSBvciAwKSBpZiBzLmdldCgnc2l6ZScpIGVsc2UgJycKICAgICAgICAgICAgbGFuZyA9IHN0cihzLmdldCgnbGFuZ3VhZ2UnKSBvciAnJykKICAgICAgICAgICAgdXBkID0gZiJVOntzLmdldCgndXBsb2FkZXJJZCcpfSIgaWYgcy5nZXQoJ3VwbG9hZGVySWQnKSBlbHNlICcnCiAgICAgICAgICAgIHByaW50KGYiICBbe2l9XSB7cmVsOjwzOH0ge2xhbmc6PDEwfSB7c3o6Pjd9IERMOntkbDo8NX0ge3JhdGVfczo8NH0gSEk6e2hpfSB7dXBkfSIpCiAgICAgICAgICAgIGNtdCA9IChzLmdldCgnY29tbWVudGFyeScpIG9yICcnKS5zdHJpcCgpCiAgICAgICAgICAgIGlmIGNtdDoKICAgICAgICAgICAgICAgIGNtdDEgPSBjbXQucmVwbGFjZSgnXG4nLCAnICcpLnN0cmlwKClbOjYwXQogICAgICAgICAgICAgICAgcHJpbnQoZiIgICAgICAgIPCfkqwge2RpbShjbXQxKX0iKQogICAgICAgIHNlbDIgPSBpbnB1dChjeShmIlxuICBQaWxpaCBub21vciBzdWJ0aXRsZSBbMS17bGVuKHN1YnMpfV06ICIpKS5zdHJpcCgpCiAgICAgICAgaWYgbm90IHNlbDIubHN0cmlwKCctJykuaXNkaWdpdCgpIG9yIG5vdCAoMSA8PSBpbnQoc2VsMikgPD0gbGVuKHN1YnMpKToKICAgICAgICAgICAgcHJpbnQoZXIoJyAgUGlsaWhhbiB0aWRhayB2YWxpZC4nKSkKICAgICAgICAgICAgY29udGludWUKICAgICAgICBzdWIgPSBzdWJzW2ludChzZWwyKSAtIDFdCiAgICAgICAgc2lkID0gc3ViLmdldCgnc3VidGl0bGVJZCcpCiAgICAgICAgZGV0YWlsID0gZ2V0X3N1YnRpdGxlX2RldGFpbChzaWQpIG9yIHN1YgogICAgICAgIGNtdCA9IChkZXRhaWwuZ2V0KCdjb21tZW50YXJ5Jykgb3IgJycpLnN0cmlwKCkKICAgICAgICBwcmV2ID0gKGRldGFpbC5nZXQoJ3ByZXZpZXcnKSBvciAnJykuc3RyaXAoKQogICAgICAgIHByaW50KCdcbicgKyAnLScgKiA2MikKICAgICAgICBwcmludChjeSgiICBEZXRhaWwgc3VidGl0bGUgIyVzIiAlIHNpZCkpCiAgICAgICAgcHJpbnQoJy0nICogNjIpCiAgICAgICAgcHJpbnQoZiIgIFJlbGVhc2UgICA6IHsnLCAnLmpvaW4oZGV0YWlsLmdldCgncmVsZWFzZUluZm8nKSBvciBbXSl9IikKICAgICAgICBwcmludChmIiAgTGFuZ3VhZ2UgIDoge2RldGFpbC5nZXQoJ2xhbmd1YWdlJyl9IikKICAgICAgICBwcmludChmIiAgVXBsb2FkZXIgIDoge2RldGFpbC5nZXQoJ3VwbG9hZGVySWQnKX0iKQogICAgICAgIHByaW50KGYiICBGcmFtZXJhdGUgOiB7ZGV0YWlsLmdldCgnZnJhbWVyYXRlJykgb3IgJy0nfSIpCiAgICAgICAgcHJpbnQoZiIgIFRpcGUgICAgICA6IHsoZGV0YWlsLmdldCgncHJvZHVjdGlvblR5cGUnKSBvciAnJyl9L3tkZXRhaWwuZ2V0KCdyZWxlYXNlVHlwZScpIG9yICctJ30iKQogICAgICAgIHByaW50KGYiICBEb3dubG9hZHMgOiB7ZGV0YWlsLmdldCgnZG93bmxvYWRzJykgb3IgMH0gICBVa3VyYW46IHtodW1hbl9zaXplKGRldGFpbC5nZXQoJ3NpemUnKSBvciAwKSBpZiBkZXRhaWwuZ2V0KCdzaXplJykgZWxzZSAnLSd9IikKICAgICAgICBpZiBjbXQ6CiAgICAgICAgICAgIHByaW50KGYiXG4gIPCfkqwge2NtdH0iKQogICAgICAgIGlmIHByZXY6CiAgICAgICAgICAgIHByZXZfbGluZXMgPSBbbCBmb3IgbCBpbiBwcmV2LnNwbGl0bGluZXMoKSBpZiBsLnN0cmlwKCldWzoxMl0KICAgICAgICAgICAgcHJpbnQoJ1xuICDwn5GBIFByZXZpZXc6JykKICAgICAgICAgICAgZm9yIGwgaW4gcHJldl9saW5lczoKICAgICAgICAgICAgICAgIHByaW50KGRpbSgnICAgIHwgJyArIGwpKQogICAgICAgIHByaW50KCctJyAqIDYyKQogICAgICAgIHAgPSBpbnB1dCh5bCgnICBMYW5qdXQgZG93bmxvYWQgc3VidGl0bGUgaW5pPyBbWS9uXTogJykpLnN0cmlwKCkubG93ZXIoKQogICAgICAgIGlmIHAgbm90IGluICgneScsICd5ZXMnLCAnJyk6CiAgICAgICAgICAgIHByaW50KGRpbSgnICBEaWJhdGFsa2FuLicpKQogICAgICAgICAgICBpbnB1dCgnXG4gIEVudGVyLi4uJykKICAgICAgICAgICAgY29udGludWUKICAgICAgICBwcmludChmIiAg4qyHIERvd25sb2FkIHN1YnRpdGxlICN7c2lkfS4uLiIpCiAgICAgICAgZGF0YSA9IGRvd25sb2FkX3N1YnRpdGxlKHNpZCkKICAgICAgICBpZiBub3QgZGF0YToKICAgICAgICAgICAgaW5wdXQoJ1xuICBFbnRlci4uLicpCiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgb3V0X2RpciA9IE9VVF9ESVIKICAgICAgICBvdXRfZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgICAgICBiYXNlID0gZiJ7dGl0bGV9Lnttb3ZpZS5nZXQoJ3JlbGVhc2VZZWFyJywnJyl9Ii5zdHJpcCgnLicpCiAgICAgICAgaWYgbW92aWUuZ2V0KCd0eXBlJykgPT0gJ3NlcmllcycgYW5kIG1vdmllLmdldCgnc2Vhc29uJyk6CiAgICAgICAgICAgIGJhc2UgKz0gZiIuU3ttb3ZpZS5nZXQoJ3NlYXNvbicpfSIKICAgICAgICBzYXZlZCA9IGV4dHJhY3Rfc3VidGl0bGVzKGRhdGEsIG91dF9kaXIsIGJhc2UpCiAgICAgICAgaWYgc2F2ZWQ6CiAgICAgICAgICAgIHByaW50KG9rKGYiXG4gIOKchSBUZXJzaW1wYW46IHtsZW4oc2F2ZWQpfSBmaWxlIC0+IHtvdXRfZGlyfSIpKQogICAgICAgICAgICBmb3IgcyBpbiBzYXZlZDoKICAgICAgICAgICAgICAgIHByaW50KGRpbSgnICAgIOKAoiAnICsgcy5uYW1lKSkKICAgICAgICBlbHNlOgogICAgICAgICAgICAjIGdhZ2FsIHVuemlwIC0+IHNpbXBhbiBtZW50YWgKICAgICAgICAgICAgcmF3X3BhdGggPSBvdXRfZGlyIC8gZiJ7YmFzZX0uemlwIgogICAgICAgICAgICByYXdfcGF0aC53cml0ZV9ieXRlcyhkYXRhKQogICAgICAgICAgICBwcmludChvayhmIiAgRmlsZSB6aXAgdGVyc2ltcGFuOiB7cmF3X3BhdGh9IikpCiAgICAgICAgdG9rLCBvaWQgPSB0Z19jcmVkZW50aWFscygpCiAgICAgICAgaWYgc2F2ZWQgYW5kIHRvayBhbmQgb2lkOgogICAgICAgICAgICBwID0gaW5wdXQoeWwoJyAgS2lyaW0gc3VidGl0bGUga2UgYm90IFRlbGVncmFtPyBbeS9OXTogJykpLnN0cmlwKCkubG93ZXIoKQogICAgICAgICAgICBpZiBwIGluICgneScsICd5ZXMnKToKICAgICAgICAgICAgICAgIGZvciBzIGluIHNhdmVkOgogICAgICAgICAgICAgICAgICAgIHRnX3NlbmRfZG9jdW1lbnQocywgZiJoYXJ1LXN1Ylxue2Jhc2V9IikKICAgICAgICBpbnB1dCgnXG4gIEVudGVyLi4uJykKCgppZiBfX25hbWVfXyA9PSAnX19tYWluX18nOgogICAgdHJ5OgogICAgICAgIG1haW4oKQogICAgZXhjZXB0IEtleWJvYXJkSW50ZXJydXB0OgogICAgICAgIHByaW50KCdcbiAgRGliYXRhbGthbi4nKQ==""",
        'haru-cookie': """IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJoYXJ1LWNvb2tpZTogYW1iaWwgY29va2llIENsb3VkZmxhcmUgKGNmX2NsZWFyYW5jZSkgaGVudGFpcmVhZCB2aWEgQ2hyb21lCmxva2FsIChyZW1vdGUgZGVidWdnaW5nKSBsYWx1IHR1bGlzaSBjb29raWVzLnR4dCAoTmV0c2NhcGUpICsgTUFOR0FDT09LSUUvVUEKa2Ugfi8uaGFydV9zZWNyZXRzLmpzb24gKGF0YXUgL2NvbnRlbnQvLmhhcnVfc2VjcmV0cy5qc29uIGJpbGEgZGkgQ29sYWIpLgoKSmFsYW5rYW4gZGkgUEMga2FtdSAoYnVrYW4gQ29sYWIpOgogICAgcGlwIGluc3RhbGwgd2Vic29ja2V0LWNsaWVudAogICAgcHl0aG9uIGhhcnUtY29va2llLnB5ICAgICAgICAgICAgIyBwYWthaSBDaHJvbWUgbm9ybWFsIChtdW5jdWwgamVuZGVsYTsga2xpayBWZXJpZnkgYmlsYSBtdW5jdWwpCk9wc2lvbmFsOiBzZXQgSEFSVV9DSFJPTUUga2UgcGF0aCBjaHJvbWUuZXhlIGJpbGEgdGlkYWsgdGVyZGV0ZWtzaS4KIiIiCmltcG9ydCBqc29uLCBvcywgcmUsIHNvY2tldCwgc3VicHJvY2Vzcywgc3lzLCB0ZW1wZmlsZSwgdGltZSwgdXJsbGliLnJlcXVlc3QKClNJVEUgPSAiaHR0cHM6Ly9oZW50YWlyZWFkLmNvbSIKRE9NQUlOID0gImhlbnRhaXJlYWQiCgpkZWYgZmluZF9jaHJvbWUoKToKICAgIGMgPSBvcy5lbnZpcm9uLmdldCgiSEFSVV9DSFJPTUUiKSBvciBvcy5lbnZpcm9uLmdldCgiQ0hST01FX1BBVEgiKQogICAgaWYgYyBhbmQgb3MucGF0aC5leGlzdHMoYyk6CiAgICAgICAgcmV0dXJuIGMKICAgIGNhbmRzID0gW10KICAgIGZvciBiYXNlIGluIChvcy5lbnZpcm9uLmdldCgiUHJvZ3JhbUZpbGVzIiwgciJDOlxQcm9ncmFtIEZpbGVzIiksCiAgICAgICAgICAgICAgICAgb3MuZW52aXJvbi5nZXQoIlByb2dyYW1GaWxlcyh4ODYpIiwgciJDOlxQcm9ncmFtIEZpbGVzICh4ODYpIikpOgogICAgICAgIGNhbmRzICs9IFsKICAgICAgICAgICAgYmFzZSArIHIiXEdvb2dsZVxDaHJvbWVcQXBwbGljYXRpb25cY2hyb21lLmV4ZSIsCiAgICAgICAgICAgIGJhc2UgKyByIlxHb29nbGVcQ2hyb21lIEJldGFcQXBwbGljYXRpb25cY2hyb21lLmV4ZSIsCiAgICAgICAgICAgIGJhc2UgKyByIlxNaWNyb3NvZnRcRWRnZVxBcHBsaWNhdGlvblxtc2VkZ2UuZXhlIiwKICAgICAgICBdCiAgICBmb3IgcCBpbiBjYW5kczoKICAgICAgICBpZiBvcy5wYXRoLmV4aXN0cyhwKToKICAgICAgICAgICAgcmV0dXJuIHAKICAgIGZvciBuYW1lIGluICgiZ29vZ2xlLWNocm9tZSIsICJjaHJvbWl1bSIsICJjaHJvbWl1bS1icm93c2VyIiwgIm1zZWRnZSIsICJjaHJvbWUiKToKICAgICAgICBwID0gc3VicHJvY2Vzcy5ydW4oWyJ3aGVyZSIsIG5hbWVdLCBjYXB0dXJlX291dHB1dD1UcnVlLCB0ZXh0PVRydWUpLnN0ZG91dC5zdHJpcCgpIGlmIG9zLm5hbWUgPT0gIm50IiBlbHNlIE5vbmUKICAgICAgICBpZiBwOgogICAgICAgICAgICBjYW5kaWRhdGUgPSBwLnNwbGl0bGluZXMoKVswXQogICAgICAgICAgICBpZiBvcy5wYXRoLmV4aXN0cyhjYW5kaWRhdGUpOgogICAgICAgICAgICAgICAgcmV0dXJuIGNhbmRpZGF0ZQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIGltcG9ydCBzaHV0aWwKICAgICAgICAgICAgcSA9IHNodXRpbC53aGljaChuYW1lKQogICAgICAgICAgICBpZiBxOgogICAgICAgICAgICAgICAgcmV0dXJuIHEKICAgIHJldHVybiBOb25lCgpjbGFzcyBXUzoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCB1cmwpOgogICAgICAgIGltcG9ydCB3ZWJzb2NrZXQKICAgICAgICBzZWxmLndzID0gd2Vic29ja2V0LmNyZWF0ZV9jb25uZWN0aW9uKHVybCwgdGltZW91dD0xNSkKICAgICAgICBzZWxmLl9pZCA9IDAKICAgIGRlZiBjYWxsKHNlbGYsIG1ldGhvZCwgcGFyYW1zPU5vbmUsIHRpbWVvdXQ9MjApOgogICAgICAgIHNlbGYuX2lkICs9IDEKICAgICAgICBtaWQgPSBzZWxmLl9pZAogICAgICAgIHNlbGYud3Muc2VuZChqc29uLmR1bXBzKHsiaWQiOiBtaWQsICJtZXRob2QiOiBtZXRob2QsICJwYXJhbXMiOiBwYXJhbXMgb3Ige319KSkKICAgICAgICB3aGlsZSBUcnVlOgogICAgICAgICAgICBtc2cgPSBqc29uLmxvYWRzKHNlbGYud3MucmVjdigpKQogICAgICAgICAgICBpZiBtc2cuZ2V0KCJpZCIpID09IG1pZDoKICAgICAgICAgICAgICAgIHJldHVybiBtc2cKICAgIGRlZiBjbG9zZShzZWxmKToKICAgICAgICB0cnk6CiAgICAgICAgICAgIHNlbGYud3MuY2xvc2UoKQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgIHBhc3MKCmRlZiBnZXRfZnJlZV9wb3J0KCk6CiAgICBzID0gc29ja2V0LnNvY2tldCgpCiAgICBzLmJpbmQoKCIxMjcuMC4wLjEiLCAwKSkKICAgIHBvcnQgPSBzLmdldHNvY2tuYW1lKClbMV0KICAgIHMuY2xvc2UoKQogICAgcmV0dXJuIHBvcnQKCmRlZiBtYWluKCk6CiAgICB0cnk6CiAgICAgICAgaW1wb3J0IHdlYnNvY2tldCAgIyBub3FhCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIHByaW50KCJCdXR1aCB3ZWJzb2NrZXQtY2xpZW50LiBKYWxhbmthbjogIHBpcCBpbnN0YWxsIHdlYnNvY2tldC1jbGllbnQiKQogICAgICAgIHN5cy5leGl0KDEpCiAgICBpZiBvcy5wYXRoLmV4aXN0cygiL2NvbnRlbnQiKToKICAgICAgICBwcmludCgiKENhdGF0YW46IGthbXUgamFsYW4gZGFyaSBDb2xhYiDigJQgQ2hyb21lIGRpIFBDIGthbXUgeWFuZyBkaXBha2FpLCIKICAgICAgICAgICAgICAiIGJ1a2FuIGRpIHNpbmkuIEthbGF1IGRpIHNpbmkgZGlwYWthaSwgYmlhc2FueWEga2VuYSBtYW51YWwgY2FwdGNoYS4pIikKICAgIGNocm9tZSA9IGZpbmRfY2hyb21lKCkKICAgIGlmIG5vdCBjaHJvbWU6CiAgICAgICAgcHJpbnQoIkNocm9tZS9FZGdlIHRpZGFrIGRpdGVtdWthbi4gSW5zdGFsbCBDaHJvbWUgYXRhdSBzZXQgSEFSVV9DSFJPTUUiKQogICAgICAgIHN5cy5leGl0KDEpCiAgICBwcmludCgiQ2hyb21lOiIsIGNocm9tZSkKICAgIHBvcnQgPSBnZXRfZnJlZV9wb3J0KCkKICAgIHByb2YgPSB0ZW1wZmlsZS5ta2R0ZW1wKHByZWZpeD0iaGFydS1jZi0iKQogICAgYXJncyA9IFtjaHJvbWUsICItLXJlbW90ZS1kZWJ1Z2dpbmctcG9ydD0lZCIgJSBwb3J0LAogICAgICAgICAgICAiLS11c2VyLWRhdGEtZGlyPSIgKyBwcm9mLCAiLS1uby1maXJzdC1ydW4iLAogICAgICAgICAgICAiLS1uby1kZWZhdWx0LWJyb3dzZXItY2hlY2siLCAiLS1kaXNhYmxlLWV4dGVuc2lvbnMiXQogICAgaWYgb3MuZW52aXJvbi5nZXQoIkhBUlVfSEVBRExFU1MiKToKICAgICAgICBhcmdzLmluc2VydCgxLCAiLS1oZWFkbGVzcz1uZXciKQogICAgcHJvYyA9IHN1YnByb2Nlc3MuUG9wZW4oYXJncywgc3Rkb3V0PXN1YnByb2Nlc3MuREVWTlVMTCwgc3RkZXJyPXN1YnByb2Nlc3MuREVWTlVMTCkKICAgIHRyeToKICAgICAgICB0YXJnZXRzID0gTm9uZQogICAgICAgIGVuZCA9IHRpbWUudGltZSgpICsgMjAKICAgICAgICB3aGlsZSB0aW1lLnRpbWUoKSA8IGVuZDoKICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgd2l0aCB1cmxsaWIucmVxdWVzdC51cmxvcGVuKCJodHRwOi8vMTI3LjAuMC4xOiVkL2pzb24iICUgcG9ydCwgdGltZW91dD0zKSBhcyByOgogICAgICAgICAgICAgICAgICAgIHRhcmdldHMgPSBqc29uLmxvYWRzKHIucmVhZCgpKQogICAgICAgICAgICAgICAgaWYgdGFyZ2V0czoKICAgICAgICAgICAgICAgICAgICBicmVhawogICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICAgICAgcGFzcwogICAgICAgICAgICB0aW1lLnNsZWVwKDAuNSkKICAgICAgICBpZiBub3QgdGFyZ2V0czoKICAgICAgICAgICAgcHJpbnQoIkdhZ2FsIGNvbm5lY3Qga2UgQ2hyb21lIGRlYnVnLiBQYXN0aWthbiB0aWRhayBhZGEgQ2hyb21lIGxhaW4geWciCiAgICAgICAgICAgICAgICAgICIga2VwYWthaSBwcm9maWxlIGluaS4iKQogICAgICAgICAgICBzeXMuZXhpdCgxKQogICAgICAgIHBhZ2UgPSBuZXh0KCh0IGZvciB0IGluIHRhcmdldHMgaWYgdC5nZXQoInR5cGUiKSA9PSAicGFnZSIpLCBOb25lKQogICAgICAgIHVybCA9IHBhZ2VbIndlYlNvY2tldERlYnVnZ2VyVXJsIl0KICAgICAgICB3cyA9IFdTKHVybCkKICAgICAgICB3cy5jYWxsKCJQYWdlLmVuYWJsZSIpCiAgICAgICAgd3MuY2FsbCgiTmV0d29yay5lbmFibGUiKQogICAgICAgIHByaW50KCJCdWthIGNoYWxsZW5nZSBkaSBqZW5kZWxhIENocm9tZSAoaGVudGFpcmVhZCkuLi4iKQogICAgICAgIHdzLmNhbGwoIlBhZ2UubmF2aWdhdGUiLCB7InVybCI6IFNJVEV9LCB0aW1lb3V0PTEwKQogICAgICAgIGNvb2tpZSA9IE5vbmUKICAgICAgICB1YSA9IE5vbmUKICAgICAgICBlbmQgPSB0aW1lLnRpbWUoKSArIDI0MAogICAgICAgIGxhc3Rfc3RhdHVzID0gIiIKICAgICAgICB3aGlsZSB0aW1lLnRpbWUoKSA8IGVuZDoKICAgICAgICAgICAgdGltZS5zbGVlcCg0KQogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICByZXAgPSB3cy5jYWxsKCJOZXR3b3JrLmdldEFsbENvb2tpZXMiLCB0aW1lb3V0PTgpCiAgICAgICAgICAgICAgICBjb29raWVzID0gcmVwLmdldCgicmVzdWx0Iiwge30pLmdldCgiY29va2llcyIsIFtdKQogICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICAgICAgY29va2llcyA9IFtdCiAgICAgICAgICAgIGNhbmRzID0gW2MgZm9yIGMgaW4gY29va2llcyBpZiBET01BSU4gaW4gKGMuZ2V0KCJkb21haW4iKSBvciAiIikubG93ZXIoKV0KICAgICAgICAgICAgaGFzX2NmID0gYW55KGMuZ2V0KCJuYW1lIikgPT0gImNmX2NsZWFyYW5jZSIgYW5kIGMuZ2V0KCJ2YWx1ZSIpIGZvciBjIGluIGNhbmRzKQogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICB0ID0gd3MuY2FsbCgiUnVudGltZS5ldmFsdWF0ZSIsIHsiZXhwcmVzc2lvbiI6ICJkb2N1bWVudC50aXRsZSJ9LCB0aW1lb3V0PTgpCiAgICAgICAgICAgICAgICB0aXRsZSA9ICh0LmdldCgicmVzdWx0Iiwge30pLmdldCgicmVzdWx0Iiwge30pLmdldCgidmFsdWUiKSBvciAiIikKICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgICAgIHRpdGxlID0gIiIKICAgICAgICAgICAgc3RhdHVzID0gZiJjb29raWVzPXtsZW4oY2FuZHMpfSBjZl9jbGVhcmFuY2U9eydZRVMnIGlmIGhhc19jZiBlbHNlICdubyd9IHwge3RpdGxlWzo0MF19IgogICAgICAgICAgICBpZiBzdGF0dXMgIT0gbGFzdF9zdGF0dXM6CiAgICAgICAgICAgICAgICBwcmludCgiICAiLCBzdGF0dXMpCiAgICAgICAgICAgICAgICBsYXN0X3N0YXR1cyA9IHN0YXR1cwogICAgICAgICAgICBpZiBoYXNfY2Y6CiAgICAgICAgICAgICAgICBjb29raWUgPSBjYW5kcwogICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICBpZiBub3QgY29va2llOgogICAgICAgICAgICBwcmludCgiVGltZW91dCDigJQgY2ZfY2xlYXJhbmNlIGJlbHVtIG11bmN1bC4gQ29iYTogdHV0dXAgamVuZGVsYSBDaHJvbWUsIgogICAgICAgICAgICAgICAgICAiIGphbGFua2FuIHVsYW5nOyBrYWxhdSBtdW5jdWwgY2FwdGNoYSAnVmVyaWZ5IHlvdSBhcmUgaHVtYW4nIGtsaWsgbWFudWFsLiIpCiAgICAgICAgICAgIHN5cy5leGl0KDEpCiAgICAgICAgdHJ5OgogICAgICAgICAgICByZXAgPSB3cy5jYWxsKCJSdW50aW1lLmV2YWx1YXRlIiwKICAgICAgICAgICAgICAgICAgICAgICAgICB7ImV4cHJlc3Npb24iOiAibmF2aWdhdG9yLnVzZXJBZ2VudCJ9LCB0aW1lb3V0PTgpCiAgICAgICAgICAgIHVhID0gcmVwLmdldCgicmVzdWx0Iiwge30pLmdldCgicmVzdWx0Iiwge30pLmdldCgidmFsdWUiKSBvciAiIgogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgIHVhID0gIiIKICAgICAgICB3cy5jbG9zZSgpCiAgICAgICAgaGVhZGVyID0gIjsgIi5qb2luKCIlcz0lcyIgJSAoY1sibmFtZSJdLCBjWyJ2YWx1ZSJdKSBmb3IgYyBpbiBjb29raWUpCiAgICAgICAgbmV0c2NhcGUgPSBbIiMgTmV0c2NhcGUgSFRUUCBDb29raWUgRmlsZSDigJQgaGVudGFpcmVhZC5jb20gKGhhcnUtY29va2llKSIsCiAgICAgICAgICAgICAgICAgICAgIiMgZG9tYWluXFx0aW5jbHVkZVN1YmRvbWFpbnNcXHRwYXRoXFx0c2VjdXJlXFx0ZXhwaXJlc1xcdG5hbWVcXHR2YWx1ZSJdCiAgICAgICAgZm9yIGMgaW4gY29va2llOgogICAgICAgICAgICBleHBpcnkgPSBpbnQoYy5nZXQoImV4cGlyZXMiKSBvciAwKQogICAgICAgICAgICBuZXRzY2FwZS5hcHBlbmQoIlx0Ii5qb2luKFsKICAgICAgICAgICAgICAgIChjLmdldCgiZG9tYWluIikgb3IgIiIpLCAiVFJVRSIsIChjLmdldCgicGF0aCIpIG9yICIvIiksCiAgICAgICAgICAgICAgICAiVFJVRSIgaWYgYy5nZXQoInNlY3VyZSIpIGVsc2UgIkZBTFNFIiwgc3RyKGV4cGlyeSksCiAgICAgICAgICAgICAgICBjLmdldCgibmFtZSIsICIiKSwgYy5nZXQoInZhbHVlIiwgIiIpXSkpCiAgICAgICAgbmV0ID0gIlxuIi5qb2luKG5ldHNjYXBlKSArICJcbiIKICAgICAgICBvdXQgPSBbXQogICAgICAgIGlmIG9zLnBhdGguZXhpc3RzKCIvY29udGVudCIpOgogICAgICAgICAgICBvdXQuYXBwZW5kKCIvY29udGVudC9kb3dubG9hZHMvaGVudGFpcmVhZF9jb29raWVzLnR4dCIpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgb3V0LmFwcGVuZChvcy5wYXRoLmpvaW4ob3MucGF0aC5leHBhbmR1c2VyKCJ+IiksICJoZW50YWlyZWFkX2Nvb2tpZXMudHh0IikpCiAgICAgICAgZm9yIGZwIGluIG91dDoKICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgb3MubWFrZWRpcnMob3MucGF0aC5kaXJuYW1lKGZwKSwgZXhpc3Rfb2s9VHJ1ZSkKICAgICAgICAgICAgICAgIG9wZW4oZnAsICJ3IiwgZW5jb2Rpbmc9InV0Zi04Iikud3JpdGUobmV0KQogICAgICAgICAgICAgICAgcHJpbnQoIiAgY29va2llcy50eHQgIC0+IiwgZnApCiAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICAgICAgICAgIHByaW50KCIgIGdhZ2FsIHR1bGlzIiwgZnAsIGUpCiAgICAgICAgc2VjcmV0cyA9IHt9CiAgICAgICAgZm9yIHNwIGluICgiL2NvbnRlbnQvLmhhcnVfc2VjcmV0cy5qc29uIiwKICAgICAgICAgICAgICAgICAgIG9zLnBhdGguam9pbihvcy5wYXRoLmV4cGFuZHVzZXIoIn4iKSwgIi5oYXJ1X3NlY3JldHMuanNvbiIpKToKICAgICAgICAgICAgaWYgb3MucGF0aC5leGlzdHMoc3ApOgogICAgICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgICAgIHNlY3JldHMgPSBqc29uLmxvYWQob3BlbihzcCkpCiAgICAgICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICAgICAgICAgIHNlY3JldHMgPSB7fQogICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICBzZWNyZXRzWyJNQU5HQUNPT0tJRSJdID0gaGVhZGVyCiAgICAgICAgaWYgdWE6CiAgICAgICAgICAgIHNlY3JldHNbIk1BTkdBQ09PS0lFX1VBIl0gPSB1YQogICAgICAgIGZvciBzcCBpbiAoIi9jb250ZW50Ly5oYXJ1X3NlY3JldHMuanNvbiIsCiAgICAgICAgICAgICAgICAgICBvcy5wYXRoLmpvaW4ob3MucGF0aC5leHBhbmR1c2VyKCJ+IiksICIuaGFydV9zZWNyZXRzLmpzb24iKSk6CiAgICAgICAgICAgIGlmIG9zLnBhdGguZXhpc3RzKHNwKSBvciBub3Qgb3MucGF0aC5leGlzdHMoIi9jb250ZW50Iik6CiAgICAgICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICAgICAganNvbi5kdW1wKHNlY3JldHMsIG9wZW4oc3AsICJ3IiksIGluZGVudD0xKQogICAgICAgICAgICAgICAgICAgIHByaW50KCIgIHNlY3JldHMgICAgICAtPiIsIHNwKQogICAgICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgICAgICAgICAgICAgcHJpbnQoIiAgZ2FnYWwgdHVsaXMiLCBzcCwgZSkKICAgICAgICBvcy5lbnZpcm9uWyJNQU5HQUNPT0tJRSJdID0gaGVhZGVyCiAgICAgICAgaWYgdWE6CiAgICAgICAgICAgIG9zLmVudmlyb25bIk1BTkdBQ09PS0lFX1VBIl0gPSB1YQogICAgICAgIHByaW50KCJcbj09PT09PT09PT0gU0FMSU4gSU5JIEtFIENPTEFCIFNFQ1JFVFMgPT09PT09PT09PSIpCiAgICAgICAgcHJpbnQoIk1BTkdBQ09PS0lFOlxuIiArIGhlYWRlcikKICAgICAgICBwcmludCgiXG5NQU5HQUNPT0tJRV9VQTpcbiIgKyAodWEgb3IgIihrb3Nvbmcg4oCUIHBha2FpIGRlZmF1bHQpIikpCiAgICAgICAgcHJpbnQoIlxuQXRhdSBidWthIGZpbGUgY29va2llcy50eHQgZGkgYXRhcyBsYWx1IHRlbXBlbCBpc2lueWEga2UgTUFOR0FDT09LSUUuIikKICAgICAgICBwcmludCgiKENvb2tpZSB0ZXJpa2F0IElQICsgVUE6IGJlcmxha3UgdW50dWsgSVAgeWcgc2FtYS4gRGkgQ29sYWIgYmlzYSBnYWdhbDsiKQogICAgICAgIHByaW50KCIgamFsYW5rYW4gaGFydS1tYW5nYSBkYXJpIFBDIGluaSBrYWxhdSBiZWdpdHUuKSIpCiAgICBmaW5hbGx5OgogICAgICAgIHRyeToKICAgICAgICAgICAgcHJvYy50ZXJtaW5hdGUoKQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgIHBhc3MKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCk=""",
        'haru-mux': """...""",
        'haru-extract': """...""",
        'haru-metadata': """...""",
        'haru-download': """IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwppbXBvcnQgc3VicHJvY2VzcyxzeXMsb3MscmUsZ2xvYixqc29uLHRpbWUKaW1wb3J0IHJlcXVlc3RzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAppbXBvcnQgdXJsbGliLnBhcnNlCmltcG9ydCBoYXNobGliCgpWPXsnLm1rdicsJy5tcDQnLCcuYXZpJywnLm1vdicsJy53ZWJtJywnLmZsdicsJy53bXYnLCcudHMnLCcubTR2J30KQT17Jy5tcDMnLCcuYWFjJywnLmZsYWMnLCcud2F2JywnLm9nZycsJy5vcHVzJywnLm1rYScsJy5hYzMnLCcuZHRzJywnLmVhYzMnLCcubTRhJ30KUz17Jy5zcnQnLCcuYXNzJywnLnNzYScsJy5zdWInLCcuaWR4JywnLnN1cCcsJy52dHQnLCcucGdzJywnLnNjYycsJy5zYW1pJ30KTD17J2lkJzonSW5kb25lc2lhbicsJ2VuJzonRW5nbGlzaCcsJ2phJzonSmFwYW5lc2UnLCdrbyc6J0tvcmVhbicsJ3poJzonQ2hpbmVzZScsJ21zJzonTWFsYXknLCdhcic6J0FyYWJpYycsJ2RlJzonR2VybWFuJywnZnInOidGcmVuY2gnLCdlcyc6J1NwYW5pc2gnLCdwdCc6J1BvcnR1Z3Vlc2UnLCdydSc6J1J1c3NpYW4nLCdpdCc6J0l0YWxpYW4nLCd0aCc6J1RoYWknLCd2aSc6J1ZpZXRuYW1lc2UnLCdoaSc6J0hpbmRpJywndW5kJzonVW5kZXRlcm1pbmVkJ30KVVBMT0FEPVBhdGgoJy9jb250ZW50L3VwbG9hZHMnKQpPVVRQVVQ9UGF0aCgnL2NvbnRlbnQvb3V0cHV0JykKVVBMT0FELm1rZGlyKGV4aXN0X29rPVRydWUpCk9VVFBVVC5ta2RpcihleGlzdF9vaz1UcnVlKQoKZGVmIGNpKCk6CiBpbXBvcnQgc3lzCiBzeXMuc3Rkb3V0LndyaXRlKCdceDFiWzJKXHgxYltIJykKIHN5cy5zdGRvdXQuZmx1c2goKQoKZGVmIG9rKHQpOnJldHVybiAnXDAzM1s5Mm0nK3QrJ1wwMzNbMG0nCmRlZiBlcih0KTpyZXR1cm4gJ1wwMzNbOTFtJyt0KydcMDMzWzBtJwpkZWYgZGltKHQpOnJldHVybiAnXDAzM1s5MG0nK3QrJ1wwMzNbMG0nCmRlZiBoZHIodGl0bGUpOnByaW50KCdcbicrJz0nKjYyKTtwcmludCgnICAnK3RpdGxlKTtwcmludCgnPScqNjIpCgpkZWYgbG9hZF9zZWNyZXRzKCk6CiB0cnk6CiAgaWYgb3MucGF0aC5leGlzdHMoJy9jb250ZW50Ly5oYXJ1X3NlY3JldHMuanNvbicpOgogICBkPWpzb24ubG9hZChvcGVuKCcvY29udGVudC8uaGFydV9zZWNyZXRzLmpzb24nKSkKICAgZm9yIGssdiBpbiBkLml0ZW1zKCk6CiAgICBpZiB2IGFuZCBub3Qgb3MuZW52aXJvbi5nZXQoayk6b3MuZW52aXJvbltrXT1zdHIodikKIGV4Y2VwdDpwYXNzCgpkZWYgZ2V0X3NlY3JldChrKToKIHY9b3MuZW52aXJvbi5nZXQoaywnJykKIGlmIHY6cmV0dXJuIHYuc3RyaXAoKQogdHJ5OgogIGZyb20gZ29vZ2xlLmNvbGFiIGltcG9ydCB1c2VyZGF0YQogIHQ9dXNlcmRhdGEuZ2V0KGspCiAgaWYgdDpyZXR1cm4gc3RyKHQpLnN0cmlwKCkKIGV4Y2VwdDpwYXNzCiByZXR1cm4gJycKCiMg4pSA4pSA4pSAIEdPRklMRSBET1dOTE9BREVSIOKUgOKUgOKUgApkZWYgZ2V0X2dvZmlsZV90b2tlbigpOgogcmV0dXJuIGdldF9zZWNyZXQoJ0dPRklMRV9BUElfVE9LRU4nKQoKZGVmIGdvZmlsZV93dChhZ2VudCx0b2tlbik6CiBzbG90PXN0cihpbnQodGltZS50aW1lKCkpLy8xNDQwMCkKIHJldHVybiBoYXNobGliLnNoYTI1NigoYWdlbnQrJzo6ZW4tVVM6OicrdG9rZW4rJzo6JytzbG90Kyc6OjEyYWYwNTZkYWNlYTBiJykuZW5jb2RlKCkpLmhleGRpZ2VzdCgpCgpkZWYgZ29maWxlX2FwaV9nZW5lcmF0ZSh1cmwscGFzc3dvcmQsdG9rZW4pOgogcGF5bG9hZD17J3VybCc6dXJsLCdwYXNzd29yZCc6cGFzc3dvcmQsJ2V4cGlyZXNJblNlY29uZHMnOjM2MDAsJ2ZpbGVQYWdlJzowLCdmaWxlU2l6ZSc6MTAwfQogaGVhZGVycz17J0F1dGhvcml6YXRpb24nOidCZWFyZXIgJyt0b2tlbiwnQ29udGVudC1UeXBlJzonYXBwbGljYXRpb24vanNvbid9CiByPXJlcXVlc3RzLnBvc3QoJ2h0dHBzOi8vZ28uZmlsbWJlZWh1Yi53b3JrZXJzLmRldi9hcGkvdjEvZ2VuZXJhdGUnLGpzb249cGF5bG9hZCxoZWFkZXJzPWhlYWRlcnMsdGltZW91dD0zMCkKIHJldHVybiByLmpzb24oKQoKZGVmIGdvZmlsZV9hcGlfbGlzdCh1cmwscGFzc3dvcmQsdG9rZW4pOgogdHJ5OgogIHJlcz1nb2ZpbGVfYXBpX2dlbmVyYXRlKHVybCxwYXNzd29yZCx0b2tlbikKICBpZiBub3QgcmVzLmdldCgnb2snKTpyZXR1cm4gW10KICBkYXRhPXJlcy5nZXQoJ2RhdGEnLHt9KQogIGlmIGRhdGEuZ2V0KCdkb3dubG9hZExpbmtzJyk6cmV0dXJuIGRhdGFbJ2Rvd25sb2FkTGlua3MnXQogIHNoYXJlX3VybD1kYXRhLmdldCgnc2hhcmVVcmwnLCcnKQogIGlmIHNoYXJlX3VybDoKICAgc2lkPXNoYXJlX3VybC5yc3RyaXAoJy8nKS5zcGxpdCgnLycpWy0xXQogICBycj1yZXF1ZXN0cy5nZXQoJ2h0dHBzOi8vZ28uZmlsbWJlZWh1Yi53b3JrZXJzLmRldi9hcGkvZGF0YS8nK3NpZCxoZWFkZXJzPXsnVXNlci1BZ2VudCc6J01vemlsbGEvNS4wJ30sdGltZW91dD0zMCkKICAgZmQ9cnIuanNvbigpCiAgIG91dD1bXQogICBmb3IgZyBpbiBmZC5nZXQoJ2dyb3VwcycsW10pOm91dC5leHRlbmQoZy5nZXQoJ2ZpbGVzJyxbXSkpCiAgIHJldHVybiBvdXQKIGV4Y2VwdDpwYXNzCiByZXR1cm4gW10KCmRlZiBnb2ZpbGVfZGlyZWN0X2ZldGNoKHVybCxwYXNzd29yZD0nJyk6CiBtPXJlLnNlYXJjaChyJ2dvZmlsZVwuaW8vZC8oXHcrKScsdXJsKQogaWYgbm90IG06cmV0dXJuIE5vbmUsJ0xpbmsgYnVrYW4gZm9ybWF0IGdvZmlsZS5pby9kL3h4eCcsTm9uZQogY2lkPW0uZ3JvdXAoMSkKIHB3PWhhc2hsaWIuc2hhMjU2KHBhc3N3b3JkLmVuY29kZSgpKS5oZXhkaWdlc3QoKSBpZiBwYXNzd29yZCBlbHNlIE5vbmUKIGFnZW50PSdNb3ppbGxhLzUuMCAoV2luZG93cyBOVCAxMC4wOyBXaW42NDsgeDY0KSBBcHBsZVdlYktpdC81MzcuMzYgKEtIVE1MLCBsaWtlIEdlY2tvKSBDaHJvbWUvMTIwLjAuMC4wIFNhZmFyaS81MzcuMzYnCiBzPXJlcXVlc3RzLlNlc3Npb24oKQogcy5oZWFkZXJzLnVwZGF0ZSh7J0FjY2VwdC1FbmNvZGluZyc6J2d6aXAnLCdVc2VyLUFnZW50JzphZ2VudCwnQ29ubmVjdGlvbic6J2tlZXAtYWxpdmUnLCdBY2NlcHQnOicqLyonLCdPcmlnaW4nOidodHRwczovL2dvZmlsZS5pbycsJ1JlZmVyZXInOidodHRwczovL2dvZmlsZS5pby8nfSkKIHRvaz1nZXRfZ29maWxlX3Rva2VuKCkKIGlmIG5vdCB0b2s6CiAgdHJ5OgogICByPXMucG9zdCgnaHR0cHM6Ly9hcGkuZ29maWxlLmlvL2FjY291bnRzJyx0aW1lb3V0PTIwKQogICB0b2s9ci5qc29uKCkuZ2V0KCdkYXRhJyx7fSkuZ2V0KCd0b2tlbicpCiAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOgogICByZXR1cm4gTm9uZSwnR2FnYWwgbWVtYnVhdCBndWVzdCB0b2tlbjogJytzdHIoZSlbOjEwMF0sTm9uZQogaWYgbm90IHRvazpyZXR1cm4gTm9uZSwnR2FnYWwgbWVuZGFwYXRrYW4gdG9rZW4gZ29maWxlJyxOb25lCiBzLmNvb2tpZXMuc2V0KCdhY2NvdW50VG9rZW4nLHRvaykKIHMuaGVhZGVycy51cGRhdGUoeydBdXRob3JpemF0aW9uJzonQmVhcmVyICcrdG9rfSkKIGZpbGVzPVtdCiB0cnk6CiAgZGVmIHdhbGsoeCk6CiAgIHU9J2h0dHBzOi8vYXBpLmdvZmlsZS5pby9jb250ZW50cy8nK3grJz9jYWNoZT10cnVlJwogICBpZiBwdzp1Kz0nJnBhc3N3b3JkPScrcHcKICAgcj1zLmdldCh1LGhlYWRlcnM9eydYLVdlYnNpdGUtVG9rZW4nOmdvZmlsZV93dChhZ2VudCx0b2spLCdYLUJMJzonZW4tVVMnfSx0aW1lb3V0PTMwKQogICBkPXIuanNvbigpCiAgIGlmIGQuZ2V0KCdzdGF0dXMnKSE9J29rJzpyYWlzZSBFeGNlcHRpb24oc3RyKGQuZ2V0KCdzdGF0dXMnKSlbOjYwXSkKICAgZGF0YT1kLmdldCgnZGF0YScse30pCiAgIGlmIGRhdGEuZ2V0KCd0eXBlJykhPSdmb2xkZXInOgogICAgaWYgZGF0YS5nZXQoJ2xpbmsnKTpmaWxlcy5hcHBlbmQoeyduYW1lJzpkYXRhWyduYW1lJ10sJ3NpemUnOmRhdGEuZ2V0KCdzaXplJywwKSwnZG93bmxvYWRVcmwnOmRhdGFbJ2xpbmsnXX0pCiAgICByZXR1cm4KICAgZm9yIGNoIGluIChkYXRhLmdldCgnY2hpbGRyZW4nLHt9KSBvciB7fSkudmFsdWVzKCk6CiAgICBpZiBjaC5nZXQoJ3R5cGUnKT09J2ZvbGRlcic6d2FsayhjaFsnaWQnXSkKICAgIGVsaWYgY2guZ2V0KCdsaW5rJyk6ZmlsZXMuYXBwZW5kKHsnbmFtZSc6Y2hbJ25hbWUnXSwnc2l6ZSc6Y2guZ2V0KCdzaXplJywwKSwnZG93bmxvYWRVcmwnOmNoWydsaW5rJ119KQogIHdhbGsoY2lkKQogZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOgogIHJldHVybiBOb25lLCdMaXN0IGRpcmVjdCBnYWdhbDogJytzdHIoZSlbOjE1MF0sTm9uZQogcmV0dXJuIGZpbGVzLE5vbmUsdG9rCgpkZWYgZ29maWxlX2RsX29uZShsaW5rLHRvaz1Ob25lLHRyaWVzPTMpOgogZHVybD1saW5rLmdldCgnZG93bmxvYWRVcmwnLCcnKQogbmFtZT1saW5rLmdldCgnbmFtZScsJ2ZpbGUnKQogaWYgbm90IGR1cmw6cHJpbnQoJyAgVGlkYWsgYWRhIGRvd25sb2FkIFVSTCwgc2tpcC4nKTtyZXR1cm4gTm9uZQogZGVzdD1VUExPQUQvbmFtZQogcGFydD1VUExPQUQvKG5hbWUrJy5wYXJ0JykKIGlmIGRlc3QuZXhpc3RzKCkgYW5kIGRlc3Quc3RhdCgpLnN0X3NpemU+MDoKICBwcmludCgnICBTS0lQICcrbmFtZSsnIChzdWRhaCBhZGEpJykKICByZXR1cm4gZGVzdAogaGRyPXsnVXNlci1BZ2VudCc6J01vemlsbGEvNS4wJywnUmVmZXJlcic6J2h0dHBzOi8vZ29maWxlLmlvLycsJ09yaWdpbic6J2h0dHBzOi8vZ29maWxlLmlvJ30KIGlmIHRvazpoZHJbJ0Nvb2tpZSddPSdhY2NvdW50VG9rZW49Jyt0b2sKIGZvciBhdHQgaW4gcmFuZ2UoMSx0cmllcysxKToKICB0cnk6CiAgIHByaW50KCcgIERvd25sb2FkaW5nICcrbmFtZSsnLi4uJysoJycgaWYgYXR0PT0xIGVsc2UgJyAoY29iYSAnK3N0cihhdHQpKycpJykpCiAgIHJyPXJlcXVlc3RzLmdldChkdXJsLGhlYWRlcnM9aGRyLHN0cmVhbT1UcnVlLHRpbWVvdXQ9NjAwKQogICByci5yYWlzZV9mb3Jfc3RhdHVzKCkKICAgdG90YWxfc2l6ZT1pbnQobGluay5nZXQoJ3NpemUnKSBvciBsaW5rLmdldCgnYnl0ZXMnKSBvciByci5oZWFkZXJzLmdldCgnY29udGVudC1sZW5ndGgnKSBvciAwKQogICBkb25lPTAKICAgdDA9dGltZS50aW1lKCkKICAgd2l0aCBvcGVuKHBhcnQsJ3diJykgYXMgZmg6CiAgICBmb3IgY2ggaW4gcnIuaXRlcl9jb250ZW50KGNodW5rX3NpemU9MTAyNCoxMDI0KToKICAgICBpZiBjaDoKICAgICAgZmgud3JpdGUoY2gpCiAgICAgIGRvbmUrPWxlbihjaCkKICAgICAgZWw9dGltZS50aW1lKCktdDAKICAgICAgc3BkPShkb25lL2VsLzEwMjQvMTAyNCkgaWYgZWw+MCBlbHNlIDAKICAgICAgaWYgdG90YWxfc2l6ZT4wOgogICAgICAgcGN0PXJvdW5kKGRvbmUvdG90YWxfc2l6ZSoxMDAsMSkKICAgICAgIHByaW50KGYnXHIgICAge3BjdH0lICB7cm91bmQoZG9uZS8xMDI0LzEwMjQsMSl9TUIgICh7cm91bmQoc3BkLDEpfSBNQi9zKScsZW5kPScnLGZsdXNoPVRydWUpCiAgICAgIGVsc2U6CiAgICAgICBwcmludChmJ1xyICAgIHtyb3VuZChkb25lLzEwMjQvMTAyNCwxKX1NQiAgKHtyb3VuZChzcGQsMSl9IE1CL3MpJyxlbmQ9JycsZmx1c2g9VHJ1ZSkKICAgcHJpbnQoKQogICBpZiBkb25lPT0wOnJhaXNlIEV4Y2VwdGlvbignMCBieXRlJykKICAgaWYgZGVzdC5leGlzdHMoKTpkZXN0LnVubGluaygpCiAgIHBhcnQucmVuYW1lKGRlc3QpCiAgIHByaW50KG9rKCcgIE9LICcpK25hbWUrJyAoJytzdHIocm91bmQoZG9uZS8xMDI0LzEwMjQsMSkpKycgTUIpJykKICAgcmV0dXJuIGRlc3QKICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgIHByaW50KGYnXG4gIEdhZ2FsIGNvYmEge2F0dH06IHtzdHIoZSlbOjEyMF19JykKICAgaWYgcGFydC5leGlzdHMoKToKICAgIHRyeTpwYXJ0LnVubGluaygpCiAgICBleGNlcHQ6cGFzcwogICBpZiBhdHQ8dHJpZXM6dGltZS5zbGVlcCg1KmF0dCkKICAgZWxzZTpwcmludChlcignICBHYWdhbCB0b3RhbDogJykrbmFtZSsnIC0gJytzdHIoZSlbOjE1MF0pCiByZXR1cm4gTm9uZQoKZGVmIGRsX2dvZmlsZSgpOgogaGRyKCdET1dOTE9BRCAtIEdvZmlsZScpCiB1cmw9aW5wdXQoJ1xuICBMaW5rIEdvZmlsZTogJykuc3RyaXAoKQogaWYgbm90IHVybDpyZXR1cm4KIHB3ZD1pbnB1dCgnICBQYXNzd29yZCAoa29zb25nID0gdGlkYWsgYWRhKTogJykuc3RyaXAoKQogdG9rZW49Z2V0X2dvZmlsZV90b2tlbigpCiBmaWxlcz1bXQogdG9rX2Zvcl9kbD10b2tlbgogaWYgdG9rZW46CiAgdHJ5OgogICBwcmludCgnICBNZW5nYW1iaWwgZGFmdGFyIGZpbGUgdmlhIHByb3h5Li4uJykKICAgZmlsZXM9Z29maWxlX2FwaV9saXN0KHVybCxwd2QsdG9rZW4pCiAgZXhjZXB0OnBhc3MKIGlmIG5vdCBmaWxlczoKICBwcmludCgnICBNZW5nYW1iaWwgZGFmdGFyIGZpbGUgdmlhIERpcmVjdCBBUEkuLi4nKQogIGRmaWxlcyxlcnIsZGlyZWN0X3Rvaz1nb2ZpbGVfZGlyZWN0X2ZldGNoKHVybCxwd2QpCiAgaWYgZXJyOgogICBwcmludChlcignICAnK2VycikpCiAgIGlucHV0KCcgIEVudGVyLi4uJyk7cmV0dXJuCiAgZmlsZXM9ZGZpbGVzCiAgdG9rX2Zvcl9kbD1kaXJlY3RfdG9rCiBpZiBub3QgZmlsZXM6CiAgcHJpbnQoJyAgRm9sZGVyIGtvc29uZyAvIHRpZGFrIGJpc2EgZGlha3Nlcy4nKQogIGlucHV0KCcgIEVudGVyLi4uJyk7cmV0dXJuCiBwcmludChmJyAgRGl0ZW11a2FuIHtsZW4oZmlsZXMpfSBmaWxlOicpCiBmb3IgaSxmZiBpbiBlbnVtZXJhdGUoZmlsZXMpOgogIHN6PWZmLmdldCgnc2l6ZScsJz8nKQogIGlmIGlzaW5zdGFuY2Uoc3osaW50KTpzej1mJ3tyb3VuZChzei8xMDI0LzEwMjQsMSl9TUInCiAgcHJpbnQoZicgICAgW3tpfV0ge2ZmLmdldCgibmFtZSIsIj8iKX0gKHtzen0pJykKIHByaW50KCkKIGM9aW5wdXQoJyAgUGlsaWggKCogc2VtdWEgLyAwIC8gMCwxIC8gMC0yKTogJykuc3RyaXAoKQogaWYgYz09JyonOnRhcmdldHM9ZmlsZXMKIGVsc2U6CiAgdHJ5OgogICBudW1zPVtdCiAgIGZvciBwYXJ0IGluIGMuc3BsaXQoJywnKToKICAgIHBhcnQ9cGFydC5zdHJpcCgpCiAgICBpZiAnLScgaW4gcGFydDoKICAgICBhLGI9cGFydC5zcGxpdCgnLScsMSk7bnVtcy5leHRlbmQocmFuZ2UoaW50KGEpLGludChiKSsxKSkKICAgIGVsc2U6bnVtcy5hcHBlbmQoaW50KHBhcnQpKQogICB0YXJnZXRzPVtmaWxlc1tuXSBmb3IgbiBpbiBudW1zIGlmIDA8PW48bGVuKGZpbGVzKV0KICBleGNlcHQ6cHJpbnQoJyAgSW5wdXQgdGlkYWsgdmFsaWQuJyk7aW5wdXQoJyAgRW50ZXIuLi4nKTtyZXR1cm4KIGlmIG5vdCB0YXJnZXRzOnByaW50KCcgIFRpZGFrIGFkYSB5YW5nIGRpcGlsaWguJyk7aW5wdXQoJyAgRW50ZXIuLi4nKTtyZXR1cm4KIGZhaWxzPVtdCiBmb3IgbGluayBpbiB0YXJnZXRzOgogIGlmIG5vdCBnb2ZpbGVfZGxfb25lKGxpbmssdG9rPXRva19mb3JfZGwpOmZhaWxzLmFwcGVuZChsaW5rLmdldCgnbmFtZScsJz8nKSkKIGlmIGZhaWxzOgogIHByaW50KGVyKGYnICBHYWdhbCB7bGVuKGZhaWxzKX0gZmlsZTonKSkKICBmb3IgbiBpbiBmYWlsczpwcmludCgnICAgIC0gJytuKQogZWxzZToKICBwcmludChvaygnICBTZW11YSBkb3dubG9hZCBHb2ZpbGUgc2VsZXNhaSEnKSkKCiMg4pSA4pSA4pSAIEdPT0dMRSBEUklWRSBET1dOTE9BREVSIChPQXV0aCBBUEkgdjMgKyBnZG93biBmYWxsYmFjaykg4pSA4pSA4pSACmRlZiBleHRyYWN0X2dkcml2ZV9pZCh1cmxfb3JfaWQpOgogcz11cmxfb3JfaWQuc3RyaXAoKQogbT1yZS5zZWFyY2gocicvZm9sZGVycy8oW2EtekEtWjAtOV8tXSspJyxzKQogaWYgbTpyZXR1cm4gbS5ncm91cCgxKSxUcnVlCiBtPXJlLnNlYXJjaChyJy9maWxlL2QvKFthLXpBLVowLTlfLV0rKScscykKIGlmIG06cmV0dXJuIG0uZ3JvdXAoMSksRmFsc2UKIG09cmUuc2VhcmNoKHInWz8mXWlkPShbYS16QS1aMC05Xy1dKyknLHMpCiBpZiBtOnJldHVybiBtLmdyb3VwKDEpLE5vbmUKIG09cmUuc2VhcmNoKHInaWQ9KFthLXpBLVowLTlfLV0rKScscykKIGlmIG06cmV0dXJuIG0uZ3JvdXAoMSksTm9uZQogaWYgcmUubWF0Y2gocideW2EtekEtWjAtOV8tXXsyMCx9JCcscyk6CiAgcmV0dXJuIHMsTm9uZQogcmV0dXJuIE5vbmUsTm9uZQoKZGVmIGdkcml2ZV90b2tlbihjaWQsc2VjLHJlZik6CiB0cnk6CiAgcj1yZXF1ZXN0cy5wb3N0KCdodHRwczovL29hdXRoMi5nb29nbGVhcGlzLmNvbS90b2tlbicsZGF0YT17J2NsaWVudF9pZCc6Y2lkLCdjbGllbnRfc2VjcmV0JzpzZWMsJ3JlZnJlc2hfdG9rZW4nOnJlZiwnZ3JhbnRfdHlwZSc6J3JlZnJlc2hfdG9rZW4nfSx0aW1lb3V0PTE1KQogIHJldHVybiByLmpzb24oKS5nZXQoJ2FjY2Vzc190b2tlbicpCiBleGNlcHQ6cmV0dXJuIE5vbmUKCmRlZiBnZHJpdmVfZG93bmxvYWRfZmlsZSh0b2ssZmlkLG5hbWUsc2l6ZSxkZXN0X2Rpcik6CiBkZXN0PWRlc3RfZGlyL25hbWUKIHBhcnQ9ZGVzdF9kaXIvKG5hbWUrJy5wYXJ0JykKIGlmIGRlc3QuZXhpc3RzKCkgYW5kIGRlc3Quc3RhdCgpLnN0X3NpemU+MDoKICBpZiBzaXplIGFuZCBkZXN0LnN0YXQoKS5zdF9zaXplPT1pbnQoc2l6ZSk6CiAgIHByaW50KCcgIFNLSVAgJytuYW1lKycgKHN1ZGFoIGFkYSknKQogICByZXR1cm4gVHJ1ZQogdXJsPWYnaHR0cHM6Ly93d3cuZ29vZ2xlYXBpcy5jb20vZHJpdmUvdjMvZmlsZXMve2ZpZH0/YWx0PW1lZGlhJwogaGVhZGVycz17J0F1dGhvcml6YXRpb24nOidCZWFyZXIgJyt0b2t9CiB0cnk6CiAgcj1yZXF1ZXN0cy5nZXQodXJsLGhlYWRlcnM9aGVhZGVycyxzdHJlYW09VHJ1ZSx0aW1lb3V0PTMwKQogIGlmIHIuc3RhdHVzX2NvZGUhPTIwMDoKICAgcHJpbnQoZXIoZicgIEdhZ2FsIGRvd25sb2FkIHtuYW1lfTogSFRUUCB7ci5zdGF0dXNfY29kZX0gKHtyLnRleHRbOjgwXX0pJykpCiAgIHJldHVybiBGYWxzZQogIHRvdGFsPWludChzaXplKSBpZiBzaXplIGVsc2UgaW50KHIuaGVhZGVycy5nZXQoJ2NvbnRlbnQtbGVuZ3RoJywwKSkKICBkb25lPTAKICB0MD10aW1lLnRpbWUoKQogIHdpdGggb3BlbihwYXJ0LCd3YicpIGFzIGZoOgogICBmb3IgY2ggaW4gci5pdGVyX2NvbnRlbnQoY2h1bmtfc2l6ZT0xNioxMDI0KjEwMjQpOgogICAgaWYgY2g6CiAgICAgZmgud3JpdGUoY2gpCiAgICAgZG9uZSs9bGVuKGNoKQogICAgIGVsPXRpbWUudGltZSgpLXQwCiAgICAgc3BkPShkb25lL2VsLzEwMjQvMTAyNCkgaWYgZWw+MCBlbHNlIDAKICAgICBpZiB0b3RhbD4wOgogICAgICBwY3Q9cm91bmQoZG9uZS90b3RhbCoxMDAsMSkKICAgICAgcHJpbnQoZidcciAgICB7cGN0fSUgIHtyb3VuZChkb25lLzEwMjQvMTAyNCwxKX1NQiAgKHtyb3VuZChzcGQsMSl9IE1CL3MpJyxlbmQ9JycsZmx1c2g9VHJ1ZSkKICAgICBlbHNlOgogICAgICBwcmludChmJ1xyICAgIHtyb3VuZChkb25lLzEwMjQvMTAyNCwxKX1NQiAgKHtyb3VuZChzcGQsMSl9IE1CL3MpJyxlbmQ9JycsZmx1c2g9VHJ1ZSkKICBwcmludCgpCiAgaWYgcGFydC5leGlzdHMoKToKICAgaWYgZGVzdC5leGlzdHMoKTpkZXN0LnVubGluaygpCiAgIHBhcnQucmVuYW1lKGRlc3QpCiAgIHByaW50KG9rKCcgIE9LICcpK25hbWUrJyAoJytzdHIocm91bmQoZG9uZS8xMDI0LzEwMjQsMSkpKycgTUIpJykKICAgcmV0dXJuIFRydWUKIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICBwcmludChmJ1xuICBFcnJvciB7bmFtZX06IHtzdHIoZSlbOjEyMF19JykKICBpZiBwYXJ0LmV4aXN0cygpOgogICB0cnk6cGFydC51bmxpbmsoKQogICBleGNlcHQ6cGFzcwogcmV0dXJuIEZhbHNlCgpkZWYgZ2RyaXZlX2xpc3RfZm9sZGVyKHRvayxmb2xkZXJfaWQpOgogZmlsZXM9W10KIHBhZ2VfdG9rZW49Tm9uZQogd2hpbGUgVHJ1ZToKICBwYXJhbXM9eydxJzpmIid7Zm9sZGVyX2lkfScgaW4gcGFyZW50cyBhbmQgdHJhc2hlZD1mYWxzZSIsJ2ZpZWxkcyc6J25leHRQYWdlVG9rZW4sIGZpbGVzKGlkLCBuYW1lLCBtaW1lVHlwZSwgc2l6ZSknLCdwYWdlU2l6ZSc6MTAwMH0KICBpZiBwYWdlX3Rva2VuOnBhcmFtc1sncGFnZVRva2VuJ109cGFnZV90b2tlbgogIHRyeToKICAgcj1yZXF1ZXN0cy5nZXQoJ2h0dHBzOi8vd3d3Lmdvb2dsZWFwaXMuY29tL2RyaXZlL3YzL2ZpbGVzJyxoZWFkZXJzPXsnQXV0aG9yaXphdGlvbic6J0JlYXJlciAnK3Rva30scGFyYW1zPXBhcmFtcyx0aW1lb3V0PTIwKQogICBkPXIuanNvbigpCiAgIGlmICdlcnJvcicgaW4gZDoKICAgIHByaW50KGVyKCcgIERyaXZlIEFQSSBlcnJvcjogJytzdHIoZFsnZXJyb3InXS5nZXQoJ21lc3NhZ2UnLCcnKSkpKQogICAgcmV0dXJuIE5vbmUKICAgZmlsZXMuZXh0ZW5kKGQuZ2V0KCdmaWxlcycsW10pKQogICBwYWdlX3Rva2VuPWQuZ2V0KCduZXh0UGFnZVRva2VuJykKICAgaWYgbm90IHBhZ2VfdG9rZW46YnJlYWsKICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgIHByaW50KGVyKCcgIEdhZ2FsIGxpc3QgZm9sZGVyOiAnK3N0cihlKVs6MTAwXSkpCiAgIHJldHVybiBOb25lCiByZXR1cm4gZmlsZXMKCmRlZiBkbF9kcml2ZSgpOgogaGRyKCdET1dOTE9BRCAtIEdvb2dsZSBEcml2ZScpCiB1cmw9aW5wdXQoJ1xuICBMaW5rIEdEcml2ZSAvIEZpbGUgSUQgLyBGb2xkZXIgSUQ6ICcpLnN0cmlwKCkKIGlmIG5vdCB1cmw6cmV0dXJuCiBjaWQ9Z2V0X3NlY3JldCgnR0RSSVZFX0NMSUVOVF9JRCcpCiBzZWM9Z2V0X3NlY3JldCgnR0RSSVZFX0NMSUVOVF9TRUNSRVQnKQogcmVmPWdldF9zZWNyZXQoJ0dEUklWRV9SRUZSRVNIX1RPS0VOJykKIHRvaz1Ob25lCiBpZiBjaWQgYW5kIHNlYyBhbmQgcmVmOgogIHByaW50KCcgIEF1dGggdmlhIEdvb2dsZSBPQXV0aCBBUEkgdjMuLi4nKQogIHRvaz1nZHJpdmVfdG9rZW4oY2lkLHNlYyxyZWYpCiBnaWQsaXNfZj1leHRyYWN0X2dkcml2ZV9pZCh1cmwpCiBpZiB0b2sgYW5kIGdpZDoKICB0cnk6CiAgIHI9cmVxdWVzdHMuZ2V0KGYnaHR0cHM6Ly93d3cuZ29vZ2xlYXBpcy5jb20vZHJpdmUvdjMvZmlsZXMve2dpZH0/ZmllbGRzPWlkLG5hbWUsbWltZVR5cGUsc2l6ZScsaGVhZGVycz17J0F1dGhvcml6YXRpb24nOidCZWFyZXIgJyt0b2t9LHRpbWVvdXQ9MTUpCiAgIGl0ZW09ci5qc29uKCkKICAgaWYgJ2Vycm9yJyBub3QgaW4gaXRlbToKICAgIG1pbWU9aXRlbS5nZXQoJ21pbWVUeXBlJywnJykKICAgIGlmIG1pbWU9PSdhcHBsaWNhdGlvbi92bmQuZ29vZ2xlLWFwcHMuZm9sZGVyJyBvciBpc19mOgogICAgIHByaW50KGYnICBGb2xkZXI6IHtpdGVtLmdldCgibmFtZSIsImRyaXZlX2ZvbGRlciIpfScpCiAgICAgcHJpbnQoJyAgTWVuZ2FtYmlsIGRhZnRhciBmaWxlLi4uJykKICAgICBmbGlzdD1nZHJpdmVfbGlzdF9mb2xkZXIodG9rLGdpZCkKICAgICBpZiBmbGlzdCBpcyBOb25lOnJldHVybgogICAgIGZsaXN0PVtmIGZvciBmIGluIGZsaXN0IGlmIGYuZ2V0KCdtaW1lVHlwZScpIT0nYXBwbGljYXRpb24vdm5kLmdvb2dsZS1hcHBzLmZvbGRlciddCiAgICAgaWYgbm90IGZsaXN0OgogICAgICBwcmludCgnICBGb2xkZXIga29zb25nLicpCiAgICAgIGlucHV0KCcgIEVudGVyLi4uJyk7cmV0dXJuCiAgICAgcHJpbnQoZicgIERpdGVtdWthbiB7bGVuKGZsaXN0KX0gZmlsZTonKQogICAgIGZvciBpLGZmIGluIGVudW1lcmF0ZShmbGlzdCk6CiAgICAgIHN6PXJvdW5kKGludChmZi5nZXQoJ3NpemUnLDApKS8xMDI0LzEwMjQsMSkKICAgICAgcHJpbnQoZicgICAgW3tpfV0ge2ZmLmdldCgibmFtZSIsIj8iKX0gKHtzen0gTUIpJykKICAgICBwcmludCgpCiAgICAgYz1pbnB1dCgnICBQaWxpaCAoKiBzZW11YSAvIDAgLyAwLDEgLyAwLTIpOiAnKS5zdHJpcCgpCiAgICAgaWYgYz09JyonOnRhcmdldHM9Zmxpc3QKICAgICBlbHNlOgogICAgICB0cnk6CiAgICAgICBudW1zPVtdCiAgICAgICBmb3IgcGFydCBpbiBjLnNwbGl0KCcsJyk6CiAgICAgICAgcGFydD1wYXJ0LnN0cmlwKCkKICAgICAgICBpZiAnLScgaW4gcGFydDoKICAgICAgICAgYSxiPXBhcnQuc3BsaXQoJy0nLDEpO251bXMuZXh0ZW5kKHJhbmdlKGludChhKSxpbnQoYikrMSkpCiAgICAgICAgZWxzZTpudW1zLmFwcGVuZChpbnQocGFydCkpCiAgICAgICB0YXJnZXRzPVtmbGlzdFtuXSBmb3IgbiBpbiBudW1zIGlmIDA8PW48bGVuKGZsaXN0KV0KICAgICAgZXhjZXB0OnByaW50KCcgIElucHV0IHRpZGFrIHZhbGlkLicpO2lucHV0KCcgIEVudGVyLi4uJyk7cmV0dXJuCiAgICAgaWYgbm90IHRhcmdldHM6cHJpbnQoJyAgVGlkYWsgYWRhIHlhbmcgZGlwaWxpaC4nKTtpbnB1dCgnICBFbnRlci4uLicpO3JldHVybgogICAgIG9rX249MDtmYWlscz1bXQogICAgIGZvciBmIGluIHRhcmdldHM6CiAgICAgIGlmIGdkcml2ZV9kb3dubG9hZF9maWxlKHRvayxmWydpZCddLGZbJ25hbWUnXSxmLmdldCgnc2l6ZScpLFVQTE9BRCk6b2tfbis9MQogICAgICBlbHNlOmZhaWxzLmFwcGVuZChmWyduYW1lJ10pCiAgICAgaWYgZmFpbHM6cHJpbnQoZXIoJyAgR2FnYWw6ICcrJywgJy5qb2luKGZhaWxzKSkpCiAgICAgZWxzZTpwcmludChvayhmJyAgRG93bmxvYWQgc2VsZXNhaSEgKHtva19ufSBmaWxlKScpKQogICAgIHJldHVybgogICAgZWxzZToKICAgICBwcmludChmJyAgRmlsZToge2l0ZW0uZ2V0KCJuYW1lIil9ICh7cm91bmQoaW50KGl0ZW0uZ2V0KCJzaXplIiwwKSkvMTAyNC8xMDI0LDEpfSBNQiknKQogICAgIGlmIGdkcml2ZV9kb3dubG9hZF9maWxlKHRvayxnaWQsaXRlbS5nZXQoJ25hbWUnLCdmaWxlJyksaXRlbS5nZXQoJ3NpemUnKSxVUExPQUQpOgogICAgICBwcmludChvaygnICBEb3dubG9hZCBmaWxlIGJlcmhhc2lsIScpKQogICAgIGVsc2U6CiAgICAgIHByaW50KGVyKCcgIERvd25sb2FkIGZpbGUgZ2FnYWwuJykpCiAgICAgcmV0dXJuCiAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOgogICBwcmludChlcignICBEcml2ZSBBUEkgcXVlcnkgZXJyb3I6ICcrc3RyKGUpWzoxMDBdKSkKICMgRmFsbGJhY2sgdG8gZ2Rvd24KIHByaW50KGRpbSgnICBPQXV0aCB0aWRhayBha3RpZiAvIElEIHRpZGFrIGRpdGVtdWthbiBkaSBBUEkuIEZhbGxiYWNrIGtlIGdkb3duLi4uJykpCiBjbWQ9WydnZG93bicsJy1PJyxzdHIoVVBMT0FEKSwnLS1yZW1haW5pbmctb2snXQogaWYgaXNfZiBvciAnL2ZvbGRlcnMvJyBpbiB1cmw6Y21kLmluc2VydCgxLCctLWZvbGRlcicpCiBjbWQuYXBwZW5kKHVybCkKIHI9c3VicHJvY2Vzcy5ydW4oY21kKQogaWYgci5yZXR1cm5jb2RlPT0wOnByaW50KG9rKCcgIERvd25sb2FkIHNlbGVzYWkgKHZpYSBnZG93bikhJykpCiBlbHNlOnByaW50KGVyKCcgIERvd25sb2FkIGdhZ2FsIChjb2RlICcrc3RyKHIucmV0dXJuY29kZSkrJyknKSkKCiMg4pSA4pSA4pSAIERJUkVDVCBVUkwgRE9XTkxPQURFUiDilIDilIDilIAKZGVmIGRsX3VybCgpOgogaGRyKCdET1dOTE9BRCAtIERpcmVjdCBVUkwnKQogdXJsPWlucHV0KCdcbiAgRGlyZWN0IFVSTDogJykuc3RyaXAoKQogaWYgbm90IHVybDpyZXR1cm4KIGZuYW1lPWlucHV0KCcgIEZpbGVuYW1lIChrb3NvbmcgPSBhdXRvKTogJykuc3RyaXAoKSBvciBOb25lCiBjbWQ9Wyd3Z2V0JywnLXEnLCctUCcsc3RyKFVQTE9BRCksJy0tY29udGVudC1kaXNwb3NpdGlvbicsJy0tbm8tY2hlY2stY2VydGlmaWNhdGUnXQogaWYgZm5hbWU6Y21kLmV4dGVuZChbJy1PJyxzdHIoVVBMT0FEL2ZuYW1lKV0pCiBjbWQuYXBwZW5kKHVybCkKIHI9c3VicHJvY2Vzcy5ydW4oY21kLHRpbWVvdXQ9NjAwKQogaWYgci5yZXR1cm5jb2RlPT0wOnByaW50KG9rKCcgIERvd25sb2FkIHNlbGVzYWkhJykpCiBlbHNlOnByaW50KGVyKCcgIERvd25sb2FkIGdhZ2FsIChjb2RlICcrc3RyKHIucmV0dXJuY29kZSkrJyknKSkKCmRlZiBtZW51X2Rvd25sb2FkKCk6CiB3aGlsZSBUcnVlOgogIGNpKCk7aGRyKCdET1dOTE9BRCcpCiAgcHJpbnQoKQogIHByaW50KCcgIFsxXSBHb2ZpbGUnKQogIHByaW50KCcgIFsyXSBHb29nbGUgRHJpdmUgKE9BdXRoIEFQSSB2MyAvIGdkb3duKScpCiAgcHJpbnQoJyAgWzNdIERpcmVjdCBVUkwnKQogIHByaW50KCkKICBwcmludCgnICBbMF0gS2VtYmFsaScpCiAgcHJpbnQoKQogIGM9aW5wdXQoJyAgUGlsaWg6ICcpLnN0cmlwKCkKICBpZiBjPT0nMCc6cmV0dXJuCiAgZWxpZiBjPT0nMSc6ZGxfZ29maWxlKCkKICBlbGlmIGM9PScyJzpkbF9kcml2ZSgpCiAgZWxpZiBjPT0nMyc6ZGxfdXJsKCkKICBpbnB1dCgnXG4gIEVudGVyLi4uJykKCmRlZiBtYWluKCk6CiBsb2FkX3NlY3JldHMoKQogbWVudV9kb3dubG9hZCgpCgppZiBfX25hbWVfXz09J19fbWFpbl9fJzptYWluKCkK""",
        'haru-upload': """IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwppbXBvcnQgc3VicHJvY2VzcyxzeXMsb3MscmUsZ2xvYixqc29uLHRpbWUKaW1wb3J0IHJlcXVlc3RzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApWPXsnLm1rdicsJy5tcDQnLCcuYXZpJywnLm1vdicsJy53ZWJtJywnLmZsdicsJy53bXYnLCcudHMnLCcubTR2J30KQT17Jy5tcDMnLCcuYWFjJywnLmZsYWMnLCcud2F2JywnLm9nZycsJy5vcHVzJywnLm1rYScsJy5hYzMnLCcuZHRzJywnLmVhYzMnLCcubTRhJ30KUz17Jy5zcnQnLCcuYXNzJywnLnNzYScsJy5zdWInLCcuaWR4JywnLnN1cCcsJy52dHQnLCcucGdzJywnLnNjYycsJy5zYW1pJ30KTD17J2lkJzonSW5kb25lc2lhbicsJ2VuJzonRW5nbGlzaCcsJ2phJzonSmFwYW5lc2UnLCdrbyc6J0tvcmVhbicsJ3poJzonQ2hpbmVzZScsJ21zJzonTWFsYXknLCdhcic6J0FyYWJpYycsJ2RlJzonR2VybWFuJywnZnInOidGcmVuY2gnLCdlcyc6J1NwYW5pc2gnLCdwdCc6J1BvcnR1Z3Vlc2UnLCdydSc6J1J1c3NpYW4nLCdpdCc6J0l0YWxpYW4nLCd0aCc6J1RoYWknLCd2aSc6J1ZpZXRuYW1lc2UnLCdoaSc6J0hpbmRpJywndW5kJzonVW5kZXRlcm1pbmVkJ30KVVBMT0FEPVBhdGgoJy9jb250ZW50L3VwbG9hZHMnKQpPVVRQVVQ9UGF0aCgnL2NvbnRlbnQvb3V0cHV0JykKVVBMT0FELm1rZGlyKGV4aXN0X29rPVRydWUpCk9VVFBVVC5ta2RpcihleGlzdF9vaz1UcnVlKQpkZWYgY2koKToKIGltcG9ydCBzeXMKIHN5cy5zdGRvdXQud3JpdGUoJ1x4MWJbMkpceDFiW0gnKQogc3lzLnN0ZG91dC5mbHVzaCgpCmRlZiBvayh0KTpyZXR1cm4gJ1wwMzNbOTJtJyt0KydcMDMzWzBtJwpkZWYgZXIodCk6cmV0dXJuICdcMDMzWzkxbScrdCsnXDAzM1swbScKZGVmIGRpbSh0KTpyZXR1cm4gJ1wwMzNbOTBtJyt0KydcMDMzWzBtJwpkZWYgaGRyKHRpdGxlKTpwcmludCgnXG4nKyc9Jyo2Mik7cHJpbnQoJyAgJyt0aXRsZSk7cHJpbnQoJz0nKjYyKQpkZWYgbG9hZF9zZWNyZXRzKCk6CiB0cnk6CiAgaWYgb3MucGF0aC5leGlzdHMoJy9jb250ZW50Ly5oYXJ1X3NlY3JldHMuanNvbicpOgogICBkPWpzb24ubG9hZChvcGVuKCcvY29udGVudC8uaGFydV9zZWNyZXRzLmpzb24nKSkKICAgZm9yIGssdiBpbiBkLml0ZW1zKCk6CiAgICBpZiB2IGFuZCBub3Qgb3MuZW52aXJvbi5nZXQoayk6b3MuZW52aXJvbltrXT1zdHIodikKIGV4Y2VwdDpwYXNzCmRlZiBnZXRfc2VjcmV0KGspOgogdj1vcy5lbnZpcm9uLmdldChrLCcnKQogaWYgdjpyZXR1cm4gdi5zdHJpcCgpCiB0cnk6CiAgZnJvbSBnb29nbGUuY29sYWIgaW1wb3J0IHVzZXJkYXRhCiAgdD11c2VyZGF0YS5nZXQoaykKICBpZiB0OnJldHVybiBzdHIodCkuc3RyaXAoKQogZXhjZXB0OnBhc3MKIHJldHVybiAnJwpkZWYgZ2V0X2dvZmlsZV90b2tlbigpOgogcmV0dXJuIGdldF9zZWNyZXQoJ0dPRklMRV9BUElfVE9LRU4nKQpkZWYgZ29maWxlX3VwbG9hZF9maWxlcyh0YXJnZXRzLCBmb2xkZXJfbmFtZT1Ob25lKToKIGlmIG5vdCB0YXJnZXRzOnJldHVybiBGYWxzZSxbXQogdG9rZW49Z2V0X2dvZmlsZV90b2tlbigpCiBpZiBub3QgdG9rZW46cmV0dXJuIEZhbHNlLFtdCiB0cnk6CiAgcj1yZXF1ZXN0cy5nZXQoJ2h0dHBzOi8vYXBpLmdvZmlsZS5pby9hY2NvdW50cycsdGltZW91dD0xNSkKICBkPXIuanNvbigpCiAgaWYgZC5nZXQoJ3N0YXR1cycpIT0nb2snOnJldHVybiBGYWxzZSxbXQogIGFjY291bnRfdG9rZW49ZFsnZGF0YSddWyd0b2tlbiddCiBleGNlcHQ6cmV0dXJuIEZhbHNlLFtdCiBmb2xkZXJfaWQ9Tm9uZQogaWYgZm9sZGVyX25hbWUgYW5kIGxlbih0YXJnZXRzKT4xOgogIHRyeToKICAgcj1yZXF1ZXN0cy5wb3N0KCdodHRwczovL2FwaS5nb2ZpbGUuaW8vY29udGVudHMnLGhlYWRlcnM9eydBdXRob3JpemF0aW9uJzonQmVhcmVyICcrYWNjb3VudF90b2tlbiwnQ29udGVudC1UeXBlJzonYXBwbGljYXRpb24vanNvbid9LGpzb249eyd0eXBlJzonZm9sZGVyJywndGl0bGUnOmZvbGRlcl9uYW1lfSx0aW1lb3V0PTE1KQogICBkPXIuanNvbigpCiAgIGlmIGQuZ2V0KCdzdGF0dXMnKT09J29rJzpmb2xkZXJfaWQ9ZFsnZGF0YSddWydpZCddCiAgZXhjZXB0OnBhc3MKIHNydj0nc3RvcmUxJwogdHJ5OgogIHN2PXJlcXVlc3RzLmdldCgnaHR0cHM6Ly9hcGkuZ29maWxlLmlvL3NlcnZlcnMnLGhlYWRlcnM9eydBdXRob3JpemF0aW9uJzonQmVhcmVyICcrYWNjb3VudF90b2tlbn0sdGltZW91dD0xNSkuanNvbigpCiAgaWYgc3YuZ2V0KCdzdGF0dXMnKT09J29rJzpzcnY9c3ZbJ2RhdGEnXVsnc2VydmVycyddWzBdWyduYW1lJ10KIGV4Y2VwdDpwYXNzCiBsaW5rcz1bXQogZm9yIGYgaW4gdGFyZ2V0czoKICBwcmludCgnICBVcGxvYWQgJytmLm5hbWUrJyAoJytzdHIocm91bmQoZi5zdGF0KCkuc3Rfc2l6ZS8xMDI0LzEwMjQsMSkpKydNQikgdmlhICcrc3J2KycuLi4nKQogIGNtZD1bJ2N1cmwnLCctcycsJy1GJywnZmlsZT1AJytzdHIoZildCiAgaWYgZm9sZGVyX2lkOmNtZC5leHRlbmQoWyctRicsJ2ZvbGRlcklkPScrZm9sZGVyX2lkXSkKICBjbWQuYXBwZW5kKCdodHRwczovLycrc3J2KycuZ29maWxlLmlvL3VwbG9hZEZpbGUnKQogIHI9c3VicHJvY2Vzcy5ydW4oY21kLGNhcHR1cmVfb3V0cHV0PVRydWUsdGV4dD1UcnVlLHRpbWVvdXQ9NjAwKQogIHRyeToKICAgZGF0YT1qc29uLmxvYWRzKHIuc3Rkb3V0KQogICBpZiBkYXRhLmdldCgnc3RhdHVzJyk9PSdvayc6CiAgICBsaW5rcy5hcHBlbmQoKGYubmFtZSxkYXRhWydkYXRhJ11bJ2Rvd25sb2FkUGFnZSddKSkKICAgIHByaW50KCcgICcrb2soJ29rJykrJyAnK2YubmFtZSkKICAgZWxzZTpwcmludCgnICAnK2VyKCdnYWdhbCcpKycgJytzdHIoZGF0YSlbOjEwMF0pCiAgZXhjZXB0OnByaW50KCcgICcrZXIoJ2dhZ2FsJykrJyAnK2YubmFtZSsnIChubyByZXNwb25zZSknKQogaWYgbm90IGxpbmtzOnJldHVybiBGYWxzZSxbXQogaWYgbGVuKGxpbmtzKT09MTpyZXR1cm4gVHJ1ZSxbbGlua3NbMF1dCiByZXR1cm4gVHJ1ZSxsaW5rcwpkZWYgZ2RyaXZlX3NlY3JldChrKToKIHJldHVybiBnZXRfc2VjcmV0KGspCmRlZiBnZHJpdmVfdG9rZW4oY2lkLHNlYyxyZWYpOgogdHJ5OgogIHI9cmVxdWVzdHMucG9zdCgnaHR0cHM6Ly9vYXV0aDIuZ29vZ2xlYXBpcy5jb20vdG9rZW4nLGRhdGE9eydjbGllbnRfaWQnOmNpZCwnY2xpZW50X3NlY3JldCc6c2VjLCdyZWZyZXNoX3Rva2VuJzpyZWYsJ2dyYW50X3R5cGUnOidyZWZyZXNoX3Rva2VuJ30sdGltZW91dD0xNSkKICByZXR1cm4gci5qc29uKCkuZ2V0KCdhY2Nlc3NfdG9rZW4nKQogZXhjZXB0OnJldHVybiBOb25lCmRlZiBnZHJpdmVfdXBsb2FkX2ZpbGUodG9rLGZwYXRoLHBhcmVudCk6CiBzaXplPWZwYXRoLnN0YXQoKS5zdF9zaXplCiBtZXRhPXsnbmFtZSc6ZnBhdGgubmFtZSwncGFyZW50cyc6W3BhcmVudF19CiB0cnk6CiAgcj1yZXF1ZXN0cy5wb3N0KCdodHRwczovL3d3dy5nb29nbGVhcGlzLmNvbS91cGxvYWQvZHJpdmUvdjMvZmlsZXM/dXBsb2FkVHlwZT1yZXN1bWFibGUnLGhlYWRlcnM9eydBdXRob3JpemF0aW9uJzonQmVhcmVyICcrdG9rLCdDb250ZW50LVR5cGUnOidhcHBsaWNhdGlvbi9qc29uJywnWC1VcGxvYWQtQ29udGVudC1UeXBlJzonYXBwbGljYXRpb24vb2N0ZXQtc3RyZWFtJywnWC1VcGxvYWQtQ29udGVudC1MZW5ndGgnOnN0cihzaXplKX0sZGF0YT1qc29uLmR1bXBzKG1ldGEpLHRpbWVvdXQ9MzApCiAgdXJpPXIuaGVhZGVycy5nZXQoJ0xvY2F0aW9uJykKICBpZiBub3QgdXJpOnByaW50KCcgIEdhZ2FsIG11bGFpIHNlc2kgdXBsb2FkLicpO3JldHVybiBGYWxzZQogZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOnByaW50KCcgIEVycm9yIGluaXNpYXNpOiAnK3N0cihlKVs6MTUwXSk7cmV0dXJuIEZhbHNlCiBDSD02NCoxMDI0KjEwMjQgaWYgc2l6ZT4xMDAqMTAyNCoxMDI0IGVsc2UgMTYqMTAyNCoxMDI0CiB1cD0wO3QwPXRpbWUudGltZSgpCiB0cnk6CiAgZmg9b3BlbihmcGF0aCwncmInKQogIHdoaWxlIHVwPHNpemU6CiAgIGNoPWZoLnJlYWQoQ0gpCiAgIGlmIG5vdCBjaDpicmVhawogICBlbmQ9dXArbGVuKGNoKS0xCiAgIHJyPXJlcXVlc3RzLnB1dCh1cmksaGVhZGVycz17J0NvbnRlbnQtUmFuZ2UnOidieXRlcyAnK3N0cih1cCkrJy0nK3N0cihlbmQpKycvJytzdHIoc2l6ZSksJ0NvbnRlbnQtTGVuZ3RoJzpzdHIobGVuKGNoKSl9LGRhdGE9Y2gsdGltZW91dD0xMjApCiAgIGlmIHJyLnN0YXR1c19jb2RlIGluICgyMDAsMjAxKTp1cCs9bGVuKGNoKTticmVhawogICBlbGlmIHJyLnN0YXR1c19jb2RlPT0zMDg6CiAgICB1cCs9bGVuKGNoKQogICAgZWw9dGltZS50aW1lKCktdDA7c3A9dXAvZWwvMTAyNC8xMDI0IGlmIGVsPjAgZWxzZSAwCiAgICBwcmludCgnICAnK3N0cihyb3VuZCh1cC9zaXplKjEwMCwxKSkrJyUgICcrc3RyKHJvdW5kKHNwLDEpKSsnIE1CL3MnKQogICBlbHNlOnByaW50KCcgIFVwbG9hZCBlcnJvciBIVFRQICcrc3RyKHJyLnN0YXR1c19jb2RlKSk7ZmguY2xvc2UoKTtyZXR1cm4gRmFsc2UKICBmaC5jbG9zZSgpCiBleGNlcHQgRXhjZXB0aW9uIGFzIGU6cHJpbnQoJyAgRXJyb3IgdXBsb2FkOiAnK3N0cihlKVs6MTUwXSk7cmV0dXJuIEZhbHNlCiBwcmludChvaygnICAxMDAlIFNlbGVzYWkuJykpCiByZXR1cm4gVHJ1ZQpkZWYgdXBsb2FkX2dvZmlsZSgpOgogaGRyKCdVUExPQUQgLSBHb2ZpbGUnKQogYWxsX2ZpbGVzPVtdCiBmb3IgZCBpbiBbVVBMT0FELE9VVFBVVCxQYXRoKCcvY29udGVudC9leHRyYWN0cycpLFBhdGgoJy9jb250ZW50L2Rvd25sb2FkcycpXToKICBpZiBkLmV4aXN0cygpOgogICBmb3IgZiBpbiBzb3J0ZWQoZC5yZ2xvYignKicpKToKICAgIGlmIGYuaXNfZmlsZSgpIGFuZCBmLnN1ZmZpeC5sb3dlcigpIGluIFZ8QXxTOmFsbF9maWxlcy5hcHBlbmQoKGQsZikpCiBpZiBub3QgYWxsX2ZpbGVzOnByaW50KGVyKCcgIFRpZGFrIGFkYSBmaWxlIHVudHVrIGRpLXVwbG9hZC4nKSk7aW5wdXQoJyAgRW50ZXIuLi4nKTtyZXR1cm4KIHByaW50KCkKIGlkeD0wCiBmb3IgZCBpbiBbVVBMT0FELE9VVFBVVCxQYXRoKCcvY29udGVudC9leHRyYWN0cycpLFBhdGgoJy9jb250ZW50L2Rvd25sb2FkcycpXToKICBncnA9WyhkZCxmKSBmb3IgZGQsZiBpbiBhbGxfZmlsZXMgaWYgZGQ9PWRdCiAgaWYgbm90IGdycDpjb250aW51ZQogIHByaW50KCcgIFsnK2QubmFtZSsnL10gICgnK3N0cihsZW4oZ3JwKSkrJyBmaWxlKScpCiAgZm9yIGRkLGYgaW4gZ3JwOgogICBzaXplPWYuc3RhdCgpLnN0X3NpemUvMTAyNC8xMDI0CiAgIHByaW50KCcgICAgWycrc3RyKGlkeCkrJ10gJytmLm5hbWUrJyAgJytkaW0oc3RyKGludChzaXplKSkrJ01CJykpCiAgIGlkeCs9MQogIHByaW50KCkKIGZsYXQ9W2YgZm9yIGRkLGYgaW4gYWxsX2ZpbGVzXQogYz1pbnB1dCgnICBQaWxpaCAoKiBzZW11YSAvIDAsMSwyIC8gMC0zIC8gUSBiYXRhbCk6ICcpLnN0cmlwKCkudXBwZXIoKQogaWYgYz09J1EnOnJldHVybgogaWYgYz09JyonOnRhcmdldHM9ZmxhdAogZWxzZToKICB0cnk6CiAgIG51bXM9W10KICAgZm9yIHBhcnQgaW4gYy5zcGxpdCgnLCcpOgogICAgcGFydD1wYXJ0LnN0cmlwKCkKICAgIGlmICctJyBpbiBwYXJ0OmEsYj1wYXJ0LnNwbGl0KCctJywxKTtudW1zLmV4dGVuZChyYW5nZShpbnQoYSksaW50KGIpKzEpKQogICAgZWxzZTpudW1zLmFwcGVuZChpbnQocGFydCkpCiAgIHRhcmdldHM9W2ZsYXRbbl0gZm9yIG4gaW4gbnVtcyBpZiAwPD1uPGxlbihmbGF0KV0KICBleGNlcHQ6cHJpbnQoJyAgSW5wdXQgdGlkYWsgdmFsaWQuJyk7aW5wdXQoJyAgRW50ZXIuLi4nKTtyZXR1cm4KICBpZiBub3QgdGFyZ2V0czpyZXR1cm4KIGlmIGxlbih0YXJnZXRzKT4xOgogIGZuYW1lPWlucHV0KCcgIE5hbWEgZm9sZGVyIFsnK3RhcmdldHNbMF0ucGFyZW50Lm5hbWUrJ106ICcpLnN0cmlwKCkgb3IgdGFyZ2V0c1swXS5wYXJlbnQubmFtZQogZWxzZTpmbmFtZT1Ob25lCiBvayxsaW5rcz1nb2ZpbGVfdXBsb2FkX2ZpbGVzKHRhcmdldHMsZm5hbWUpCiBpZiBsaW5rczoKICBtc2c9JzxiPlVwbG9hZCBHb2ZpbGU8L2I+JwogIGZvciBuYW1lLHVybCBpbiBsaW5rczoKICAgcHJpbnQob2soJyAgJytuYW1lKSkKICAgcHJpbnQoJyAgJyt1cmwrJ1xuJykKICAgbXNnPW1zZysnXG4nK25hbWUrJ1xuJyt1cmwKICB0Z19zZW5kKG1zZykKIGVsc2U6cHJpbnQoZXIoJyAgU2VtdWEgdXBsb2FkIGdhZ2FsLicpKQogaW5wdXQoJ1xuICBFbnRlci4uLicpCmRlZiB1cGxvYWRfZHJpdmUoKToKIGhkcignVVBMT0FEIC0gR29vZ2xlIERyaXZlJykKIGFsbF9maWxlcz1bXQogZm9yIGQgaW4gW1VQTE9BRCxPVVRQVVQsUGF0aCgnL2NvbnRlbnQvZXh0cmFjdHMnKSxQYXRoKCcvY29udGVudC9kb3dubG9hZHMnKV06CiAgaWYgZC5leGlzdHMoKToKICAgZm9yIGYgaW4gc29ydGVkKGQucmdsb2IoJyonKSk6CiAgICBpZiBmLmlzX2ZpbGUoKSBhbmQgZi5zdWZmaXgubG93ZXIoKSBpbiBWfEF8UzphbGxfZmlsZXMuYXBwZW5kKChkLGYpKQogaWYgbm90IGFsbF9maWxlczpwcmludChlcignICBUaWRhayBhZGEgZmlsZSB1bnR1ayBkaS11cGxvYWQuJykpO2lucHV0KCcgIEVudGVyLi4uJyk7cmV0dXJuCiBwcmludCgpCiBpZHg9MAogZm9yIGQgaW4gW1VQTE9BRCxPVVRQVVQsUGF0aCgnL2NvbnRlbnQvZXh0cmFjdHMnKSxQYXRoKCcvY29udGVudC9kb3dubG9hZHMnKV06CiAgZ3JwPVsoZGQsZikgZm9yIGRkLGYgaW4gYWxsX2ZpbGVzIGlmIGRkPT1kXQogIGlmIG5vdCBncnA6Y29udGludWUKICBwcmludCgnICBbJytkLm5hbWUrJy9dICAoJytzdHIobGVuKGdycCkpKycgZmlsZSknKQogIGZvciBkZCxmIGluIGdycDoKICAgc2l6ZT1mLnN0YXQoKS5zdF9zaXplLzEwMjQvMTAyNAogICBwcmludCgnICAgIFsnK3N0cihpZHgpKyddICcrZi5uYW1lKycgICcrZGltKHN0cihpbnQoc2l6ZSkpKydNQicpKQogICBpZHgrPTEKICBwcmludCgpCiBmbGF0PVtmIGZvciBkZCxmIGluIGFsbF9maWxlc10KIGM9aW5wdXQoJyAgUGlsaWggKCogc2VtdWEgLyAwLDEsMiAvIDAtMyAvIFEgYmF0YWwpOiAnKS5zdHJpcCgpLnVwcGVyKCkKIGlmIGM9PSdRJzpyZXR1cm4KIGlmIGM9PScqJzp0YXJnZXRzPWZsYXQKIGVsc2U6CiAgdHJ5OgogICBudW1zPVtdCiAgIGZvciBwYXJ0IGluIGMuc3BsaXQoJywnKToKICAgIHBhcnQ9cGFydC5zdHJpcCgpCiAgICBpZiAnLScgaW4gcGFydDphLGI9cGFydC5zcGxpdCgnLScsMSk7bnVtcy5leHRlbmQocmFuZ2UoaW50KGEpLGludChiKSsxKSkKICAgIGVsc2U6bnVtcy5hcHBlbmQoaW50KHBhcnQpKQogICB0YXJnZXRzPVtmbGF0W25dIGZvciBuIGluIG51bXMgaWYgMDw9bjxsZW4oZmxhdCldCiAgZXhjZXB0OnByaW50KCcgIElucHV0IHRpZGFrIHZhbGlkLicpO2lucHV0KCcgIEVudGVyLi4uJyk7cmV0dXJuCiAgaWYgbm90IHRhcmdldHM6cmV0dXJuCiBjaWQ9Z2RyaXZlX3NlY3JldCgnR0RSSVZFX0NMSUVOVF9JRCcpO3NlYz1nZHJpdmVfc2VjcmV0KCdHRFJJVkVfQ0xJRU5UX1NFQ1JFVCcpO3JlZj1nZHJpdmVfc2VjcmV0KCdHRFJJVkVfUkVGUkVTSF9UT0tFTicpCiBwYXJlbnRfaWQ9Z2RyaXZlX3NlY3JldCgnR0RSSVZFX0ZPTERFUl9JRCcpIG9yICcxcGpwZDYzUFRGdndZZDhpSTdkdk13Y1UtZV9MTXF2VUUnCiBpZiBub3QoY2lkIGFuZCBzZWMgYW5kIHJlZik6CiAgcHJpbnQoZXIoJyAgU2VjcmV0IEdEcml2ZSB0aWRhayBrZWJhY2EuJykpO3ByaW50KCcgIEFrdGlma2FuIHRvZ2dsZSBzZWNyZXQgKyByZS1ydW4gY2VsbCBJbnN0YWxsLicpO2lucHV0KCcgIEVudGVyLi4uJyk7cmV0dXJuCiBwcmludCgnICBBdXRoIHZpYSBBUEkuLi4nKQogdG9rPWdkcml2ZV90b2tlbihjaWQsc2VjLHJlZikKIGlmIG5vdCB0b2s6cHJpbnQoZXIoJyAgR2FnYWwgZGFwYXQgYWNjZXNzIHRva2VuLicpKTtyZXR1cm4KIG09cmUuc2VhcmNoKHInL2ZvbGRlcnMvKFtBLVphLXowLTlfLV0rKScscGFyZW50X2lkKQogaWYgbTpwYXJlbnRfaWQ9bS5ncm91cCgxKQogZWxpZiBsZW4ocGFyZW50X2lkKTwyMDoKICBxPSJuYW1lPSciK3BhcmVudF9pZCsiJyBhbmQgbWltZVR5cGU9J2FwcGxpY2F0aW9uL3ZuZC5nb29nbGUtYXBwcy5mb2xkZXInIGFuZCB0cmFzaGVkPWZhbHNlIgogIHRyeToKICAgcj1yZXF1ZXN0cy5nZXQoJ2h0dHBzOi8vd3d3Lmdvb2dsZWFwaXMuY29tL2RyaXZlL3YzL2ZpbGVzJyxoZWFkZXJzPXsnQXV0aG9yaXphdGlvbic6J0JlYXJlciAnK3Rva30scGFyYW1zPXsncSc6cSwnZmllbGRzJzonZmlsZXMoaWQpJ30sdGltZW91dD0xNSkKICAgZnM9ci5qc29uKCkuZ2V0KCdmaWxlcycsW10pCiAgIGlmIGZzOnBhcmVudF9pZD1mc1swXVsnaWQnXQogIGV4Y2VwdDpwYXNzCiBzdWI9aW5wdXQoJyAgU3ViZm9sZGVyIFsnK2RpbSgnbGFuZ3N1bmcga2UgcGFyZW50JykrJ106ICcpLnN0cmlwKCkKIHRhcmdldD1wYXJlbnRfaWQKIGlmIHN1YjoKICB0cnk6CiAgIHEyPSJuYW1lPSciK3N1YisiJyBhbmQgJyIrcGFyZW50X2lkKyInIGluIHBhcmVudHMgYW5kIG1pbWVUeXBlPSdhcHBsaWNhdGlvbi92bmQuZ29vZ2xlLWFwcHMuZm9sZGVyJyBhbmQgdHJhc2hlZD1mYWxzZSIKICAgcjI9cmVxdWVzdHMuZ2V0KCdodHRwczovL3d3dy5nb29nbGVhcGlzLmNvbS9kcml2ZS92My9maWxlcycsaGVhZGVycz17J0F1dGhvcml6YXRpb24nOidCZWFyZXIgJyt0b2t9LHBhcmFtcz17J3EnOnEyLCdmaWVsZHMnOidmaWxlcyhpZCknfSx0aW1lb3V0PTE1KQogICBmczI9cjIuanNvbigpLmdldCgnZmlsZXMnLFtdKQogICBpZiBmczI6dGFyZ2V0PWZzMlswXVsnaWQnXQogICBlbHNlOgogICAgbWV0YT17J25hbWUnOnN1YiwnbWltZVR5cGUnOidhcHBsaWNhdGlvbi92bmQuZ29vZ2xlLWFwcHMuZm9sZGVyJywncGFyZW50cyc6W3BhcmVudF9pZF19CiAgICByMz1yZXF1ZXN0cy5wb3N0KCdodHRwczovL3d3dy5nb29nbGVhcGlzLmNvbS9kcml2ZS92My9maWxlcycsaGVhZGVycz17J0F1dGhvcml6YXRpb24nOidCZWFyZXIgJyt0b2ssJ0NvbnRlbnQtVHlwZSc6J2FwcGxpY2F0aW9uL2pzb24nfSxkYXRhPWpzb24uZHVtcHMobWV0YSksdGltZW91dD0xNSkKICAgIG5pZD1yMy5qc29uKCkuZ2V0KCdpZCcpCiAgICBpZiBuaWQ6dGFyZ2V0PW5pZDtwcmludCgnICBTdWJmb2xkZXIgZGlidWF0OiAnK3N1YikKICAgIGVsc2U6cHJpbnQoZXIoJyAgR2FnYWwgYnVhdCBzdWJmb2xkZXIuJykpCiAgZXhjZXB0OnByaW50KGVyKCcgIEVycm9yIGJ1YXQgc3ViZm9sZGVyLicpKQogb2tfbj0wO2ZhaWw9W10KIGZvciBmIGluIHRhcmdldHM6CiAgcHJpbnQoJyAgVXBsb2FkICcrZi5uYW1lKycgKCcrc3RyKHJvdW5kKGYuc3RhdCgpLnN0X3NpemUvMTAyNC8xMDI0LDEpKSsnTUIpLi4uJykKICBpZiBnZHJpdmVfdXBsb2FkX2ZpbGUodG9rLGYsdGFyZ2V0KTpva19uKz0xO3ByaW50KCcgICcrb2soJ29rJykrJyAnK2YubmFtZSkKICBlbHNlOmZhaWwuYXBwZW5kKGYubmFtZSk7cHJpbnQoJyAgJytlcignZ2FnYWwnKSsnICcrZi5uYW1lKQogaWYgb2tfbjp0Z19zZW5kKCc8Yj5VcGxvYWQgR0RyaXZlPC9iPlxuJytzdHIob2tfbikrJyBmaWxlIGJlcmhhc2lsJykKIGlmIGZhaWw6cHJpbnQoZXIoJyAgR2FnYWw6ICcrJywgJy5qb2luKGZhaWwpKSkKIGlucHV0KCdcbiAgRW50ZXIuLi4nKQpkZWYgdGdfb3duZXIoKToKIHJldHVybiBnZXRfc2VjcmV0KCdPV05FUl9JRCcpCmRlZiB0Z190b2tlbigpOgogcmV0dXJuIGdldF9zZWNyZXQoJ0hBUlVfQk9UX1RPS0VOJykKZGVmIHRnX3NlbmQobXNnKToKIG9pZD10Z19vd25lcigpO3Rvaz10Z190b2tlbigpCiBpZiBub3Qgb2lkIG9yIG5vdCB0b2s6cmV0dXJuCiB0cnk6cmVxdWVzdHMucG9zdCgnaHR0cHM6Ly9hcGkudGVsZWdyYW0ub3JnL2JvdCcrdG9rKycvc2VuZE1lc3NhZ2UnLGpzb249eydjaGF0X2lkJzpvaWQsJ3RleHQnOm1zZywncGFyc2VfbW9kZSc6J0hUTUwnLCdkaXNhYmxlX3dlYl9wYWdlX3ByZXZpZXcnOlRydWV9LHRpbWVvdXQ9MTApCiBleGNlcHQ6cGFzcwpkZWYgbWVudV91cGxvYWQoKToKIHdoaWxlIFRydWU6CiAgY2koKTtoZHIoJ1VQTE9BRCcpCiAgcHJpbnQoKQogIHByaW50KCcgIFsxXSBHb2ZpbGUgIChmb2xkZXIgZ2FidW5nYW4pJykKICBwcmludCgnICBbMl0gR29vZ2xlIERyaXZlIChtdWx0aS1maWxlICsgc3ViZm9sZGVyKScpCiAgcHJpbnQoKQogIHByaW50KCcgIFswXSBLZW1iYWxpJykKICBwcmludCgpCiAgYz1pbnB1dCgnICBQaWxpaDogJykuc3RyaXAoKQogIGlmIGM9PScwJzpyZXR1cm4KICBlbGlmIGM9PScxJzp1cGxvYWRfZ29maWxlKCkKICBlbGlmIGM9PScyJzp1cGxvYWRfZHJpdmUoKQogIGlucHV0KCdcbiAgRW50ZXIuLi4nKQpkZWYgbWFpbigpOgogbG9hZF9zZWNyZXRzKCkKIG1lbnVfdXBsb2FkKCkKaWYgX19uYW1lX189PSdfX21haW5fXyc6bWFpbigpCg==""",
        'auto-rename': """IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwppbXBvcnQgc3VicHJvY2VzcyxzeXMsb3MscmUsZ2xvYixqc29uLHRpbWUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoClY9eycubWt2JywnLm1wNCcsJy5hdmknLCcubW92JywnLndlYm0nLCcuZmx2JywnLndtdicsJy50cycsJy5tNHYnfQpBPXsnLm1wMycsJy5hYWMnLCcuZmxhYycsJy53YXYnLCcub2dnJywnLm9wdXMnLCcubWthJywnLmFjMycsJy5kdHMnLCcuZWFjMycsJy5tNGEnfQpTPXsnLnNydCcsJy5hc3MnLCcuc3NhJywnLnN1YicsJy5pZHgnLCcuc3VwJywnLnZ0dCcsJy5wZ3MnLCcuc2NjJywnLnNhbWknfQpBTExfRVhUPVZ8QXxTClVQTE9BRD1QYXRoKCcvY29udGVudC91cGxvYWRzJykKT1VUUFVUPVBhdGgoJy9jb250ZW50L291dHB1dCcpCkVYVFJBQ1RTPVBhdGgoJy9jb250ZW50L2V4dHJhY3RzJykKZGVmIGNpKCk6CiBpbXBvcnQgc3lzCiBzeXMuc3Rkb3V0LndyaXRlKCdceDFiWzJKXHgxYltIJykKIHN5cy5zdGRvdXQuZmx1c2goKQpkZWYgb2sodCk6cmV0dXJuICdcMDMzWzkybScrdCsnXDAzM1swbScKZGVmIGVyKHQpOnJldHVybiAnXDAzM1s5MW0nK3QrJ1wwMzNbMG0nCmRlZiBkaW0odCk6cmV0dXJuICdcMDMzWzkwbScrdCsnXDAzM1swbScKZGVmIGhkcih0aXRsZSk6cHJpbnQoJ1xuJysnPScqNjIpO3ByaW50KCcgICcrdGl0bGUpO3ByaW50KCc9Jyo2MikKCmRlZiBjbGVhbl9maWxlbmFtZShuYW1lKToKIG5hbWU9bmFtZS5zdHJpcCgpCiBuYW1lPXJlLnN1YihyJ1xbKFtBLVphLXowLTldKylcXScscidbXDFdICcsbmFtZSkKIG5hbWU9cmUuc3ViKHInXHMrJywnICcsbmFtZSkKIG5hbWU9cmUuc3ViKHInXChEdWFsIEF1ZGlvXCknLCcoRHVhbC1BdWRpbyknLG5hbWUpCiBuYW1lPXJlLnN1YihyJ1woRHVhbCBBdWRpbyAnLCcoRHVhbC1BdWRpbyAnLG5hbWUpCiBuYW1lPXJlLnN1YihyJyAoRHVhbCBBdWRpbykgJywnIChEdWFsLUF1ZGlvKSAnLG5hbWUpCiBtPXJlLnNlYXJjaChyJyg/PCFcZCkoXGR7MSwzfSkoPyFcZCknLG5hbWUpCiBpZiBtOgogIGVwPW0uZ3JvdXAoMSkuemZpbGwoMikKICBiZWZvcmU9bmFtZVs6bS5zdGFydCgpXQogIGFmdGVyPW5hbWVbbS5lbmQoKTpdCiAgaWYgbm90IHJlLnNlYXJjaChyJ1tTc11cZCtbRWVdXGQrJyxuYW1lKToKICAgc2Vhc29uPScwMScKICAgc209cmUuc2VhcmNoKHInW1NzXShcZHsxLDJ9KScsYmVmb3JlKQogICBpZiBzbTpzZWFzb249c20uZ3JvdXAoMSkuemZpbGwoMikKICAgbmFtZT1iZWZvcmUrJ1MnK3NlYXNvbisnRScrZXArYWZ0ZXIKIG5hbWU9cmUuc3ViKHInXHMqXChccyonLCcgKCcsbmFtZSkKIG5hbWU9cmUuc3ViKHInXHMqXClccyonLCcpICcsbmFtZSkKIG5hbWU9cmUuc3ViKHInICArJywnICcsbmFtZSkKIG5hbWU9bmFtZS5zdHJpcCgpCiByZXR1cm4gbmFtZQoKZGVmIHBpY2tfZm9sZGVyKCk6CiBjaSgpO2hkcignQVVUTyBSRU5BTUUgLSBQaWxpaCBGb2xkZXInKQogcHJpbnQoKQogcHJpbnQoJyAgWzFdIC9jb250ZW50L3VwbG9hZHMnKQogcHJpbnQoJyAgWzJdIC9jb250ZW50L291dHB1dCcpCiBwcmludCgnICBbM10gL2NvbnRlbnQvZXh0cmFjdHMnKQogcHJpbnQoJyAgWzRdIFNlbXVhIGZvbGRlcicpCiBwcmludCgnICBbUV0gS2VtYmFsaScpCiBwcmludCgpCiBjPWlucHV0KCcgIFBpbGloOiAnKS5zdHJpcCgpLnVwcGVyKCkKIGlmIGM9PSdRJzpyZXR1cm4gTm9uZQogaWYgYz09JzEnOnJldHVybiBVUExPQUQKIGlmIGM9PScyJzpyZXR1cm4gT1VUUFVUCiBpZiBjPT0nMyc6cmV0dXJuIEVYVFJBQ1RTCiBpZiBjPT0nNCc6cmV0dXJuIFtVUExPQUQsT1VUUFVULEVYVFJBQ1RTXQogcmV0dXJuIE5vbmUKCmRlZiBzY2FuX2ZpbGVzKGZvbGRlcnMpOgogaWYgbm90IGlzaW5zdGFuY2UoZm9sZGVycyxsaXN0KTpmb2xkZXJzPVtmb2xkZXJzXQogZmlsZXM9W10KIGZvciBkIGluIGZvbGRlcnM6CiAgaWYgbm90IGQuZXhpc3RzKCk6Y29udGludWUKICBmb3IgcCBpbiBzb3J0ZWQoZC5yZ2xvYignKicpKToKICAgaWYgbm90IHAuaXNfZmlsZSgpOmNvbnRpbnVlCiAgIGlmIHAuc3VmZml4Lmxvd2VyKCkgaW4gQUxMX0VYVDoKICAgIGNsZWFuZWQ9Y2xlYW5fZmlsZW5hbWUocC5uYW1lKQogICAgaWYgY2xlYW5lZCE9cC5uYW1lOmZpbGVzLmFwcGVuZCgocCxjbGVhbmVkKSkKIHJldHVybiBmaWxlcwoKZGVmIHNob3dfZmlsZXMoZmlsZXMpOgogcHJpbnQoKQogcHJpbnQoJyAgTm8gIE9yaWdpbmFsICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgLT4gQ2xlYW5lZCcpCiBwcmludCgnICAnKyctJyo4MCkKIGZvciBpLChvcmlnLGNsZWFuZWQpIGluIGVudW1lcmF0ZShmaWxlcyk6CiAgcHJpbnQoJyAgJytzdHIoaSkubGp1c3QoNCkrb3JpZy5uYW1lWzo1MF0ubGp1c3QoNTIpKyctPiAnK2NsZWFuZWRbOjQwXSkKCmRlZiBkb19yZW5hbWUoZmlsZXMsc2VsPU5vbmUpOgogb2tfbj0wCiB0YXJnZXRzPWZpbGVzIGlmIHNlbCBpcyBOb25lIGVsc2UgWyhmaWxlc1tpXSkgZm9yIGkgaW4gc2VsIGlmIDA8PWk8bGVuKGZpbGVzKV0KIGZvciBvcmlnLGNsZWFuZWQgaW4gdGFyZ2V0czoKICBuZXdfcGF0aD1vcmlnLnBhcmVudC9jbGVhbmVkCiAgaWYgbmV3X3BhdGguZXhpc3RzKCkgYW5kIG5ld19wYXRoIT1vcmlnOgogICBwcmludCgnICBTa2lwIChleGlzdHMpOiAnK2NsZWFuZWQpO2NvbnRpbnVlCiAgdHJ5OgogICBvcmlnLnJlbmFtZShuZXdfcGF0aCkKICAgcHJpbnQoJyAgJytvaygnT0snKSsnICcrb3JpZy5uYW1lKycgLT4gJytjbGVhbmVkKQogICBva19uKz0xCiAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOnByaW50KCcgICcrZXIoJ0VSUicpKycgJytzdHIoZSlbOjYwXSkKIHByaW50KCdcbiAgUmVuYW1lZDogJytzdHIob2tfbikrJy8nK3N0cihsZW4odGFyZ2V0cykpKQoKZGVmIG1haW4oKToKIHdoaWxlIFRydWU6CiAgZm9sZGVycz1waWNrX2ZvbGRlcigpCiAgaWYgZm9sZGVycyBpcyBOb25lOnJldHVybgogIGZpbGVzPXNjYW5fZmlsZXMoZm9sZGVycykKICBpZiBub3QgZmlsZXM6CiAgIHByaW50KCcgIFRpZGFrIGFkYSBmaWxlIHlhbmcgcGVybHUgZGktcmVuYW1lLicpO2lucHV0KCcgIEVudGVyLi4uJyk7Y29udGludWUKICB3aGlsZSBUcnVlOgogICBjaSgpO2hkcignQVVUTyBSRU5BTUUnKQogICBzaG93X2ZpbGVzKGZpbGVzKQogICBwcmludCgpCiAgIHByaW50KCcgIFtZXSBSZW5hbWUgc2VtdWEgICBbbm9tb3JdIHBpbGloICgwLDIsNSkgICBbUF0gUHJldmlldyAgIFtGXSBHYW50aSBmb2xkZXIgICBbUV0gS2VtYmFsaScpCiAgIHByaW50KCkKICAgYz1pbnB1dCgnICA+ICcpLnN0cmlwKCkudXBwZXIoKQogICBpZiBjPT0nUSc6YnJlYWsKICAgaWYgYz09J0YnOmJyZWFrCiAgIGlmIGM9PSdQJzoKICAgIGZvciBvcmlnLGNsZWFuZWQgaW4gZmlsZXM6CiAgICAgcHJpbnQoJyAgJytvcmlnLm5hbWUpCiAgICAgcHJpbnQoJyAgICAtPiAnK2NsZWFuZWQpCiAgICAgcHJpbnQoKQogICAgaW5wdXQoJyAgRW50ZXIuLi4nKTtjb250aW51ZQogICBpZiBjPT0nWSc6CiAgICBkb19yZW5hbWUoZmlsZXMpCiAgICBpbnB1dCgnICBFbnRlci4uLicpO2NvbnRpbnVlCiAgIHRyeToKICAgIHNlbD1zZXQoKQogICAgZm9yIHBhcnQgaW4gYy5zcGxpdCgnLCcpOgogICAgIHBhcnQ9cGFydC5zdHJpcCgpCiAgICAgaWYgcGFydC5pc2RpZ2l0KCk6c2VsLmFkZChpbnQocGFydCkpCiAgICBkb19yZW5hbWUoZmlsZXMsc2VsKQogICAgaW5wdXQoJyAgRW50ZXIuLi4nKQogICBleGNlcHQ6cGFzcwoKaWYgX19uYW1lX189PSdfX21haW5fXyc6bWFpbigpCg==""",
    }
    for _name, _blob in TOOLS.items():
        _p = '/usr/local/bin/' + _name
        _code = base64.b64decode(_blob).decode('utf-8').replace('\r\n', '\n').replace('\r', '\n')
        if not _code.startswith('#!'):
            _code = '#!/usr/bin/env python3\n' + _code
        with open(_p, 'w', encoding='utf-8') as _f:
            _f.write(_code)
        os.chmod(_p, 0o755)
        # Symlink to /usr/bin to guarantee PATH lookup everywhere
        try:
            _p_usr = '/usr/bin/' + _name
            if os.path.exists(_p_usr) or os.path.islink(_p_usr):
                try: os.remove(_p_usr)
                except Exception: pass
            os.symlink(_p, _p_usr)
        except Exception:
            try:
                import shutil
                shutil.copy2(_p, '/usr/bin/' + _name)
                os.chmod('/usr/bin/' + _name, 0o755)
            except Exception: pass
        print('  OK ' + _name)

    # Configure aliases and PATH for ALL shells
    _all_tool_names = list(TOOLS.keys()) + ['yazi', 'mc']
    _bashrc_entries = [
        "\n# Haru CLI PATH and Aliases",
        "export PATH=/usr/local/bin:/usr/bin:$PATH"
    ]
    for _tn in _all_tool_names:
        _bashrc_entries.append(f"alias {_tn}='/usr/local/bin/{_tn}'")
    _bashrc_entries.append("hash -r 2>/dev/null\n")
    _bashrc_text = "\n".join(_bashrc_entries)

    try:
        with open('/etc/bash.bashrc', 'a', encoding='utf-8') as _f:
            _f.write(_bashrc_text)
    except Exception: pass

    try:
        with open('/etc/profile.d/haru.sh', 'w', encoding='utf-8') as _f:
            _f.write(_bashrc_text)
        os.chmod('/etc/profile.d/haru.sh', 0o755)
    except Exception: pass

    for _rc in ['/root/.bashrc', os.path.expanduser('~/.bashrc'), '/root/.profile']:
        try:
            with open(_rc, 'a', encoding='utf-8') as _f:
                _f.write(_bashrc_text)
        except Exception: pass

    for _d in ['/content/downloads/Video', '/content/downloads/Audio', '/content/downloads/Playlist', '/content/downloads/lrc', '/content/downloads/manga', '/content/downloads/subtitles', '/content/downloads/checker', '/content/uploads', '/content/output', '/content/input']:
        os.makedirs(_d, exist_ok=True)
    try:
        _secrets = {}
        try:
            from google.colab import userdata as _ud
            for _k in ['GOFILE_API_TOKEN','GDRIVE_CLIENT_ID','GDRIVE_CLIENT_SECRET','GDRIVE_REFRESH_TOKEN','GDRIVE_FOLDER_ID','OWNER_ID','HARU_BOT_TOKEN','SUBSOURCE_API_KEY','MANGACOOKIE','MANGACOOKIE_UA','HF_TOKEN','HF_REPO_ID']:
                try:
                    _v = _ud.get(_k)
                    if _v: _secrets[_k]=str(_v).strip()
                except Exception: pass
        except Exception: pass
        for _k,_v in _secrets.items(): os.environ[_k]=_v
        if _secrets:
            import json as _js
            _old = {}
            if os.path.exists('/content/.haru_secrets.json'):
                try: _old = _js.load(open('/content/.haru_secrets.json'))
                except Exception: pass
            _old.update(_secrets)
            with open('/content/.haru_secrets.json','w', encoding='utf-8') as _sf: _js.dump(_old,_sf)
            os.chmod('/content/.haru_secrets.json',0o600)
            print('  Secrets untuk terminal: '+', '.join(sorted(_old.keys())))
        else:
            print('  (Belum ada secret terbaca - aktifkan di menu Rahasia.)')
    except Exception:
        print('  (Skip export secrets.)')
    print()
    print('=' * 66)
    print('✅ Setup Selesai! Semua tools siap digunakan di Terminal bawaan Colab.')
    print('📌 Ketik di terminal: haru-mirror | haru-mux | haru-extract | haru-metadata | haru-download | haru-upload | haru-ytdl | haru-lrc | haru-manga | yazi | mc')
    print('=' * 66)
else:
    print('Install dinonaktifkan.')

## 2 — Web Terminal di Browser (ttyd + Cloudflare) — disarankan
Jalankan cell di bawah, klik link yang muncul. Copy-paste & arrow keys jalan.


In [ ]:
#@title Buka Web Terminal { display-mode: "form" }
import os, time, re, subprocess, requests
from IPython.display import HTML, display

print('Setup web terminal...')
try:
    _s2 = {}
    try:
        from google.colab import userdata as _ud
        for _k in ['GOFILE_API_TOKEN','GDRIVE_CLIENT_ID','GDRIVE_CLIENT_SECRET','GDRIVE_REFRESH_TOKEN','GDRIVE_FOLDER_ID','OWNER_ID','HARU_BOT_TOKEN','SUBSOURCE_API_KEY','MANGACOOKIE','MANGACOOKIE_UA','HF_TOKEN','HF_REPO_ID']:
            try:
                _v = _ud.get(_k)
                if _v: _s2[_k]=str(_v).strip()
            except Exception: pass
    except Exception: pass
    if _s2:
        import json as _js
        try: _old = _js.load(open('/content/.haru_secrets.json'))
        except Exception: _old = {}
        _old.update(_s2)
        with open('/content/.haru_secrets.json','w') as _sf: _js.dump(_old,_sf)
        os.chmod('/content/.haru_secrets.json',0o600)
        print('  Secrets refresh: '+', '.join(sorted(_old.keys())))
except Exception: pass
if not os.path.exists('/usr/local/bin/cloudflared'):
    print('  Download cloudflared...')
    subprocess.run(['curl', '-s', '-L', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', '-o', '/usr/local/bin/cloudflared'])
    subprocess.run(['chmod', '+x', '/usr/local/bin/cloudflared'])
if not os.path.exists('/usr/local/bin/ttyd'):
    print('  Download ttyd...')
    subprocess.run(['curl', '-s', '-L', 'https://github.com/tsl0922/ttyd/releases/latest/download/ttyd.x86_64', '-o', '/usr/local/bin/ttyd'])
    subprocess.run(['chmod', '+x', '/usr/local/bin/ttyd'])
subprocess.run(['pkill', '-f', 'ttyd'], capture_output=True)
subprocess.run(['pkill', '-f', 'cloudflared tunnel'], capture_output=True)
time.sleep(1)
subprocess.run(['tmux', 'set', '-g', 'history-limit', '50000'], capture_output=True)
subprocess.run(['tmux', 'set', '-g', 'mouse', 'on'], capture_output=True)
subprocess.Popen(['/usr/local/bin/ttyd', '-p', '7681', '-W', '-t', 'fontSize=15', 'tmux', 'new-session', '-A', '-s', 'aio', 'bash'], cwd='/content', stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)
print('  Buka tunnel Cloudflare...')
cf = subprocess.Popen(['/usr/local/bin/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:7681'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
web_url = None
end = time.time() + 35
while time.time() < end:
    line = cf.stdout.readline()
    if not line:
        time.sleep(0.3)
        continue
    m = re.findall(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if m:
        web_url = m[-1]
        break
print()
print('=' * 62)
if web_url:
    print('WEB TERMINAL SIAP:')
    print('  ' + web_url)
    print()
    print('  Perintah: haru-mirror | haru-ytdl | haru-lrc | haru-manga')
    display(HTML('<a href="' + web_url + '" target="_blank" style="background:#238636;color:#fff;padding:12px 24px;text-decoration:none;border-radius:6px;font-weight:bold;display:inline-block;">Buka Web Terminal</a>'))
    try:
        from google.colab import userdata as _ud
        _oid = _ud.get('OWNER_ID') or ''
    except Exception:
        _oid = ''
    try:
        from google.colab import userdata as _ud2
        _tg = _ud2.get('HARU_BOT_TOKEN') or ''
    except Exception:
        _tg = ''
    if _oid and _tg:
        try:
            requests.post('https://api.telegram.org/bot' + _tg + '/sendMessage', json={'chat_id': _oid, 'text': '<b>Haru AIO terminal siap!</b>\nWeb: ' + web_url + '\nKetik: haru-ytdl / haru-lrc / haru-manga', 'parse_mode': 'HTML', 'disable_web_page_preview': True}, timeout=8)
            print('  Notif Telegram terkirim.')
        except Exception as _e:
            print('  Gagal kirim Telegram.')
    else:
        print('  (Aktifkan HARU_BOT_TOKEN & OWNER_ID di Secrets biar link auto-post.)')
else:
    print('Gagal dapat URL tunnel. Jalankan ulang cell ini.')
print('=' * 62)
print('Biarkan cell ini running agar tunnel tetap hidup.')
try:
    while True:
        time.sleep(30)
except KeyboardInterrupt:
    print('Web terminal ditutup.')


## 3 — Upload cookies.txt (opsional, untuk ytdl login)


In [ ]:
#@title Upload cookies.txt { display-mode: "form" }
try:
    from google.colab import files
    print('Pilih file cookies.txt dari PC:')
    up = files.upload()
    for name, data in up.items():
        dest = '/content/cookies.txt'
        open(dest, 'wb').write(data)
        print('OK tersimpan di /content/cookies.txt (%d bytes)' % len(data))
except ImportError:
    print('Bukan di Colab - skip.')

## 4 — Upload hasil


In [ ]:
#@title Upload hasil download { display-mode: "form" }
upload_path = "/content/downloads" #@param {type:"string"}
upload_target = "Gofile" #@param ["Gofile", "Google Drive"]

import subprocess, os, re, json, time, requests
from pathlib import Path

def _sec(k):
    v = os.environ.get(k, '')
    if v: return v.strip()
    try:
        from google.colab import userdata
        t = userdata.get(k)
        if t: return str(t).strip()
    except Exception: pass
    if os.path.exists('/content/.haru_secrets.json'):
        try:
            d = json.load(open('/content/.haru_secrets.json'))
            if d.get(k): return str(d[k]).strip()
        except Exception: pass
    return ''

def _tg(msg):
    tok, oid = _sec('HARU_BOT_TOKEN'), _sec('OWNER_ID')
    if not tok or not oid: return
    try: requests.post('https://api.telegram.org/bot'+tok+'/sendMessage', json={'chat_id': oid, 'text': msg, 'parse_mode': 'HTML', 'disable_web_page_preview': True}, timeout=10)
    except Exception: pass

p = Path(upload_path)
files = sorted([f for f in p.rglob('*') if f.is_file()]) if p.exists() else []
if not files:
    print('Tidak ada file di ' + str(p))
else:
    print('%d file:' % len(files))
    for i, f in enumerate(files): print('  [%d] %s (%.1f MB)' % (i, f.name, f.stat().st_size/1024/1024))
    sel = input('Pilih (nomor / * semua): ').strip()
    tgts = files if sel == '*' else [files[int(sel)]] if sel.isdigit() and 0 <= int(sel) < len(files) else []
    for f in tgts:
        if upload_target == 'Gofile':
            srv = 'store1'
            try:
                sv = requests.get('https://api.gofile.io/servers', timeout=15).json()
                if sv.get('status') == 'ok': srv = sv['data']['servers'][0]['name']
            except Exception: pass
            print('Upload %s via %s...' % (f.name, srv))
            r = subprocess.run(['curl','-s','-F','file=@'+str(f),'https://'+srv+'.gofile.io/uploadFile'], capture_output=True, text=True, timeout=600)
            try:
                d = json.loads(r.stdout)
                if d.get('status') == 'ok':
                    print('OK ' + d['data']['downloadPage'])
                    _tg('<b>Upload Gofile</b>\n'+f.name+'\n'+d['data']['downloadPage'])
                else: print('Gagal: ' + r.stdout[:200])
            except Exception: print('Gagal: ' + r.stdout[:200])
        else:
            cid, sec, ref = _sec('GDRIVE_CLIENT_ID'), _sec('GDRIVE_CLIENT_SECRET'), _sec('GDRIVE_REFRESH_TOKEN')
            folder = _sec('GDRIVE_FOLDER_ID') or 'HaruDownloads'
            if not (cid and sec and ref):
                print('Secret GDrive belum lengkap.')
                continue
            tok = requests.post('https://oauth2.googleapis.com/token', data={'client_id': cid, 'client_secret': sec, 'refresh_token': ref, 'grant_type': 'refresh_token'}, timeout=15).json().get('access_token')
            if not tok:
                print('Gagal auth GDrive.')
                continue
            q = "name='"+folder+"' and mimeType='application/vnd.google-apps.folder' and trashed=false"
            fl = requests.get('https://www.googleapis.com/drive/v3/files', headers={'Authorization': 'Bearer '+tok}, params={'q': q, 'fields': 'files(id,name)'}, timeout=15).json().get('files', [])
            if fl: fid = fl[0]['id']
            else: fid = requests.post('https://www.googleapis.com/drive/v3/files', headers={'Authorization': 'Bearer '+tok, 'Content-Type': 'application/json'}, data=json.dumps({'name': folder, 'mimeType': 'application/vnd.google-apps.folder'}), timeout=15).json().get('id')
            size = f.stat().st_size
            ri = requests.post('https://www.googleapis.com/upload/drive/v3/files?uploadType=resumable', headers={'Authorization': 'Bearer '+tok, 'Content-Type': 'application/json', 'X-Upload-Content-Type': 'application/octet-stream', 'X-Upload-Content-Length': str(size)}, data=json.dumps({'name': f.name, 'parents': [fid]}), timeout=30)
            uri = ri.headers.get('Location')
            up, CH, t0, done = 0, 64*1024*1024 if size > 100*1024*1024 else 16*1024*1024, time.time(), False
            fh = open(f, 'rb')
            while up < size:
                ch = fh.read(CH)
                if not ch: break
                end = up + len(ch) - 1
                pr = requests.put(uri, headers={'Content-Range': 'bytes %d-%d/%d' % (up, end, size), 'Content-Length': str(len(ch))}, data=ch, timeout=120)
                if pr.status_code in (200, 201): up += len(ch); done = True; break
                elif pr.status_code == 308:
                    up += len(ch)
                    print('  %.1f%%' % (up/size*100))
                else: print('  HTTP %d' % pr.status_code); break
            fh.close()
            if done or up >= size:
                print('OK terupload ke GDrive.')
                _tg('<b>Upload GDrive</b>\n'+f.name)
